In [1]:
from pathlib import Path
import hashlib
import random
import sys
import time
import librosa
import numpy as np
import pandas as pd
import scipy.io as sio
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

DATA_DIR = Path("/workspace/E-DAIC/original_data")
SPLIT_DIR = Path("/workspace/E-DAIC/splits")
LABEL_DIR = Path("/workspace/E-DAIC/others")

TRAIN_SPLIT_PATH = SPLIT_DIR / "train_split.csv"
VAL_SPLIT_PATH = SPLIT_DIR / "dev_split.csv"
TEST_SPLIT_PATH = SPLIT_DIR / "test_split.csv"
DETAILED_LABEL_PATH = LABEL_DIR / "Detailed_PHQ8_Labels.csv"

VIDEO_CACHE_DIR = Path("/workspace/E-DAIC/cache/phq8_video_resnet/resnet2048_turnmean_turns120_v1")
VIDEO_CHECKPOINT_ROOT = Path("/workspace/E-DAIC/checkpoints/phq8_video_paperlike")

PHQ8_QUESTION_COLUMNS = ["PHQ_8NoInterest", "PHQ_8Depressed", "PHQ_8Sleep", "PHQ_8Tired", "PHQ_8Appetite", "PHQ_8Failure", "PHQ_8Concentrating", "PHQ_8Moving"]

MAX_TURNS = 120
VIDEO_FEATURE_DIM = 2048
LSTM_HIDDEN_SIZE = 50
ENCODER_OUTPUT_DIM = 100
NUM_CLASSES = 4
ATTENTION_HEADS = 4

BATCH_SIZE = 10
NUM_EPOCHS = 50
LEARNING_RATE = 5e-4
ADAM_EPSILON = 1e-8
WEIGHT_DECAY = 1e-3
MAX_GRAD_NORM = 1.0

ATTENTION_DROPOUT = 0.2
MLP_FIRST_DROPOUT = 0.2
MLP_LAST_DROPOUT = 0.2

ALPHA = 1.0
BETA = 0.5
LOSS_EPSILON = 1e-12
SEEDS = [42, 100, 1234]

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print("Python：", sys.version.split()[0])
print("PyTorch：", torch.__version__)
print("设备：", device)
print("数据目录存在：", DATA_DIR.exists())
print("训练划分存在：", TRAIN_SPLIT_PATH.exists())
print("验证划分存在：", VAL_SPLIT_PATH.exists())
print("测试划分存在：", TEST_SPLIT_PATH.exists())
print("详细标签存在：", DETAILED_LABEL_PATH.exists())
print("视频缓存目录：", VIDEO_CACHE_DIR)
print("视频Checkpoint目录：", VIDEO_CHECKPOINT_ROOT)

Python： 3.12.3
PyTorch： 2.9.0
设备： cuda:0
数据目录存在： True
训练划分存在： True
验证划分存在： True
测试划分存在： True
详细标签存在： True
视频缓存目录： /workspace/E-DAIC/cache/phq8_video_resnet/resnet2048_turnmean_turns120_v1
视频Checkpoint目录： /workspace/E-DAIC/checkpoints/phq8_video_paperlike


### 读取元数据

In [2]:
train_split_df = pd.read_csv(TRAIN_SPLIT_PATH)
val_split_df = pd.read_csv(VAL_SPLIT_PATH)
test_split_df = pd.read_csv(TEST_SPLIT_PATH)
detailed_label_df = pd.read_csv(DETAILED_LABEL_PATH)

train_metadata_df = train_split_df.merge(detailed_label_df[["Participant_ID"] + PHQ8_QUESTION_COLUMNS], on="Participant_ID", how="left", validate="one_to_one")
val_metadata_df = val_split_df.merge(detailed_label_df[["Participant_ID"] + PHQ8_QUESTION_COLUMNS], on="Participant_ID", how="left", validate="one_to_one")
test_metadata_df = test_split_df.copy()

if train_metadata_df[PHQ8_QUESTION_COLUMNS].isna().any().any():
    raise ValueError("训练集存在缺失的逐题标签")

if val_metadata_df[PHQ8_QUESTION_COLUMNS].isna().any().any():
    raise ValueError("验证集存在缺失的逐题标签")

print("训练人数：", len(train_metadata_df))
print("验证人数：", len(val_metadata_df))
print("测试人数：", len(test_metadata_df))
print("训练逐题标签形状：", train_metadata_df[PHQ8_QUESTION_COLUMNS].shape)
print("验证逐题标签形状：", val_metadata_df[PHQ8_QUESTION_COLUMNS].shape)

训练人数： 163
验证人数： 55
测试人数： 56
训练逐题标签形状： (163, 8)
验证逐题标签形状： (55, 8)


### 文件路径函数

In [3]:
def get_video_feature_path(participant_id):
    participant_id = int(participant_id)
    return DATA_DIR / f"{participant_id}_P" / "features" / f"{participant_id}_CNN_ResNet.mat"

def get_audio_path(participant_id):
    participant_id = int(participant_id)
    return DATA_DIR / f"{participant_id}_P" / f"{participant_id}_AUDIO.wav"

def get_transcript_path(participant_id):
    participant_id = int(participant_id)
    return DATA_DIR / f"{participant_id}_P" / f"{participant_id}_Transcript.csv"

### 检查视觉文件头

In [9]:
def inspect_video_participant(participant_id, split_name):
    participant_id = int(participant_id)
    video_path = get_video_feature_path(participant_id)
    audio_path = get_audio_path(participant_id)
    transcript_path = get_transcript_path(participant_id)
    result = {"Participant_ID": participant_id, "Split": split_name, "Video_Path": str(video_path), "Video_Exists": video_path.exists(), "Audio_Exists": audio_path.exists(), "Transcript_Exists": transcript_path.exists(), "Feature_Exists": False, "Frame_Count": None, "Feature_Dim": None, "Audio_Duration": None, "Estimated_Rate": None, "Error": None}
    if not video_path.exists() or not audio_path.exists() or not transcript_path.exists():
        return result
    try:
        variable_metadata = sio.whosmat(video_path)
        print(variable_metadata)
        feature_metadata = [variable for variable in variable_metadata if variable[0] == "feature"]
        if len(feature_metadata) != 1:
            result["Error"] = f"feature字段数量为{len(feature_metadata)}"
            return result
        feature_shape = feature_metadata[0][1]
        result["Feature_Exists"] = True
        result["Frame_Count"] = int(feature_shape[0])
        result["Feature_Dim"] = int(feature_shape[1]) if len(feature_shape) == 2 else None
        result["Audio_Duration"] = float(librosa.get_duration(path=audio_path))
        result["Estimated_Rate"] = result["Frame_Count"] / result["Audio_Duration"] if result["Audio_Duration"] > 0 else None
    except Exception as exception:
        result["Error"] = repr(exception)
    return result

In [10]:
video_inventory_rows = []

for split_name, metadata_df in {"train": train_metadata_df, "val": val_metadata_df, "test": test_metadata_df}.items():
    for participant_id in metadata_df["Participant_ID"].astype(int).tolist():
        video_inventory_rows.append(inspect_video_participant(participant_id, split_name))

video_inventory_df = pd.DataFrame(video_inventory_rows)

print("扫描数量：", len(video_inventory_df))
display(video_inventory_df.head())

[('feature', (22766, 2048), 'single')]
[('feature', (29565, 2048), 'single')]
[('feature', (23780, 2048), 'single')]
[('feature', (51122, 2048), 'single')]
[('feature', (37167, 2048), 'single')]
[('feature', (26031, 2048), 'single')]
[('feature', (21391, 2048), 'single')]
[('feature', (25609, 2048), 'single')]
[('feature', (23810, 2048), 'single')]
[('feature', (23945, 2048), 'single')]
[('feature', (22848, 2048), 'single')]
[('feature', (46873, 2048), 'single')]
[('feature', (29561, 2048), 'single')]
[('feature', (26337, 2048), 'single')]
[('feature', (17833, 2048), 'single')]
[('feature', (20603, 2048), 'single')]
[('feature', (31412, 2048), 'single')]
[('feature', (25110, 2048), 'single')]
[('feature', (21640, 2048), 'single')]
[('feature', (26251, 2048), 'single')]
[('feature', (20982, 2048), 'single')]
[('feature', (20427, 2048), 'single')]
[('feature', (31956, 2048), 'single')]
[('feature', (21192, 2048), 'single')]
[('feature', (23146, 2048), 'single')]
[('feature', (26228, 2048

,Participant_ID,Split,Video_Path,Video_Exists,Audio_Exists,Transcript_Exists,Feature_Exists,Frame_Count,Feature_Dim,Audio_Duration,Estimated_Rate,Error
0,302,train,/workspace/E-DAIC/original_data/302_P/features...,True,True,True,True,22766,2048,758.8,30.002636,None
1,303,train,/workspace/E-DAIC/original_data/303_P/features...,True,True,True,True,29565,2048,985.3,30.006090,None
2,304,train,/workspace/E-DAIC/original_data/304_P/features...,True,True,True,True,23780,2048,792.6,30.002523,None
3,305,train,/workspace/E-DAIC/original_data/305_P/features...,True,True,True,True,51122,2048,1704.0,30.001174,None
4,307,train,/workspace/E-DAIC/original_data/307_P/features...,True,True,True,True,37167,2048,1238.8,30.002422,None


In [11]:
invalid_video_df = video_inventory_df[(video_inventory_df["Video_Exists"] != True) | (video_inventory_df["Audio_Exists"] != True) | (video_inventory_df["Transcript_Exists"] != True) | (video_inventory_df["Feature_Exists"] != True) | (video_inventory_df["Feature_Dim"] != VIDEO_FEATURE_DIM) | (video_inventory_df["Frame_Count"].fillna(0) <= 0)]

print("缺失或损坏文件数量：", len(invalid_video_df))
display(invalid_video_df)

缺失或损坏文件数量： 0


,Participant_ID,Split,Video_Path,Video_Exists,Audio_Exists,Transcript_Exists,Feature_Exists,Frame_Count,Feature_Dim,Audio_Duration,Estimated_Rate,Error


### 识别可能被截断的视频

In [12]:
valid_video_df = video_inventory_df[(video_inventory_df["Feature_Exists"] == True) & (video_inventory_df["Feature_Dim"] == VIDEO_FEATURE_DIM) & (video_inventory_df["Frame_Count"] > 0) & video_inventory_df["Estimated_Rate"].notna()].copy()
median_video_rate = valid_video_df["Estimated_Rate"].median()
valid_video_df["Rate_Ratio"] = valid_video_df["Estimated_Rate"] / median_video_rate
video_rate_outliers_df = valid_video_df[(valid_video_df["Rate_Ratio"] < 0.8) | (valid_video_df["Rate_Ratio"] > 1.2)].copy()

print("有效视觉文件数量：", len(valid_video_df))
print("估计采样率最小值：", valid_video_df["Estimated_Rate"].min())
print("估计采样率中位数：", median_video_rate)
print("估计采样率最大值：", valid_video_df["Estimated_Rate"].max())
print("采样率异常数量：", len(video_rate_outliers_df))

display(video_rate_outliers_df[["Participant_ID", "Split", "Frame_Count", "Audio_Duration", "Estimated_Rate", "Rate_Ratio"]])

有效视觉文件数量： 274
估计采样率最小值： 15.784537886306437
估计采样率中位数： 30.001217879611822
估计采样率最大值： 79.14909090909092
采样率异常数量： 3


,Participant_ID,Split,Frame_Count,Audio_Duration,Estimated_Rate,Rate_Ratio
157,692,train,16175,1024.7370,15.784538,0.526130
210,627,val,108830,1375.0000,79.149091,2.638196
239,637,test,37358,1686.6265,22.149539,0.738288


In [13]:
invalid_participant_ids = set(invalid_video_df["Participant_ID"].astype(int).tolist())
rate_outlier_participant_ids = set(video_rate_outliers_df["Participant_ID"].astype(int).tolist())
video_exclusion_candidate_ids = sorted(invalid_participant_ids.union(rate_outlier_participant_ids))

print("全部排除候选：", video_exclusion_candidate_ids)

for split_name in ["train", "val", "test"]:
    split_candidate_ids = sorted(video_inventory_df[(video_inventory_df["Split"] == split_name) & (video_inventory_df["Participant_ID"].isin(video_exclusion_candidate_ids))]["Participant_ID"].astype(int).tolist())
    print(f"{split_name}排除候选：", split_candidate_ids)

全部排除候选： [627, 637, 692]
train排除候选： [692]
val排除候选： [627]
test排除候选： [637]


In [14]:
sample_participant_id = int(valid_video_df.iloc[0]["Participant_ID"])
sample_video_path = get_video_feature_path(sample_participant_id)
sample_video_mat = sio.loadmat(sample_video_path)
sample_video_features = sample_video_mat["feature"]
sample_audio_duration = librosa.get_duration(path=get_audio_path(sample_participant_id))
sample_transcript_df = pd.read_csv(get_transcript_path(sample_participant_id))

print("示例参与者：", sample_participant_id)
print("MAT字段：", [key for key in sample_video_mat.keys() if not key.startswith("__")])
print("ResNet特征形状：", sample_video_features.shape)
print("ResNet特征类型：", sample_video_features.dtype)
print("全部有限：", np.isfinite(sample_video_features).all())
print("音频时长：", sample_audio_duration)
print("视觉估计采样率：", len(sample_video_features) / sample_audio_duration)
print("转录轮数：", len(sample_transcript_df))
print("转录字段：", sample_transcript_df.columns.tolist())

示例参与者： 302
MAT字段： ['feature']
ResNet特征形状： (22766, 2048)
ResNet特征类型： float32
全部有限： True
音频时长： 758.8
视觉估计采样率： 30.00263574064312
转录轮数： 99
转录字段： ['Start_Time', 'End_Time', 'Text', 'Confidence']


In [20]:
VIDEO_EXCLUDED_IDS = set()

video_train_metadata_df = train_metadata_df.reset_index(drop=True).copy()
video_val_metadata_df = val_metadata_df.reset_index(drop=True).copy()
video_test_metadata_df = test_metadata_df.reset_index(drop=True).copy()

assert len(video_train_metadata_df) == 163
assert len(video_val_metadata_df) == 55
assert len(video_test_metadata_df) == 56

print("Video训练人数：", len(video_train_metadata_df))
print("Video验证人数：", len(video_val_metadata_df))
print("Video测试人数：", len(video_test_metadata_df))
print("训练集包含692：", 692 in video_train_metadata_df["Participant_ID"].tolist())
print("验证集包含627：", 627 in video_val_metadata_df["Participant_ID"].tolist())
print("测试集包含637：", 637 in video_test_metadata_df["Participant_ID"].tolist())

Video训练人数： 163
Video验证人数： 55
Video测试人数： 56
训练集包含692： True
验证集包含627： True
测试集包含637： True


In [22]:
def get_file_signature(file_path):
    file_path = Path(file_path)
    file_stat = file_path.stat()
    return {"size": int(file_stat.st_size), "mtime_ns": int(file_stat.st_mtime_ns)}

In [23]:
def create_video_turn_intervals(transcript_df, frame_count, audio_duration):
    if audio_duration <= 0:
        raise ValueError("音频时长必须大于0")
    video_rate = frame_count / audio_duration
    start_times = transcript_df["Start_Time"].to_numpy(dtype=np.float64) * video_rate
    end_times = transcript_df["End_Time"].to_numpy(dtype=np.float64) * video_rate
    cleaned_intervals = []
    kept_turn_indices = []
    previous_start = float("-inf")
    previous_end = float("-inf")
    for turn_index, (start_time, end_time) in enumerate(zip(start_times, end_times)):
        if not np.isfinite(start_time) or not np.isfinite(end_time):
            continue
        if start_time < 0 or end_time < 0:
            continue
        if start_time < previous_start or end_time < previous_end:
            continue
        if start_time > frame_count or end_time > frame_count:
            continue
        start_index = int(round(start_time))
        end_index = int(round(end_time))
        start_index = max(0, min(start_index, frame_count))
        end_index = max(0, min(end_index, frame_count))
        if end_index <= start_index:
            continue
        cleaned_intervals.append((start_index, end_index))
        kept_turn_indices.append(turn_index)
        previous_start = start_time

        previous_end = end_time
    if len(cleaned_intervals) == 0:
        raise RuntimeError("清理后没有有效视觉轮次")
    return cleaned_intervals, kept_turn_indices, video_rate

### 生成单个参与者的视频缓存

In [24]:
VIDEO_CACHE_VERSION = "resnet2048_turnmean_turns120_v1"
VIDEO_CACHE_DIR.mkdir(parents=True, exist_ok=True)

In [25]:
def create_video_cache(participant_id, force_rebuild=False):
    participant_id = int(participant_id)
    if participant_id in VIDEO_EXCLUDED_IDS:
        raise ValueError(f"参与者{participant_id}位于视觉排除列表")
    video_path = get_video_feature_path(participant_id)
    audio_path = get_audio_path(participant_id)
    transcript_path = get_transcript_path(participant_id)
    cache_path = VIDEO_CACHE_DIR / f"{participant_id}.pt"
    if not video_path.exists():
        raise FileNotFoundError(f"视觉文件不存在：{video_path}")
    if not audio_path.exists():
        raise FileNotFoundError(f"音频文件不存在：{audio_path}")
    if not transcript_path.exists():
        raise FileNotFoundError(f"转录文件不存在：{transcript_path}")
    video_signature = get_file_signature(video_path)
    audio_signature = get_file_signature(audio_path)
    transcript_signature = get_file_signature(transcript_path)
    if cache_path.exists() and not force_rebuild:
        try:
            existing_cache = torch.load(cache_path, map_location="cpu", weights_only=False)
            signatures_match = existing_cache.get("video_signature") == video_signature and existing_cache.get("audio_signature") == audio_signature and existing_cache.get("transcript_signature") == transcript_signature
            configuration_matches = existing_cache.get("cache_version") == VIDEO_CACHE_VERSION and existing_cache.get("feature_dim") == VIDEO_FEATURE_DIM and existing_cache.get("max_turns") == MAX_TURNS
            shape_matches = tuple(existing_cache.get("video_features", torch.empty(0)).shape) == (MAX_TURNS, VIDEO_FEATURE_DIM) and tuple(existing_cache.get("padding_mask", torch.empty(0)).shape) == (MAX_TURNS,)
            if signatures_match and configuration_matches and shape_matches:
                return cache_path, existing_cache, False
        except Exception:
            pass
    transcript_df = pd.read_csv(transcript_path)
    required_transcript_columns = {"Start_Time", "End_Time"}
    if not required_transcript_columns.issubset(transcript_df.columns):
        raise RuntimeError(f"参与者{participant_id}转录文件缺少时间字段")
    video_mat = sio.loadmat(video_path)
    if "feature" not in video_mat:
        raise RuntimeError(f"参与者{participant_id}的MAT文件缺少feature字段")
    raw_video_features = np.asarray(video_mat["feature"], dtype=np.float32)
    if raw_video_features.ndim != 2 or raw_video_features.shape[1] != VIDEO_FEATURE_DIM:
        raise RuntimeError(f"参与者{participant_id}视觉特征形状错误：{raw_video_features.shape}")
    if not np.isfinite(raw_video_features).all():
        raise RuntimeError(f"参与者{participant_id}原始视觉特征包含NaN或Inf")
    frame_count = int(raw_video_features.shape[0])
    audio_duration = float(librosa.get_duration(path=audio_path))
    cleaned_intervals, kept_turn_indices, video_rate = create_video_turn_intervals(transcript_df, frame_count, audio_duration)
    turn_mean_list = []
    valid_turn_indices = []
    valid_frame_intervals = []
    for original_turn_index, (start_index, end_index) in zip(kept_turn_indices, cleaned_intervals):
        turn_segment = raw_video_features[start_index:end_index]
        if turn_segment.shape[0] == 0:
            continue
        turn_mean = np.mean(turn_segment, axis=0, dtype=np.float32)
        if not np.isfinite(turn_mean).all():
            continue
        turn_mean_list.append(torch.from_numpy(turn_mean))
        valid_turn_indices.append(int(original_turn_index))
        valid_frame_intervals.append((int(start_index), int(end_index)))
    if len(turn_mean_list) == 0:
        raise RuntimeError(f"参与者{participant_id}没有可用的视频轮次")
    all_turn_means = torch.stack(turn_mean_list, dim=0).float()
    all_turn_means = F.normalize(all_turn_means, p=2, dim=1, eps=LOSS_EPSILON)
    real_turn_count = min(all_turn_means.shape[0], MAX_TURNS)
    video_features = torch.zeros((MAX_TURNS, VIDEO_FEATURE_DIM), dtype=torch.float32)
    padding_mask = torch.ones((MAX_TURNS,), dtype=torch.bool)
    video_features[:real_turn_count] = all_turn_means[:real_turn_count]
    padding_mask[:real_turn_count] = False
    cache = {"participant_id": participant_id, "cache_version": VIDEO_CACHE_VERSION, "feature_dim": VIDEO_FEATURE_DIM, "max_turns": MAX_TURNS, "frame_count": frame_count, "audio_duration_seconds": audio_duration, "estimated_video_rate": video_rate, "original_transcript_turns": len(transcript_df), "cleaned_turns": len(turn_mean_list), "real_turn_count": real_turn_count, "kept_turn_indices": valid_turn_indices[:MAX_TURNS], "frame_intervals": valid_frame_intervals[:MAX_TURNS], "video_features": video_features, "padding_mask": padding_mask, "video_signature": video_signature, "audio_signature": audio_signature, "transcript_signature": transcript_signature}
    temporary_cache_path = cache_path.with_suffix(".tmp")
    torch.save(cache, temporary_cache_path)
    temporary_cache_path.replace(cache_path)
    return cache_path, cache, True

In [26]:
def validate_video_cache(cache):
    participant_id = int(cache["participant_id"])
    video_features = cache["video_features"]
    padding_mask = cache["padding_mask"]
    real_turn_count = int(cache["real_turn_count"])
    if video_features.shape != (MAX_TURNS, VIDEO_FEATURE_DIM):
        raise RuntimeError(f"参与者{participant_id}缓存形状错误：{video_features.shape}")
    if padding_mask.shape != (MAX_TURNS,):
        raise RuntimeError(f"参与者{participant_id}的Mask形状错误")
    if video_features.dtype != torch.float32:
        raise RuntimeError(f"参与者{participant_id}缓存类型错误：{video_features.dtype}")
    if padding_mask.dtype != torch.bool:
        raise RuntimeError(f"参与者{participant_id}的Mask类型错误")
    if not torch.isfinite(video_features).all():
        raise RuntimeError(f"参与者{participant_id}缓存包含NaN或Inf")
    if int((~padding_mask).sum().item()) != real_turn_count:
        raise RuntimeError(f"参与者{participant_id}真实轮次数与Mask不一致")
    if padding_mask[:real_turn_count].any() or not padding_mask[real_turn_count:].all():
        raise RuntimeError(f"参与者{participant_id}的Mask不是前真后补齐结构")
    if real_turn_count < MAX_TURNS and torch.count_nonzero(video_features[real_turn_count:]).item() != 0:
        raise RuntimeError(f"参与者{participant_id}填充位置不是0")
    real_features = video_features[:real_turn_count]
    real_norms = torch.linalg.vector_norm(real_features, ord=2, dim=1)
    nonzero_norms = real_norms[real_norms > LOSS_EPSILON]
    if nonzero_norms.numel() > 0 and not torch.allclose(nonzero_norms, torch.ones_like(nonzero_norms), atol=1e-5, rtol=1e-5):
        raise RuntimeError(f"参与者{participant_id}的L2归一化错误")
    return {"participant_id": participant_id, "frame_count": cache["frame_count"], "estimated_video_rate": cache["estimated_video_rate"], "original_turns": cache["original_transcript_turns"], "cleaned_turns": cache["cleaned_turns"], "real_turn_count": real_turn_count, "minimum_nonzero_norm": float(nonzero_norms.min().item()) if nonzero_norms.numel() > 0 else 0.0, "maximum_nonzero_norm": float(nonzero_norms.max().item()) if nonzero_norms.numel() > 0 else 0.0}

In [29]:
video_cache_test_rows = []

for participant_id in [302, 627, 692]:
    cache_path, cache, cache_created = create_video_cache(participant_id, force_rebuild=True)
    validation_result = validate_video_cache(cache)
    validation_result["cache_path"] = str(cache_path)
    validation_result["cache_created"] = cache_created
    video_cache_test_rows.append(validation_result)

video_cache_test_df = pd.DataFrame(video_cache_test_rows)
display(video_cache_test_df)

,participant_id,frame_count,estimated_video_rate,original_turns,cleaned_turns,real_turn_count,minimum_nonzero_norm,maximum_nonzero_norm,cache_path,cache_created
0,302,22766,30.002636,99,98,98,1.0,1.0,/workspace/E-DAIC/cache/phq8_video_resnet/resn...,True
1,627,108830,79.149091,154,154,120,1.0,1.0,/workspace/E-DAIC/cache/phq8_video_resnet/resn...,True
2,692,16175,15.784538,57,57,57,1.0,1.0,/workspace/E-DAIC/cache/phq8_video_resnet/resn...,True


In [30]:
for participant_id in [302, 627, 692]:
    cache = torch.load(VIDEO_CACHE_DIR / f"{participant_id}.pt", map_location="cpu", weights_only=False)
    print("=" * 70)
    print("参与者：", participant_id)
    print("估计视频采样率：", cache["estimated_video_rate"])
    print("原始转录轮数：", cache["original_transcript_turns"])
    print("清理后轮数：", cache["cleaned_turns"])
    print("模型实际轮数：", cache["real_turn_count"])
    print("特征形状：", cache["video_features"].shape)
    print("Mask真实轮数：", int((~cache["padding_mask"]).sum().item()))
    print("全部有限：", torch.isfinite(cache["video_features"]).all().item())

参与者： 302
估计视频采样率： 30.00263574064312
原始转录轮数： 99
清理后轮数： 98
模型实际轮数： 98
特征形状： torch.Size([120, 2048])
Mask真实轮数： 98
全部有限： True
参与者： 627
估计视频采样率： 79.14909090909092
原始转录轮数： 154
清理后轮数： 154
模型实际轮数： 120
特征形状： torch.Size([120, 2048])
Mask真实轮数： 120
全部有限： True
参与者： 692
估计视频采样率： 15.784537886306437
原始转录轮数： 57
清理后轮数： 57
模型实际轮数： 57
特征形状： torch.Size([120, 2048])
Mask真实轮数： 57
全部有限： True


In [31]:
import gc

train_participant_ids = video_train_metadata_df["Participant_ID"].astype(int).tolist()
val_participant_ids = video_val_metadata_df["Participant_ID"].astype(int).tolist()
test_participant_ids = video_test_metadata_df["Participant_ID"].astype(int).tolist()

if set(train_participant_ids).intersection(val_participant_ids):
    raise RuntimeError("训练集和验证集存在重复参与者")

if set(train_participant_ids).intersection(test_participant_ids):
    raise RuntimeError("训练集和测试集存在重复参与者")

if set(val_participant_ids).intersection(test_participant_ids):
    raise RuntimeError("验证集和测试集存在重复参与者")

participant_split_map = {}

for participant_id in train_participant_ids:
    participant_split_map[participant_id] = "train"

for participant_id in val_participant_ids:
    participant_split_map[participant_id] = "val"

for participant_id in test_participant_ids:
    participant_split_map[participant_id] = "test"

all_video_participant_ids = sorted(participant_split_map.keys())

print("训练参与者：", len(train_participant_ids))
print("验证参与者：", len(val_participant_ids))
print("测试参与者：", len(test_participant_ids))
print("全部唯一参与者：", len(all_video_participant_ids))

训练参与者： 163
验证参与者： 55
测试参与者： 56
全部唯一参与者： 274


In [32]:
video_cache_manifest_rows = []
video_cache_error_rows = []
created_cache_count = 0
reused_cache_count = 0
cache_start_time = time.time()

for participant_index, participant_id in enumerate(all_video_participant_ids):
    participant_start_time = time.time()
    try:
        cache_path, cache, cache_created = create_video_cache(participant_id, force_rebuild=False)
        validation_result = validate_video_cache(cache)
        manifest_row = {}
        manifest_row["Participant_ID"] = participant_id
        manifest_row["Split"] = participant_split_map[participant_id]
        manifest_row["Cache_Path"] = str(cache_path)
        manifest_row["Cache_Created"] = cache_created
        manifest_row["Frame_Count"] = validation_result["frame_count"]
        manifest_row["Estimated_Video_Rate"] = validation_result["estimated_video_rate"]
        manifest_row["Original_Turns"] = validation_result["original_turns"]
        manifest_row["Cleaned_Turns"] = validation_result["cleaned_turns"]
        manifest_row["Real_Turn_Count"] = validation_result["real_turn_count"]
        manifest_row["Minimum_Nonzero_Norm"] = validation_result["minimum_nonzero_norm"]
        manifest_row["Maximum_Nonzero_Norm"] = validation_result["maximum_nonzero_norm"]
        manifest_row["Seconds"] = time.time() - participant_start_time
        video_cache_manifest_rows.append(manifest_row)
        if cache_created:
            created_cache_count += 1
        else:
            reused_cache_count += 1
        del cache
    except Exception as exception:
        video_cache_error_rows.append({"Participant_ID": participant_id, "Split": participant_split_map[participant_id], "Error": repr(exception)})
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if (participant_index + 1) % 10 == 0 or participant_index + 1 == len(all_video_participant_ids):
        elapsed_seconds = time.time() - cache_start_time
        print(f"视频缓存进度：{participant_index + 1}/{len(all_video_participant_ids)} | 新建：{created_cache_count} | 复用：{reused_cache_count} | 错误：{len(video_cache_error_rows)} | 用时：{elapsed_seconds:.1f}s")

视频缓存进度：10/274 | 新建：9 | 复用：1 | 错误：0 | 用时：4.1s
视频缓存进度：20/274 | 新建：19 | 复用：1 | 错误：0 | 用时：7.8s
视频缓存进度：30/274 | 新建：29 | 复用：1 | 错误：0 | 用时：11.4s
视频缓存进度：40/274 | 新建：39 | 复用：1 | 错误：0 | 用时：15.0s
视频缓存进度：50/274 | 新建：49 | 复用：1 | 错误：0 | 用时：18.4s
视频缓存进度：60/274 | 新建：59 | 复用：1 | 错误：0 | 用时：21.4s
视频缓存进度：70/274 | 新建：69 | 复用：1 | 错误：0 | 用时：25.6s
视频缓存进度：80/274 | 新建：79 | 复用：1 | 错误：0 | 用时：29.7s
视频缓存进度：90/274 | 新建：89 | 复用：1 | 错误：0 | 用时：33.3s
视频缓存进度：100/274 | 新建：99 | 复用：1 | 错误：0 | 用时：36.4s
视频缓存进度：110/274 | 新建：109 | 复用：1 | 错误：0 | 用时：40.3s
视频缓存进度：120/274 | 新建：119 | 复用：1 | 错误：0 | 用时：43.9s
视频缓存进度：130/274 | 新建：129 | 复用：1 | 错误：0 | 用时：47.4s
视频缓存进度：140/274 | 新建：139 | 复用：1 | 错误：0 | 用时：51.3s
视频缓存进度：150/274 | 新建：149 | 复用：1 | 错误：0 | 用时：54.9s
视频缓存进度：160/274 | 新建：159 | 复用：1 | 错误：0 | 用时：58.3s
视频缓存进度：170/274 | 新建：169 | 复用：1 | 错误：0 | 用时：61.9s
视频缓存进度：180/274 | 新建：179 | 复用：1 | 错误：0 | 用时：65.5s
视频缓存进度：190/274 | 新建：189 | 复用：1 | 错误：0 | 用时：68.7s
视频缓存进度：200/274 | 新建：199 | 复用：1 | 错误：0 | 用时：72.2s
视频缓存进度：210/274 | 新建：209 | 复用：1 | 错误：0 | 用时

#### 检查生成错误

In [33]:
video_cache_error_df = pd.DataFrame(video_cache_error_rows)

print("缓存错误数量：", len(video_cache_error_df))

if len(video_cache_error_df) > 0:
    display(video_cache_error_df)
    raise RuntimeError("存在视频缓存生成错误，禁止进入模型训练")

print("全部视频缓存生成成功")

缓存错误数量： 0
全部视频缓存生成成功


#### 保存缓存清单#%%


In [35]:
video_cache_manifest_df = pd.DataFrame(video_cache_manifest_rows).sort_values(["Split", "Participant_ID"]).reset_index(drop=True)
video_cache_manifest_path = VIDEO_CACHE_DIR / "cache_manifest.csv"
video_cache_manifest_df.to_csv(video_cache_manifest_path, index=False)

display(video_cache_manifest_df.head())
print("缓存清单：", video_cache_manifest_path)

,Participant_ID,Split,Cache_Path,Cache_Created,Frame_Count,Estimated_Video_Rate,Original_Turns,Cleaned_Turns,Real_Turn_Count,Minimum_Nonzero_Norm,Maximum_Nonzero_Norm,Seconds
0,600,test,/workspace/E-DAIC/cache/phq8_video_resnet/resn...,True,20579,30.007728,70,70,70,1.0,1.0,0.138849
1,602,test,/workspace/E-DAIC/cache/phq8_video_resnet/resn...,True,25550,32.845261,55,54,54,1.0,1.0,0.200553
2,604,test,/workspace/E-DAIC/cache/phq8_video_resnet/resn...,True,18832,30.044192,57,57,57,1.0,1.0,0.103916
3,605,test,/workspace/E-DAIC/cache/phq8_video_resnet/resn...,True,24269,30.014346,59,59,59,1.0,1.0,0.164316
4,606,test,/workspace/E-DAIC/cache/phq8_video_resnet/resn...,True,15880,30.008126,54,54,54,1.0,1.0,0.095459


缓存清单： /workspace/E-DAIC/cache/phq8_video_resnet/resnet2048_turnmean_turns120_v1/cache_manifest.csv


#### 全局缓存验收

In [36]:
expected_cache_paths = [VIDEO_CACHE_DIR / f"{participant_id}.pt" for participant_id in all_video_participant_ids]
missing_cache_paths = [cache_path for cache_path in expected_cache_paths if not cache_path.exists()]

print("预期缓存数量：", len(expected_cache_paths))
print("实际存在数量：", sum(cache_path.exists() for cache_path in expected_cache_paths))
print("缺失缓存数量：", len(missing_cache_paths))
print("新建缓存数量：", created_cache_count)
print("复用缓存数量：", reused_cache_count)
print("全部特征维度正确：", (video_cache_manifest_df["Minimum_Nonzero_Norm"] > 0).all())
print("全部真实轮数合法：", video_cache_manifest_df["Real_Turn_Count"].between(1, MAX_TURNS).all())
print("达到120轮的参与者数量：", int((video_cache_manifest_df["Real_Turn_Count"] == MAX_TURNS).sum()))
print("真实轮数最小值：", int(video_cache_manifest_df["Real_Turn_Count"].min()))
print("真实轮数中位数：", float(video_cache_manifest_df["Real_Turn_Count"].median()))
print("真实轮数最大值：", int(video_cache_manifest_df["Real_Turn_Count"].max()))

if missing_cache_paths:
    raise RuntimeError(f"存在缺失缓存：{missing_cache_paths[:10]}")

预期缓存数量： 274
实际存在数量： 274
缺失缓存数量： 0
新建缓存数量： 271
复用缓存数量： 3
全部特征维度正确： True
全部真实轮数合法： True
达到120轮的参与者数量： 50
真实轮数最小值： 42
真实轮数中位数： 88.0
真实轮数最大值： 120


In [37]:
video_cache_split_summary_df = video_cache_manifest_df.groupby("Split").agg(Participant_Count=("Participant_ID", "count"), Mean_Original_Turns=("Original_Turns", "mean"), Mean_Cleaned_Turns=("Cleaned_Turns", "mean"), Mean_Real_Turns=("Real_Turn_Count", "mean"), Min_Video_Rate=("Estimated_Video_Rate", "min"), Median_Video_Rate=("Estimated_Video_Rate", "median"), Max_Video_Rate=("Estimated_Video_Rate", "max"), Total_Seconds=("Seconds", "sum")).reset_index()

display(video_cache_split_summary_df)

,Split,Participant_Count,Mean_Original_Turns,Mean_Cleaned_Turns,Mean_Real_Turns,Min_Video_Rate,Median_Video_Rate,Max_Video_Rate,Total_Seconds
0,test,56,99.482143,99.196429,93.357143,22.149539,30.000241,32.845261,12.225127
1,train,163,92.803681,92.503067,88.128834,15.784538,30.001637,32.490017,37.192718
2,val,55,96.072727,95.727273,92.327273,26.163300,30.001165,79.149091,11.185398


### 建立 Video Dataset

In [39]:
class VideoPHQ8Dataset(Dataset):
    def __init__(self, metadata_df, split_name):
        super().__init__()
        if split_name not in {"train", "val"}:
            raise ValueError("split_name只能是train或val")
        self.split_name = split_name
        self.metadata_df = metadata_df.reset_index(drop=True).copy()
        self.participant_ids = self.metadata_df["Participant_ID"].astype(int).tolist()
        self.labels = torch.tensor(self.metadata_df[PHQ8_QUESTION_COLUMNS].to_numpy(), dtype=torch.long)
        video_feature_list = []
        video_mask_list = []
        for index, participant_id in enumerate(self.participant_ids):
            cache_path = VIDEO_CACHE_DIR / f"{participant_id}.pt"
            if not cache_path.exists():
                raise FileNotFoundError(f"Video缓存不存在：{cache_path}")
            cache = torch.load(cache_path, map_location="cpu", weights_only=False)
            video_features = cache["video_features"].float()
            video_padding_mask = cache["padding_mask"].bool()
            if int(cache["participant_id"]) != participant_id:
                raise RuntimeError(f"缓存参与者编号错误：预期{participant_id}，实际{cache['participant_id']}")
            if video_features.shape != (MAX_TURNS, VIDEO_FEATURE_DIM):
                raise RuntimeError(f"参与者{participant_id}特征形状错误：{video_features.shape}")
            if video_padding_mask.shape != (MAX_TURNS,):
                raise RuntimeError(f"参与者{participant_id}Mask形状错误：{video_padding_mask.shape}")
            if not torch.isfinite(video_features).all():
                raise RuntimeError(f"参与者{participant_id}特征包含NaN或Inf")
            if int((~video_padding_mask).sum().item()) != int(cache["real_turn_count"]):
                raise RuntimeError(f"参与者{participant_id}真实轮数与Mask不一致")
            video_feature_list.append(video_features)
            video_mask_list.append(video_padding_mask)
            if (index + 1) % 50 == 0 or index + 1 == len(self.participant_ids):
                print(f"{split_name} Dataset加载进度：{index + 1}/{len(self.participant_ids)}")
        self.video_features = torch.stack(video_feature_list, dim=0)
        self.video_masks = torch.stack(video_mask_list, dim=0)
        if self.video_features.shape != (len(self.participant_ids), MAX_TURNS, VIDEO_FEATURE_DIM):
            raise RuntimeError(f"{split_name}整体特征形状错误：{self.video_features.shape}")
        if self.video_masks.shape != (len(self.participant_ids), MAX_TURNS):
            raise RuntimeError(f"{split_name}整体Mask形状错误：{self.video_masks.shape}")
        if self.labels.shape != (len(self.participant_ids), len(PHQ8_QUESTION_COLUMNS)):
            raise RuntimeError(f"{split_name}标签形状错误：{self.labels.shape}")

    def __len__(self):
        return len(self.participant_ids)

    def __getitem__(self, index):
        return self.video_features[index], self.video_masks[index], self.labels[index]

In [40]:
video_train_dataset = VideoPHQ8Dataset(video_train_metadata_df, split_name="train")
video_val_dataset = VideoPHQ8Dataset(video_val_metadata_df, split_name="val")
print("Video训练样本数：", len(video_train_dataset))
print("Video验证样本数：", len(video_val_dataset))
print("训练特征整体形状：", video_train_dataset.video_features.shape)
print("验证特征整体形状：", video_val_dataset.video_features.shape)
print("训练Mask整体形状：", video_train_dataset.video_masks.shape)
print("验证Mask整体形状：", video_val_dataset.video_masks.shape)
print("训练标签整体形状：", video_train_dataset.labels.shape)
print("验证标签整体形状：", video_val_dataset.labels.shape)

train Dataset加载进度：50/163
train Dataset加载进度：100/163
train Dataset加载进度：150/163
train Dataset加载进度：163/163
val Dataset加载进度：50/55
val Dataset加载进度：55/55
Video训练样本数： 163
Video验证样本数： 55
训练特征整体形状： torch.Size([163, 120, 2048])
验证特征整体形状： torch.Size([55, 120, 2048])
训练Mask整体形状： torch.Size([163, 120])
验证Mask整体形状： torch.Size([55, 120])
训练标签整体形状： torch.Size([163, 8])
验证标签整体形状： torch.Size([55, 8])


In [41]:
def create_video_dataloaders(seed):
    train_generator = torch.Generator()
    train_generator.manual_seed(seed)
    train_loader = DataLoader(video_train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, drop_last=False, generator=train_generator)
    val_loader = DataLoader(video_val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, drop_last=False)
    return train_loader, val_loader

In [42]:
video_train_loader, video_val_loader = create_video_dataloaders(seed=42)
batch_video_features, batch_video_masks, batch_all_labels = next(iter(video_train_loader))
print("训练批次数：", len(video_train_loader))
print("验证批次数：", len(video_val_loader))
print("Batch Video特征：", batch_video_features.shape)
print("Batch Video Mask：", batch_video_masks.shape)
print("Batch全部标签：", batch_all_labels.shape)
print("Batch Q1标签：", batch_all_labels[:, 0])
print("Video特征全部有限：", bool(torch.isfinite(batch_video_features).all()))
print("每名参与者真实轮数：", (~batch_video_masks).sum(dim=1))
print("标签范围：", int(batch_all_labels.min()), "～", int(batch_all_labels.max()))

训练批次数： 17
验证批次数： 6
Batch Video特征： torch.Size([10, 120, 2048])
Batch Video Mask： torch.Size([10, 120])
Batch全部标签： torch.Size([10, 8])
Batch Q1标签： tensor([1, 0, 1, 0, 0, 0, 0, 0, 0, 1])
Video特征全部有限： True
每名参与者真实轮数： tensor([120,  64,  66,  56, 120,  42,  84,  88, 114, 115])
标签范围： 0 ～ 3


### 定义 Video-only 模型|

In [43]:
class VideoOnlyQuestModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=VIDEO_FEATURE_DIM, hidden_size=50, batch_first=True, bidirectional=True)
        self.attention1 = nn.MultiheadAttention(embed_dim=100, num_heads=ATTENTION_HEADS, dropout=0.2, batch_first=True)
        self.attention2 = nn.MultiheadAttention(embed_dim=100, num_heads=ATTENTION_HEADS, dropout=0.2, batch_first=True)
        self.mlp = nn.Sequential(nn.Flatten(), nn.Dropout(0.2), nn.Linear(MAX_TURNS * 100, 256), nn.ReLU(), nn.Dropout(0.2), nn.Linear(256, NUM_CLASSES))

    def forward(self, video_features, video_padding_mask, return_intermediates=False):
        lstm_output, _ = self.lstm(video_features)
        attention1_output, _ = self.attention1(lstm_output, lstm_output, lstm_output, key_padding_mask=video_padding_mask, need_weights=False)
        attention1_output = attention1_output.masked_fill(video_padding_mask.unsqueeze(-1), 0.0)
        attention2_output, _ = self.attention2(attention1_output, attention1_output, attention1_output, key_padding_mask=video_padding_mask, need_weights=False)
        attention2_output = attention2_output.masked_fill(video_padding_mask.unsqueeze(-1), 0.0)
        logits = self.mlp(attention2_output)
        if return_intermediates:
            return logits, {"lstm_output": lstm_output, "attention1_output": attention1_output, "attention2_output": attention2_output}
        return logits

In [45]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [46]:
set_seed(42)
video_q1_model = VideoOnlyQuestModel().to(device)
video_q1_model.eval()
with torch.no_grad():
    video_logits, video_intermediates = video_q1_model(batch_video_features.to(device), batch_video_masks.to(device), return_intermediates=True)
    video_probabilities = torch.softmax(video_logits, dim=1)
print("输入特征：", batch_video_features.shape)
print("LSTM输出：", video_intermediates["lstm_output"].shape)
print("第一层Attention：", video_intermediates["attention1_output"].shape)
print("第二层Attention：", video_intermediates["attention2_output"].shape)
print("最终Logits：", video_logits.shape)
print("第一个样本概率：", video_probabilities[0])
print("概率之和：", video_probabilities[0].sum().item())
print("Logits全部有限：", bool(torch.isfinite(video_logits).all()))
print("模型总参数：", f"{sum(parameter.numel() for parameter in video_q1_model.parameters()):,}")
print("模型可训练参数：", f"{sum(parameter.numel() for parameter in video_q1_model.parameters() if parameter.requires_grad):,}")

输入特征： torch.Size([10, 120, 2048])
LSTM输出： torch.Size([10, 120, 100])
第一层Attention： torch.Size([10, 120, 100])
第二层Attention： torch.Size([10, 120, 100])
最终Logits： torch.Size([10, 4])
第一个样本概率： tensor([0.2558, 0.2634, 0.2384, 0.2424], device='cuda:0')
概率之和： 1.0
Logits全部有限： True
模型总参数： 3,994,084
模型可训练参数： 3,994,084


In [47]:
class ImbOLLLoss(nn.Module):
    def __init__(self, class_weights, alpha=1.0, epsilon=1e-12):
        super().__init__()
        self.register_buffer("class_weights", class_weights.float())
        self.alpha = alpha
        self.epsilon = epsilon

    def forward(self, logits, targets):
        probabilities = torch.softmax(logits, dim=1)
        class_indices = torch.arange(logits.shape[1], device=logits.device).view(1, -1)
        ordinal_distances = torch.abs(targets.view(-1, 1) - class_indices).float()
        sample_weights = self.class_weights[targets].view(-1, 1)
        weighted_distances = torch.pow(sample_weights * ordinal_distances, self.alpha)
        losses = -torch.log(1.0 - probabilities + self.epsilon) * weighted_distances
        return losses.sum(dim=1).mean()

In [48]:
q1_train_labels = video_train_dataset.labels[:, 0]
q1_class_counts = torch.bincount(q1_train_labels, minlength=NUM_CLASSES)
q1_class_weights = torch.pow(len(q1_train_labels) / q1_class_counts.float(), BETA)
q1_criterion = ImbOLLLoss(q1_class_weights, alpha=ALPHA).to(device)
print("Q1类别数量：", q1_class_counts.tolist())
print("Q1 ImbOLL权重：", q1_class_weights.tolist())

Q1类别数量： [81, 57, 20, 5]
Q1 ImbOLL权重： [1.4185717105865479, 1.6910496950149536, 2.854820728302002, 5.709641456604004]


In [49]:
set_seed(42)
video_q1_model = VideoOnlyQuestModel().to(device)
video_q1_optimizer = torch.optim.AdamW(video_q1_model.parameters(), lr=LEARNING_RATE, eps=ADAM_EPSILON, weight_decay=WEIGHT_DECAY)
video_q1_model.train()
diagnostic_video_features = batch_video_features.to(device)
diagnostic_video_masks = batch_video_masks.to(device)
diagnostic_q1_labels = batch_all_labels[:, 0].to(device)
video_q1_optimizer.zero_grad(set_to_none=True)
diagnostic_logits = video_q1_model(diagnostic_video_features, diagnostic_video_masks)
diagnostic_loss = q1_criterion(diagnostic_logits, diagnostic_q1_labels)
diagnostic_loss.backward()
gradient_norm_before_clipping = torch.sqrt(sum(parameter.grad.detach().pow(2).sum() for parameter in video_q1_model.parameters() if parameter.grad is not None)).item()
gradient_tensor_count = sum(parameter.grad is not None for parameter in video_q1_model.parameters())
torch.nn.utils.clip_grad_norm_(video_q1_model.parameters(), MAX_GRAD_NORM)
video_q1_optimizer.step()
print("Logits形状：", diagnostic_logits.shape)
print("ImbOLL：", diagnostic_loss.item())
print("预测概率全部有限：", bool(torch.isfinite(torch.softmax(diagnostic_logits, dim=1)).all()))
print("预测类别：", diagnostic_logits.argmax(dim=1))
print("真实类别：", diagnostic_q1_labels)
print("梯度范数：", gradient_norm_before_clipping)
print("具有梯度的参数张量数：", gradient_tensor_count)

Logits形状： torch.Size([10, 4])
ImbOLL： 2.24094557762146
预测概率全部有限： True
预测类别： tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1], device='cuda:0')
真实类别： tensor([1, 0, 1, 0, 0, 0, 0, 0, 0, 1], device='cuda:0')
梯度范数： 1.0265271663665771
具有梯度的参数张量数： 20


In [50]:
def calculate_ccc(predictions, targets, epsilon=1e-12):
    predictions = predictions.double()
    targets = targets.double()
    prediction_mean = predictions.mean()
    target_mean = targets.mean()
    covariance = ((predictions - prediction_mean) * (targets - target_mean)).mean()
    prediction_variance = ((predictions - prediction_mean) ** 2).mean()
    target_variance = ((targets - target_mean) ** 2).mean()
    ccc = (2.0 * covariance) / (prediction_variance + target_variance + (prediction_mean - target_mean) ** 2 + epsilon)
    return float(ccc.item())

In [51]:
def calculate_video_metrics(predictions, targets):
    predictions = predictions.long()
    targets = targets.long()
    confusion_matrix = torch.zeros(NUM_CLASSES, NUM_CLASSES, dtype=torch.long)
    for target, prediction in zip(targets, predictions):
        confusion_matrix[target, prediction] += 1
    accuracy = float((predictions == targets).float().mean().item())
    class_f1_scores = []
    class_supports = []
    for class_index in range(NUM_CLASSES):
        true_positive = confusion_matrix[class_index, class_index].item()
        false_positive = confusion_matrix[:, class_index].sum().item() - true_positive
        false_negative = confusion_matrix[class_index, :].sum().item() - true_positive
        precision = true_positive / (true_positive + false_positive) if true_positive + false_positive > 0 else 0.0
        recall = true_positive / (true_positive + false_negative) if true_positive + false_negative > 0 else 0.0
        class_f1 = 2.0 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0
        class_f1_scores.append(class_f1)
        class_supports.append(confusion_matrix[class_index, :].sum().item())
    macro_f1 = float(sum(class_f1_scores) / NUM_CLASSES)
    weighted_f1 = float(sum(class_f1_scores[index] * class_supports[index] for index in range(NUM_CLASSES)) / len(targets))
    rmse = float(torch.sqrt(torch.mean((predictions.float() - targets.float()) ** 2)).item())
    mae = float(torch.mean(torch.abs(predictions.float() - targets.float())).item())
    ccc = calculate_ccc(predictions, targets)
    return {"accuracy": accuracy, "micro_f1": accuracy, "macro_f1": macro_f1, "weighted_f1": weighted_f1, "ccc": ccc, "rmse": rmse, "mae": mae, "confusion_matrix": confusion_matrix}

In [52]:
def run_video_epoch(model, data_loader, question_index, criterion, optimizer=None):
    is_training = optimizer is not None
    if is_training:
        model.train()
    else:
        model.eval()
    total_loss = 0.0
    total_samples = 0
    prediction_list = []
    target_list = []
    for video_features, video_masks, all_labels in data_loader:
        video_features = video_features.to(device)
        video_masks = video_masks.to(device)
        targets = all_labels[:, question_index].to(device)
        if is_training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(is_training):
            logits = model(video_features, video_masks)
            loss = criterion(logits, targets)
            if is_training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                optimizer.step()
        batch_size = targets.shape[0]
        total_loss += loss.item() * batch_size
        total_samples += batch_size
        prediction_list.append(logits.argmax(dim=1).detach().cpu())
        target_list.append(targets.detach().cpu())
    predictions = torch.cat(prediction_list)
    targets = torch.cat(target_list)
    metrics = calculate_video_metrics(predictions, targets)
    metrics["loss"] = total_loss / total_samples
    metrics["predictions"] = predictions
    metrics["targets"] = targets
    return metrics

In [53]:
set_seed(42)
video_train_loader, video_val_loader = create_video_dataloaders(seed=42)
video_q1_model = VideoOnlyQuestModel().to(device)
q1_train_labels = video_train_dataset.labels[:, 0]
q1_class_counts = torch.bincount(q1_train_labels, minlength=NUM_CLASSES)
q1_class_weights = torch.pow(len(q1_train_labels) / q1_class_counts.float(), BETA)
q1_criterion = ImbOLLLoss(q1_class_weights, alpha=ALPHA).to(device)
video_q1_optimizer = torch.optim.AdamW(video_q1_model.parameters(), lr=LEARNING_RATE, eps=ADAM_EPSILON, weight_decay=WEIGHT_DECAY)
video_q1_train_metrics = run_video_epoch(video_q1_model, video_train_loader, question_index=0, criterion=q1_criterion, optimizer=video_q1_optimizer)
video_q1_val_metrics = run_video_epoch(video_q1_model, video_val_loader, question_index=0, criterion=q1_criterion)

In [54]:
print(f"一轮训练 Loss：{video_q1_train_metrics['loss']:.6f}")
print(f"一轮训练 Accuracy：{video_q1_train_metrics['accuracy']:.6f}")
print(f"一轮训练 Micro F1：{video_q1_train_metrics['micro_f1']:.6f}")
print(f"一轮训练 Macro F1：{video_q1_train_metrics['macro_f1']:.6f}")
print(f"一轮训练 Weighted F1：{video_q1_train_metrics['weighted_f1']:.6f}")
print(f"一轮训练 CCC：{video_q1_train_metrics['ccc']:.6f}")
print(f"一轮训练 RMSE：{video_q1_train_metrics['rmse']:.6f}")
print(f"一轮训练 MAE：{video_q1_train_metrics['mae']:.6f}")
print("一轮训练混淆矩阵：")
print(video_q1_train_metrics["confusion_matrix"])
print(f"一轮验证 Loss：{video_q1_val_metrics['loss']:.6f}")
print(f"一轮验证 Accuracy：{video_q1_val_metrics['accuracy']:.6f}")
print(f"一轮验证 Micro F1：{video_q1_val_metrics['micro_f1']:.6f}")
print(f"一轮验证 Macro F1：{video_q1_val_metrics['macro_f1']:.6f}")
print(f"一轮验证 Weighted F1：{video_q1_val_metrics['weighted_f1']:.6f}")
print(f"一轮验证 CCC：{video_q1_val_metrics['ccc']:.6f}")
print(f"一轮验证 RMSE：{video_q1_val_metrics['rmse']:.6f}")
print(f"一轮验证 MAE：{video_q1_val_metrics['mae']:.6f}")
print("一轮验证混淆矩阵：")
print(video_q1_val_metrics["confusion_matrix"])

一轮训练 Loss：2.385463
一轮训练 Accuracy：0.398773
一轮训练 Micro F1：0.398773
一轮训练 Macro F1：0.244706
一轮训练 Weighted F1：0.355641
一轮训练 CCC：-0.026707
一轮训练 RMSE：0.959294
一轮训练 MAE：0.699386
一轮训练混淆矩阵：
tensor([[20, 59,  2,  0],
        [12, 43,  2,  0],
        [ 7, 11,  2,  0],
        [ 2,  3,  0,  0]])
一轮验证 Loss：2.537260
一轮验证 Accuracy：0.381818
一轮验证 Micro F1：0.381818
一轮验证 Macro F1：0.261029
一轮验证 Weighted F1：0.394920
一轮验证 CCC：0.237715
一轮验证 RMSE：1.095445
一轮验证 MAE：0.800000
一轮验证混淆矩阵：
tensor([[14,  5,  6,  0],
        [ 7,  5, 10,  0],
        [ 1,  1,  2,  0],
        [ 1,  1,  2,  0]])


In [56]:
import time
def compact_video_metrics(metrics):
    return {"loss": float(metrics["loss"]), "accuracy": float(metrics["accuracy"]), "micro_f1": float(metrics["micro_f1"]), "macro_f1": float(metrics["macro_f1"]), "weighted_f1": float(metrics["weighted_f1"]), "ccc": float(metrics["ccc"]), "rmse": float(metrics["rmse"]), "mae": float(metrics["mae"]), "confusion_matrix": metrics["confusion_matrix"].tolist()}

In [57]:
def save_video_checkpoint(path, checkpoint_type, epoch, question_index, seed, model, optimizer, train_metrics, val_metrics, class_counts, class_weights, best_val_loss, best_val_ccc, best_loss_epoch, best_ccc_epoch):
    checkpoint = {}
    checkpoint["checkpoint_type"] = checkpoint_type
    checkpoint["epoch"] = epoch
    checkpoint["question_index"] = question_index
    checkpoint["question_number"] = question_index + 1
    checkpoint["question"] = PHQ8_QUESTION_COLUMNS[question_index]
    checkpoint["seed"] = seed
    checkpoint["model_state_dict"] = model.state_dict()
    checkpoint["optimizer_state_dict"] = optimizer.state_dict()
    checkpoint["train_metrics"] = compact_video_metrics(train_metrics)
    checkpoint["val_metrics"] = compact_video_metrics(val_metrics)
    checkpoint["class_counts"] = class_counts.tolist()
    checkpoint["class_weights"] = class_weights.tolist()
    checkpoint["best_val_loss"] = float(best_val_loss)
    checkpoint["best_val_ccc"] = float(best_val_ccc)
    checkpoint["best_loss_epoch"] = int(best_loss_epoch)
    checkpoint["best_ccc_epoch"] = int(best_ccc_epoch)
    checkpoint["config"] = {"max_turns": MAX_TURNS, "video_feature_dim": VIDEO_FEATURE_DIM, "lstm_hidden_size": 50, "encoder_output_dim": 100, "attention_heads": ATTENTION_HEADS, "attention_dropout": 0.2, "mlp_dropout": 0.2, "num_classes": NUM_CLASSES, "batch_size": BATCH_SIZE, "num_epochs": NUM_EPOCHS, "learning_rate": LEARNING_RATE, "adam_epsilon": ADAM_EPSILON, "weight_decay": WEIGHT_DECAY, "max_grad_norm": MAX_GRAD_NORM, "alpha": ALPHA, "beta": BETA}
    torch.save(checkpoint, path)

In [58]:
def train_video_question(seed, question_index):
    set_seed(seed)
    train_loader, val_loader = create_video_dataloaders(seed)
    model = VideoOnlyQuestModel().to(device)
    train_labels = video_train_dataset.labels[:, question_index]
    class_counts = torch.bincount(train_labels, minlength=NUM_CLASSES)
    class_weights = torch.pow(len(train_labels) / class_counts.float(), BETA)
    criterion = ImbOLLLoss(class_weights, alpha=ALPHA).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, eps=ADAM_EPSILON, weight_decay=WEIGHT_DECAY)
    checkpoint_dir = VIDEO_CHECKPOINT_ROOT / f"seed_{seed}" / f"question_{question_index + 1}"
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    best_loss_path = checkpoint_dir / "best_loss.pt"
    best_ccc_path = checkpoint_dir / "best_ccc.pt"
    last_path = checkpoint_dir / "last.pt"
    history_path = checkpoint_dir / "history.csv"
    best_val_loss = float("inf")
    best_val_ccc = float("-inf")
    best_loss_epoch = -1
    best_ccc_epoch = -1
    history_rows = []
    print("=" * 100)
    print(f"开始训练：Seed={seed}，Q{question_index + 1}，{PHQ8_QUESTION_COLUMNS[question_index]}")
    print("类别数量：", class_counts.tolist())
    print("ImbOLL权重：", class_weights.tolist())
    print("=" * 100)
    for epoch in range(1, NUM_EPOCHS + 1):
        epoch_start_time = time.perf_counter()
        train_metrics = run_video_epoch(model, train_loader, question_index, criterion, optimizer)
        val_metrics = run_video_epoch(model, val_loader, question_index, criterion)
        elapsed_seconds = time.perf_counter() - epoch_start_time
        improved_loss = val_metrics["loss"] < best_val_loss
        improved_ccc = val_metrics["ccc"] > best_val_ccc
        if improved_loss:
            best_val_loss = val_metrics["loss"]
            best_loss_epoch = epoch
        if improved_ccc:
            best_val_ccc = val_metrics["ccc"]
            best_ccc_epoch = epoch
        if improved_loss:
            save_video_checkpoint(best_loss_path, "best_loss", epoch, question_index, seed, model, optimizer, train_metrics, val_metrics, class_counts, class_weights, best_val_loss, best_val_ccc, best_loss_epoch, best_ccc_epoch)
        if improved_ccc:
            save_video_checkpoint(best_ccc_path, "best_ccc", epoch, question_index, seed, model, optimizer, train_metrics, val_metrics, class_counts, class_weights, best_val_loss, best_val_ccc, best_loss_epoch, best_ccc_epoch)
        history_rows.append({"Epoch": epoch, "Train_Loss": train_metrics["loss"], "Train_Accuracy": train_metrics["accuracy"], "Train_CCC": train_metrics["ccc"], "Train_RMSE": train_metrics["rmse"], "Train_MAE": train_metrics["mae"], "Val_Loss": val_metrics["loss"], "Val_Accuracy": val_metrics["accuracy"], "Val_CCC": val_metrics["ccc"], "Val_RMSE": val_metrics["rmse"], "Val_MAE": val_metrics["mae"], "Elapsed_Seconds": elapsed_seconds})
        print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | Train Loss {train_metrics['loss']:.6f} | Val Loss {val_metrics['loss']:.6f} | Val CCC {val_metrics['ccc']:.6f} | Val Acc {val_metrics['accuracy']:.6f} | Val RMSE {val_metrics['rmse']:.6f} | {elapsed_seconds:.2f}s")
    save_video_checkpoint(last_path, "last", NUM_EPOCHS, question_index, seed, model, optimizer, train_metrics, val_metrics, class_counts, class_weights, best_val_loss, best_val_ccc, best_loss_epoch, best_ccc_epoch)
    history_df = pd.DataFrame(history_rows)
    history_df.to_csv(history_path, index=False)
    print("=" * 100)
    print(f"训练完成：Seed={seed}，Q{question_index + 1}")
    print(f"最佳Val Loss：{best_val_loss:.6f}，Epoch={best_loss_epoch}")
    print(f"最佳Val CCC：{best_val_ccc:.6f}，Epoch={best_ccc_epoch}")
    print("=" * 100)
    return {"seed": seed, "question_index": question_index, "question_number": question_index + 1, "question": PHQ8_QUESTION_COLUMNS[question_index], "best_val_loss": best_val_loss, "best_val_ccc": best_val_ccc, "best_loss_epoch": best_loss_epoch, "best_ccc_epoch": best_ccc_epoch, "best_loss_path": best_loss_path, "best_ccc_path": best_ccc_path, "last_path": last_path, "history_path": history_path}

In [59]:
video_q1_seed42_result = train_video_question(seed=42, question_index=0)

开始训练：Seed=42，Q1，PHQ_8NoInterest
类别数量： [81, 57, 20, 5]
ImbOLL权重： [1.4185717105865479, 1.6910496950149536, 2.854820728302002, 5.709641456604004]
Epoch 01/50 | Train Loss 2.385463 | Val Loss 2.537260 | Val CCC 0.237715 | Val Acc 0.381818 | Val RMSE 1.095445 | 1.06s
Epoch 02/50 | Train Loss 2.096551 | Val Loss 2.502337 | Val CCC 0.272326 | Val Acc 0.490909 | Val RMSE 0.934199 | 0.66s
Epoch 03/50 | Train Loss 2.107416 | Val Loss 2.646679 | Val CCC 0.259259 | Val Acc 0.436364 | Val RMSE 0.934199 | 0.72s
Epoch 04/50 | Train Loss 2.042489 | Val Loss 2.551044 | Val CCC 0.220674 | Val Acc 0.418182 | Val RMSE 1.053134 | 0.60s
Epoch 05/50 | Train Loss 2.013167 | Val Loss 2.517005 | Val CCC 0.210687 | Val Acc 0.509091 | Val RMSE 0.924416 | 0.51s
Epoch 06/50 | Train Loss 2.075038 | Val Loss 2.546816 | Val CCC 0.000000 | Val Acc 0.400000 | Val RMSE 0.904534 | 0.66s
Epoch 07/50 | Train Loss 2.046445 | Val Loss 2.679875 | Val CCC 0.024160 | Val Acc 0.454545 | Val RMSE 0.924416 | 0.53s
Epoch 08/50 | Tra

In [60]:
print("Best Loss存在：", video_q1_seed42_result["best_loss_path"].exists())
print("Best CCC存在：", video_q1_seed42_result["best_ccc_path"].exists())
print("Last存在：", video_q1_seed42_result["last_path"].exists())
print("History存在：", video_q1_seed42_result["history_path"].exists())
print("Best Loss路径：", video_q1_seed42_result["best_loss_path"])
print("Best CCC路径：", video_q1_seed42_result["best_ccc_path"])

Best Loss存在： True
Best CCC存在： True
Last存在： True
History存在： True
Best Loss路径： /workspace/E-DAIC/checkpoints/phq8_video_paperlike/seed_42/question_1/best_loss.pt
Best CCC路径： /workspace/E-DAIC/checkpoints/phq8_video_paperlike/seed_42/question_1/best_ccc.pt


In [61]:
video_q1_best_ccc_path = video_q1_seed42_result["best_ccc_path"]
video_q1_checkpoint = torch.load(video_q1_best_ccc_path, map_location="cpu", weights_only=False)
restored_video_q1_model = VideoOnlyQuestModel().to(device)
load_result = restored_video_q1_model.load_state_dict(video_q1_checkpoint["model_state_dict"], strict=True)
restored_q1_class_weights = torch.tensor(video_q1_checkpoint["class_weights"], dtype=torch.float32)
restored_q1_criterion = ImbOLLLoss(restored_q1_class_weights, alpha=ALPHA).to(device)
_, restored_video_val_loader = create_video_dataloaders(seed=42)
restored_q1_val_metrics = run_video_epoch(restored_video_q1_model, restored_video_val_loader, question_index=0, criterion=restored_q1_criterion)
print("Checkpoint：", video_q1_best_ccc_path)
print("类型：", video_q1_checkpoint["checkpoint_type"])
print("Epoch：", video_q1_checkpoint["epoch"])
print("缺少参数：", load_result.missing_keys)
print("多余参数：", load_result.unexpected_keys)

Checkpoint： /workspace/E-DAIC/checkpoints/phq8_video_paperlike/seed_42/question_1/best_ccc.pt
类型： best_ccc
Epoch： 2
缺少参数： []
多余参数： []


In [62]:
checkpoint_metric_names = ["loss", "accuracy", "micro_f1", "macro_f1", "weighted_f1", "ccc", "rmse", "mae"]
print("保存指标与恢复指标对比：")
for metric_name in checkpoint_metric_names:
    saved_value = float(video_q1_checkpoint["val_metrics"][metric_name])
    restored_value = float(restored_q1_val_metrics[metric_name])
    difference = abs(saved_value - restored_value)
    print(f"{metric_name:>12} | 保存 {saved_value:.9f} | 恢复 {restored_value:.9f} | 差值 {difference:.3e}")

保存指标与恢复指标对比：
        loss | 保存 2.502336784 | 恢复 2.502336784 | 差值 0.000e+00
    accuracy | 保存 0.490909100 | 恢复 0.490909100 | 差值 0.000e+00
    micro_f1 | 保存 0.490909100 | 恢复 0.490909100 | 差值 0.000e+00
    macro_f1 | 保存 0.320833333 | 恢复 0.320833333 | 差值 0.000e+00
 weighted_f1 | 保存 0.479696970 | 恢复 0.479696970 | 差值 0.000e+00
         ccc | 保存 0.272326351 | 恢复 0.272326351 | 差值 0.000e+00
        rmse | 保存 0.934198737 | 恢复 0.934198737 | 差值 0.000e+00
         mae | 保存 0.618181825 | 恢复 0.618181825 | 差值 0.000e+00


In [63]:
maximum_metric_difference = max(abs(float(video_q1_checkpoint["val_metrics"][metric_name]) - float(restored_q1_val_metrics[metric_name])) for metric_name in checkpoint_metric_names)
assert load_result.missing_keys == []
assert load_result.unexpected_keys == []
assert maximum_metric_difference < 1e-6
assert torch.equal(torch.tensor(video_q1_checkpoint["val_metrics"]["confusion_matrix"]), restored_q1_val_metrics["confusion_matrix"])
print("最大指标差值：", maximum_metric_difference)
print("Checkpoint恢复验证通过")

最大指标差值： 0.0
Checkpoint恢复验证通过


In [64]:
import gc

In [65]:
video_seed42_results = [video_q1_seed42_result]
for question_index in range(1, len(PHQ8_QUESTION_COLUMNS)):
    question_result = train_video_question(seed=42, question_index=question_index)
    video_seed42_results.append(question_result)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

开始训练：Seed=42，Q2，PHQ_8Depressed
类别数量： [68, 63, 18, 14]
ImbOLL权重： [1.548243761062622, 1.6085091829299927, 3.009244918823242, 3.412163257598877]
Epoch 01/50 | Train Loss 2.506367 | Val Loss 2.334554 | Val CCC 0.141844 | Val Acc 0.363636 | Val RMSE 1.000000 | 0.75s
Epoch 02/50 | Train Loss 2.337957 | Val Loss 2.340672 | Val CCC 0.096051 | Val Acc 0.327273 | Val RMSE 1.183216 | 0.63s
Epoch 03/50 | Train Loss 2.313057 | Val Loss 2.353148 | Val CCC 0.061805 | Val Acc 0.272727 | Val RMSE 1.120065 | 0.55s
Epoch 04/50 | Train Loss 2.304281 | Val Loss 2.311789 | Val CCC 0.224460 | Val Acc 0.418182 | Val RMSE 0.943880 | 0.63s
Epoch 05/50 | Train Loss 2.229037 | Val Loss 2.381508 | Val CCC 0.126984 | Val Acc 0.272727 | Val RMSE 1.000000 | 0.48s
Epoch 06/50 | Train Loss 2.328542 | Val Loss 2.334506 | Val CCC 0.000000 | Val Acc 0.345455 | Val RMSE 0.934199 | 0.59s
Epoch 07/50 | Train Loss 2.265647 | Val Loss 2.409309 | Val CCC 0.151235 | Val Acc 0.309091 | Val RMSE 0.953463 | 0.50s
Epoch 08/50 | Trai

In [66]:
print("Seed 42结果数量：", len(video_seed42_results))
print("Best Loss全部存在：", all(result["best_loss_path"].exists() for result in video_seed42_results))
print("Best CCC全部存在：", all(result["best_ccc_path"].exists() for result in video_seed42_results))
print("Last全部存在：", all(result["last_path"].exists() for result in video_seed42_results))
print("History全部存在：", all(result["history_path"].exists() for result in video_seed42_results))

Seed 42结果数量： 8
Best Loss全部存在： True
Best CCC全部存在： True
Last全部存在： True
History全部存在： True


In [67]:
video_seed42_summary_rows = []
for result in video_seed42_results:
    video_seed42_summary_rows.append({"Seed": result["seed"], "Question_Index": result["question_index"], "Question_Number": result["question_number"], "Question_Name": result["question"], "Best_Val_Loss": result["best_val_loss"], "Best_Loss_Epoch": result["best_loss_epoch"], "Best_Val_CCC": result["best_val_ccc"], "Best_CCC_Epoch": result["best_ccc_epoch"]})
video_seed42_summary_df = pd.DataFrame(video_seed42_summary_rows)
video_seed42_summary_path = VIDEO_CHECKPOINT_ROOT / "seed_42_question_summary.csv"
video_seed42_summary_df.to_csv(video_seed42_summary_path, index=False)
display(video_seed42_summary_df)
print("汇总文件：", video_seed42_summary_path)

,Seed,Question_Index,Question_Number,Question_Name,Best_Val_Loss,Best_Loss_Epoch,Best_Val_CCC,Best_CCC_Epoch
0,42,0,1,PHQ_8NoInterest,2.502337,2,0.272326,2
1,42,1,2,PHQ_8Depressed,2.309061,8,0.224460,4
2,42,2,3,PHQ_8Sleep,2.410559,14,0.309804,21
3,42,3,4,PHQ_8Tired,2.385631,7,0.340570,40
4,42,4,5,PHQ_8Appetite,2.233981,2,0.144876,27
5,42,5,6,PHQ_8Failure,2.260065,24,0.429989,41
6,42,6,7,PHQ_8Concentrating,2.333310,1,0.188525,9
7,42,7,8,PHQ_8Moving,1.807733,2,0.226757,10


汇总文件： /workspace/E-DAIC/checkpoints/phq8_video_paperlike/seed_42_question_summary.csv


In [68]:
video_seed42_best_ccc_rows = []
for result in video_seed42_results:
    checkpoint = torch.load(result["best_ccc_path"], map_location="cpu", weights_only=False)
    video_seed42_best_ccc_rows.append({"Question_Number": checkpoint["question_number"], "Question_Name": checkpoint["question"], "Checkpoint_Epoch": checkpoint["epoch"], "CCC": checkpoint["val_metrics"]["ccc"], "RMSE": checkpoint["val_metrics"]["rmse"], "MAE": checkpoint["val_metrics"]["mae"], "Accuracy": checkpoint["val_metrics"]["accuracy"]})
video_seed42_best_ccc_df = pd.DataFrame(video_seed42_best_ccc_rows)
display(video_seed42_best_ccc_df)

,Question_Number,Question_Name,Checkpoint_Epoch,CCC,RMSE,MAE,Accuracy
0,1,PHQ_8NoInterest,2,0.272326,0.934199,0.618182,0.490909
1,2,PHQ_8Depressed,4,0.224460,0.943880,0.672727,0.418182
2,3,PHQ_8Sleep,21,0.309804,1.078720,0.800000,0.381818
3,4,PHQ_8Tired,40,0.340570,1.159937,0.836364,0.381818
4,5,PHQ_8Appetite,27,0.144876,1.264911,0.981818,0.290909
5,6,PHQ_8Failure,41,0.429989,0.924416,0.672727,0.418182
6,7,PHQ_8Concentrating,9,0.188525,0.990867,0.727273,0.400000
7,8,PHQ_8Moving,10,0.226757,0.750757,0.527273,0.490909


In [69]:
def evaluate_video_seed_on_validation(seed):
    val_loader = DataLoader(video_val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, drop_last=False)
    question_prediction_list = []
    selected_checkpoint_rows = []
    for question_index in range(len(PHQ8_QUESTION_COLUMNS)):
        checkpoint_path = VIDEO_CHECKPOINT_ROOT / f"seed_{seed}" / f"question_{question_index + 1}" / "best_ccc.pt"
        if not checkpoint_path.exists():
            raise FileNotFoundError(f"Checkpoint不存在：{checkpoint_path}")
        checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
        if checkpoint["seed"] != seed:
            raise RuntimeError(f"Checkpoint Seed错误：{checkpoint_path}")
        if checkpoint["question_index"] != question_index:
            raise RuntimeError(f"Checkpoint题号错误：{checkpoint_path}")
        if checkpoint["checkpoint_type"] != "best_ccc":
            raise RuntimeError(f"Checkpoint类型错误：{checkpoint_path}")
        model = VideoOnlyQuestModel().to(device)
        load_result = model.load_state_dict(checkpoint["model_state_dict"], strict=True)
        if load_result.missing_keys or load_result.unexpected_keys:
            raise RuntimeError(f"Checkpoint参数不一致：{checkpoint_path}")
        model.eval()
        batch_prediction_list = []
        with torch.no_grad():
            for video_features, video_masks, all_labels in val_loader:
                logits = model(video_features.to(device), video_masks.to(device))
                predictions = logits.argmax(dim=1)
                batch_prediction_list.append(predictions.cpu())
        question_predictions = torch.cat(batch_prediction_list)
        if len(question_predictions) != len(video_val_dataset):
            raise RuntimeError(f"Q{question_index + 1}预测数量错误")
        question_prediction_list.append(question_predictions)
        selected_checkpoint_rows.append({"Seed": seed, "Question_Index": question_index, "Question_Number": question_index + 1, "Question_Name": PHQ8_QUESTION_COLUMNS[question_index], "Checkpoint_Path": str(checkpoint_path), "Checkpoint_Epoch": checkpoint["epoch"], "Val_CCC": checkpoint["val_metrics"]["ccc"]})
        print(f"Q{question_index + 1} | Epoch {checkpoint['epoch']:02d} | Val CCC {checkpoint['val_metrics']['ccc']:.6f}")
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    question_predictions = torch.stack(question_prediction_list, dim=1)
    question_targets = video_val_dataset.labels.clone()
    predicted_totals = question_predictions.sum(dim=1)
    true_totals = question_targets.sum(dim=1)
    total_ccc = calculate_ccc(predicted_totals, true_totals)
    total_rmse = float(torch.sqrt(torch.mean((predicted_totals.float() - true_totals.float()) ** 2)).item())
    total_mae = float(torch.mean(torch.abs(predicted_totals.float() - true_totals.float())).item())
    exact_accuracy = float((predicted_totals == true_totals).float().mean().item())
    prediction_df = pd.DataFrame({"Participant_ID": video_val_dataset.participant_ids})
    for question_index in range(len(PHQ8_QUESTION_COLUMNS)):
        prediction_df[f"Q{question_index + 1}_Predicted"] = question_predictions[:, question_index].numpy()
        prediction_df[f"Q{question_index + 1}_True"] = question_targets[:, question_index].numpy()
    prediction_df["Predicted_Total"] = predicted_totals.numpy()
    prediction_df["True_Total"] = true_totals.numpy()
    prediction_path = VIDEO_CHECKPOINT_ROOT / f"seed_{seed}_validation_predictions.csv"
    selected_checkpoint_path = VIDEO_CHECKPOINT_ROOT / f"seed_{seed}_validation_selected_checkpoints.csv"
    prediction_df.to_csv(prediction_path, index=False)
    pd.DataFrame(selected_checkpoint_rows).to_csv(selected_checkpoint_path, index=False)
    print("=" * 70)
    print("验证参与者数量：", len(prediction_df))
    print(f"总分CCC：{total_ccc:.6f}")
    print(f"总分RMSE：{total_rmse:.6f}")
    print(f"总分MAE：{total_mae:.6f}")
    print(f"完全相等比例：{exact_accuracy:.6f}")
    print(f"预测总分范围：{int(predicted_totals.min())}～{int(predicted_totals.max())}")
    print(f"真实总分范围：{int(true_totals.min())}～{int(true_totals.max())}")
    print("预测文件：", prediction_path)
    return {"seed": seed, "sample_count": len(prediction_df), "ccc": total_ccc, "rmse": total_rmse, "mae": total_mae, "exact_accuracy": exact_accuracy, "question_predictions": question_predictions, "question_targets": question_targets, "predicted_totals": predicted_totals, "true_totals": true_totals, "prediction_path": prediction_path, "selected_checkpoint_path": selected_checkpoint_path}

In [70]:
video_seed42_validation_result = evaluate_video_seed_on_validation(seed=42)

Q1 | Epoch 02 | Val CCC 0.272326
Q2 | Epoch 04 | Val CCC 0.224460
Q3 | Epoch 21 | Val CCC 0.309804
Q4 | Epoch 40 | Val CCC 0.340570
Q5 | Epoch 27 | Val CCC 0.144876
Q6 | Epoch 41 | Val CCC 0.429989
Q7 | Epoch 09 | Val CCC 0.188525
Q8 | Epoch 10 | Val CCC 0.226757
验证参与者数量： 55
总分CCC：0.356659
总分RMSE：5.642372
总分MAE：4.600000
完全相等比例：0.090909
预测总分范围：0～16
真实总分范围：0～20
预测文件： /workspace/E-DAIC/checkpoints/phq8_video_paperlike/seed_42_validation_predictions.csv


In [71]:
def train_all_video_questions_for_seed(seed):
    seed_results = []
    print("#" * 100)
    print(f"开始训练Video-only全部问题：Seed={seed}")
    print("#" * 100)
    for question_index in range(len(PHQ8_QUESTION_COLUMNS)):
        result = train_video_question(seed=seed, question_index=question_index)
        seed_results.append(result)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    summary_rows = []
    for result in seed_results:
        summary_rows.append({"Seed": result["seed"], "Question_Index": result["question_index"], "Question_Number": result["question_number"], "Question_Name": result["question"], "Best_Val_Loss": result["best_val_loss"], "Best_Loss_Epoch": result["best_loss_epoch"], "Best_Val_CCC": result["best_val_ccc"], "Best_CCC_Epoch": result["best_ccc_epoch"]})
    summary_df = pd.DataFrame(summary_rows)
    summary_path = VIDEO_CHECKPOINT_ROOT / f"seed_{seed}_question_summary.csv"
    summary_df.to_csv(summary_path, index=False)
    print(f"Seed {seed}全部问题训练完成")
    print("汇总文件：", summary_path)
    return seed_results, summary_df

In [72]:
video_seed100_results, video_seed100_summary_df = train_all_video_questions_for_seed(seed=100)
display(video_seed100_summary_df)

####################################################################################################
开始训练Video-only全部问题：Seed=100
####################################################################################################
开始训练：Seed=100，Q1，PHQ_8NoInterest
类别数量： [81, 57, 20, 5]
ImbOLL权重： [1.4185717105865479, 1.6910496950149536, 2.854820728302002, 5.709641456604004]
Epoch 01/50 | Train Loss 2.276058 | Val Loss 2.608974 | Val CCC 0.170616 | Val Acc 0.436364 | Val RMSE 1.128152 | 0.64s
Epoch 02/50 | Train Loss 2.078172 | Val Loss 2.485649 | Val CCC 0.000000 | Val Acc 0.400000 | Val RMSE 0.904534 | 0.74s
Epoch 03/50 | Train Loss 2.134733 | Val Loss 2.543665 | Val CCC 0.195122 | Val Acc 0.436364 | Val RMSE 0.934199 | 0.73s
Epoch 04/50 | Train Loss 2.041648 | Val Loss 2.625105 | Val CCC -0.030445 | Val Acc 0.436364 | Val RMSE 0.934199 | 0.50s
Epoch 05/50 | Train Loss 2.009647 | Val Loss 2.560080 | Val CCC 0.151473 | Val Acc 0.509091 | Val RMSE 0.894427 | 0.50s
Epoch 06/50 | Train Loss 

,Seed,Question_Index,Question_Number,Question_Name,Best_Val_Loss,Best_Loss_Epoch,Best_Val_CCC,Best_CCC_Epoch
0,100,0,1,PHQ_8NoInterest,2.460209,14,0.279613,6
1,100,1,2,PHQ_8Depressed,2.284547,2,0.212202,4
2,100,2,3,PHQ_8Sleep,2.419215,9,0.319661,45
3,100,3,4,PHQ_8Tired,2.383127,6,0.308466,21
4,100,4,5,PHQ_8Appetite,2.235897,3,0.132492,22
5,100,5,6,PHQ_8Failure,2.278898,3,0.478824,43
6,100,6,7,PHQ_8Concentrating,2.345040,2,0.220881,33
7,100,7,8,PHQ_8Moving,1.793559,9,0.135560,3


In [73]:
video_seed1234_results, video_seed1234_summary_df = train_all_video_questions_for_seed(seed=1234)
display(video_seed1234_summary_df)

####################################################################################################
开始训练Video-only全部问题：Seed=1234
####################################################################################################
开始训练：Seed=1234，Q1，PHQ_8NoInterest
类别数量： [81, 57, 20, 5]
ImbOLL权重： [1.4185717105865479, 1.6910496950149536, 2.854820728302002, 5.709641456604004]
Epoch 01/50 | Train Loss 2.263757 | Val Loss 2.502578 | Val CCC 0.107366 | Val Acc 0.327273 | Val RMSE 0.972345 | 0.71s
Epoch 02/50 | Train Loss 2.048276 | Val Loss 2.692895 | Val CCC 0.165376 | Val Acc 0.527273 | Val RMSE 0.943880 | 0.75s
Epoch 03/50 | Train Loss 2.107237 | Val Loss 2.505362 | Val CCC 0.034351 | Val Acc 0.472727 | Val RMSE 0.914529 | 0.62s
Epoch 04/50 | Train Loss 2.024514 | Val Loss 2.530042 | Val CCC 0.200238 | Val Acc 0.418182 | Val RMSE 1.053134 | 0.53s
Epoch 05/50 | Train Loss 2.006978 | Val Loss 2.540952 | Val CCC 0.169611 | Val Acc 0.509091 | Val RMSE 0.924416 | 0.64s
Epoch 06/50 | Train Loss

,Seed,Question_Index,Question_Number,Question_Name,Best_Val_Loss,Best_Loss_Epoch,Best_Val_CCC,Best_CCC_Epoch
0,1234,0,1,PHQ_8NoInterest,2.452970,15,0.331347,26
1,1234,1,2,PHQ_8Depressed,2.291186,2,0.195862,42
2,1234,2,3,PHQ_8Sleep,2.424653,13,0.384895,27
3,1234,3,4,PHQ_8Tired,2.391727,9,0.273231,43
4,1234,4,5,PHQ_8Appetite,2.215777,2,0.213037,7
5,1234,5,6,PHQ_8Failure,2.234637,42,0.442278,49
6,1234,6,7,PHQ_8Concentrating,2.295482,1,0.213162,36
7,1234,7,8,PHQ_8Moving,1.867959,5,0.111516,2


In [74]:
all_video_training_results = {}
all_video_training_results[42] = video_seed42_results
all_video_training_results[100] = video_seed100_results
all_video_training_results[1234] = video_seed1234_results

In [75]:
for seed, seed_results in all_video_training_results.items():
    print("=" * 70)
    print("Seed：", seed)
    print("问题数量：", len(seed_results))
    print("Best Loss全部存在：", all(result["best_loss_path"].exists() for result in seed_results))
    print("Best CCC全部存在：", all(result["best_ccc_path"].exists() for result in seed_results))
    print("Last全部存在：", all(result["last_path"].exists() for result in seed_results))
    print("History全部存在：", all(result["history_path"].exists() for result in seed_results))

Seed： 42
问题数量： 8
Best Loss全部存在： True
Best CCC全部存在： True
Last全部存在： True
History全部存在： True
Seed： 100
问题数量： 8
Best Loss全部存在： True
Best CCC全部存在： True
Last全部存在： True
History全部存在： True
Seed： 1234
问题数量： 8
Best Loss全部存在： True
Best CCC全部存在： True
Last全部存在： True
History全部存在： True


In [76]:
video_seed100_validation_result = evaluate_video_seed_on_validation(seed=100)
video_seed1234_validation_result = evaluate_video_seed_on_validation(seed=1234)

Q1 | Epoch 06 | Val CCC 0.279613
Q2 | Epoch 04 | Val CCC 0.212202
Q3 | Epoch 45 | Val CCC 0.319661
Q4 | Epoch 21 | Val CCC 0.308466
Q5 | Epoch 22 | Val CCC 0.132492
Q6 | Epoch 43 | Val CCC 0.478824
Q7 | Epoch 33 | Val CCC 0.220881
Q8 | Epoch 03 | Val CCC 0.135560
验证参与者数量： 55
总分CCC：0.360905
总分RMSE：5.623004
总分MAE：4.600000
完全相等比例：0.054545
预测总分范围：0～16
真实总分范围：0～20
预测文件： /workspace/E-DAIC/checkpoints/phq8_video_paperlike/seed_100_validation_predictions.csv
Q1 | Epoch 26 | Val CCC 0.331347
Q2 | Epoch 42 | Val CCC 0.195862
Q3 | Epoch 27 | Val CCC 0.384895
Q4 | Epoch 43 | Val CCC 0.273231
Q5 | Epoch 07 | Val CCC 0.213037
Q6 | Epoch 49 | Val CCC 0.442278
Q7 | Epoch 36 | Val CCC 0.213162
Q8 | Epoch 02 | Val CCC 0.111516
验证参与者数量： 55
总分CCC：0.355454
总分RMSE：5.916080
总分MAE：4.927273
完全相等比例：0.054545
预测总分范围：0～18
真实总分范围：0～20
预测文件： /workspace/E-DAIC/checkpoints/phq8_video_paperlike/seed_1234_validation_predictions.csv


In [77]:
all_video_validation_results = [video_seed42_validation_result, video_seed100_validation_result, video_seed1234_validation_result]
video_validation_seed_rows = []
for result in all_video_validation_results:
    video_validation_seed_rows.append({"Seed": result["seed"], "Sample_Count": result["sample_count"], "CCC": result["ccc"], "RMSE": result["rmse"], "MAE": result["mae"], "Exact_Accuracy": result["exact_accuracy"], "Predicted_Min": int(result["predicted_totals"].min()), "Predicted_Max": int(result["predicted_totals"].max()), "True_Min": int(result["true_totals"].min()), "True_Max": int(result["true_totals"].max())})
video_validation_seed_summary_df = pd.DataFrame(video_validation_seed_rows)
display(video_validation_seed_summary_df)

,Seed,Sample_Count,CCC,RMSE,MAE,Exact_Accuracy,Predicted_Min,Predicted_Max,True_Min,True_Max
0,42,55,0.356659,5.642372,4.600000,0.090909,0,16,0,20
1,100,55,0.360905,5.623004,4.600000,0.054545,0,16,0,20
2,1234,55,0.355454,5.916080,4.927273,0.054545,0,18,0,20


In [78]:
video_validation_statistics_rows = []
for metric_name in ["CCC", "RMSE", "MAE", "Exact_Accuracy"]:
    video_validation_statistics_rows.append({"Metric": metric_name, "Mean": video_validation_seed_summary_df[metric_name].mean(), "Std": video_validation_seed_summary_df[metric_name].std(ddof=1)})
video_validation_statistics_df = pd.DataFrame(video_validation_statistics_rows)
display(video_validation_statistics_df)

,Metric,Mean,Std
0,CCC,0.357673,0.002863
1,RMSE,5.727152,0.163903
2,MAE,4.709091,0.188951
3,Exact_Accuracy,0.066667,0.020995


In [79]:
video_validation_seed_summary_path = VIDEO_CHECKPOINT_ROOT / "three_seed_validation_results.csv"
video_validation_statistics_path = VIDEO_CHECKPOINT_ROOT / "three_seed_validation_summary.csv"
video_validation_seed_summary_df.to_csv(video_validation_seed_summary_path, index=False)
video_validation_statistics_df.to_csv(video_validation_statistics_path, index=False)
print("逐Seed验证结果：", video_validation_seed_summary_path)
print("三Seed验证汇总：", video_validation_statistics_path)
print("逐Seed文件存在：", video_validation_seed_summary_path.exists())
print("汇总文件存在：", video_validation_statistics_path.exists())

逐Seed验证结果： /workspace/E-DAIC/checkpoints/phq8_video_paperlike/three_seed_validation_results.csv
三Seed验证汇总： /workspace/E-DAIC/checkpoints/phq8_video_paperlike/three_seed_validation_summary.csv
逐Seed文件存在： True
汇总文件存在： True


In [80]:
print("测试集字段：", video_test_metadata_df.columns.tolist())
print("测试参与者数量：", len(video_test_metadata_df))
print("测试集包含637：", 637 in video_test_metadata_df["Participant_ID"].astype(int).tolist())

测试集字段： ['Participant_ID', 'Gender', 'PHQ_Binary', 'PHQ_Score', 'PCL-C (PTSD)', 'PTSD Severity']
测试参与者数量： 56
测试集包含637： True


In [83]:
class VideoPHQ8TestDataset(Dataset):
    def __init__(self, metadata_df):
        super().__init__()
        self.metadata_df = metadata_df.reset_index(drop=True).copy()
        if "PHQ_Score" not in self.metadata_df.columns:
            raise KeyError(f"测试数据缺少PHQ8_Score字段，现有字段：{self.metadata_df.columns.tolist()}")
        self.participant_ids = self.metadata_df["Participant_ID"].astype(int).tolist()
        self.true_totals = torch.tensor(self.metadata_df["PHQ_Score"].to_numpy(), dtype=torch.long)
        video_feature_list = []
        video_mask_list = []
        for index, participant_id in enumerate(self.participant_ids):
            cache_path = VIDEO_CACHE_DIR / f"{participant_id}.pt"
            if not cache_path.exists():
                raise FileNotFoundError(f"Video测试缓存不存在：{cache_path}")
            cache = torch.load(cache_path, map_location="cpu", weights_only=False)
            video_features = cache["video_features"].float()
            video_padding_mask = cache["padding_mask"].bool()
            if int(cache["participant_id"]) != participant_id:
                raise RuntimeError(f"参与者编号错误：预期{participant_id}，实际{cache['participant_id']}")
            if video_features.shape != (MAX_TURNS, VIDEO_FEATURE_DIM):
                raise RuntimeError(f"参与者{participant_id}特征形状错误：{video_features.shape}")
            if video_padding_mask.shape != (MAX_TURNS,):
                raise RuntimeError(f"参与者{participant_id}Mask形状错误：{video_padding_mask.shape}")
            if not torch.isfinite(video_features).all():
                raise RuntimeError(f"参与者{participant_id}特征包含NaN或Inf")
            video_feature_list.append(video_features)
            video_mask_list.append(video_padding_mask)
            if (index + 1) % 20 == 0 or index + 1 == len(self.participant_ids):
                print(f"test Dataset加载进度：{index + 1}/{len(self.participant_ids)}")
        self.video_features = torch.stack(video_feature_list, dim=0)
        self.video_masks = torch.stack(video_mask_list, dim=0)
        if self.video_features.shape != (len(self.participant_ids), MAX_TURNS, VIDEO_FEATURE_DIM):
            raise RuntimeError(f"测试特征整体形状错误：{self.video_features.shape}")
        if self.video_masks.shape != (len(self.participant_ids), MAX_TURNS):
            raise RuntimeError(f"测试Mask整体形状错误：{self.video_masks.shape}")

    def __len__(self):
        return len(self.participant_ids)

    def __getitem__(self, index):
        return self.video_features[index], self.video_masks[index], self.true_totals[index], self.participant_ids[index]

In [84]:
video_test_dataset = VideoPHQ8TestDataset(video_test_metadata_df)
video_test_loader = DataLoader(video_test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, drop_last=False)
test_batch_features, test_batch_masks, test_batch_totals, test_batch_participant_ids = next(iter(video_test_loader))
print("测试参与者数量：", len(video_test_dataset))
print("测试批次数：", len(video_test_loader))
print("测试Batch特征：", test_batch_features.shape)
print("测试Batch Mask：", test_batch_masks.shape)
print("测试Batch真实总分：", test_batch_totals.shape)
print("测试Batch参与者：", test_batch_participant_ids)
print("测试特征全部有限：", bool(torch.isfinite(test_batch_features).all()))
print("测试真实总分范围：", int(video_test_dataset.true_totals.min()), "～", int(video_test_dataset.true_totals.max()))

test Dataset加载进度：20/56
test Dataset加载进度：40/56
test Dataset加载进度：56/56
测试参与者数量： 56
测试批次数： 6
测试Batch特征： torch.Size([10, 120, 2048])
测试Batch Mask： torch.Size([10, 120])
测试Batch真实总分： torch.Size([10])
测试Batch参与者： tensor([600, 602, 604, 605, 606, 607, 609, 615, 618, 619])
测试特征全部有限： True
测试真实总分范围： 0 ～ 22


In [85]:
def evaluate_video_seed_on_test(seed):
    question_prediction_list = []
    selected_checkpoint_rows = []
    for question_index in range(len(PHQ8_QUESTION_COLUMNS)):
        checkpoint_path = VIDEO_CHECKPOINT_ROOT / f"seed_{seed}" / f"question_{question_index + 1}" / "best_ccc.pt"
        if not checkpoint_path.exists():
            raise FileNotFoundError(f"Checkpoint不存在：{checkpoint_path}")
        checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
        if checkpoint["seed"] != seed:
            raise RuntimeError(f"Checkpoint Seed错误：{checkpoint_path}")
        if checkpoint["question_index"] != question_index:
            raise RuntimeError(f"Checkpoint题号错误：{checkpoint_path}")
        if checkpoint["checkpoint_type"] != "best_ccc":
            raise RuntimeError(f"Checkpoint类型错误：{checkpoint_path}")
        model = VideoOnlyQuestModel().to(device)
        load_result = model.load_state_dict(checkpoint["model_state_dict"], strict=True)
        if load_result.missing_keys or load_result.unexpected_keys:
            raise RuntimeError(f"Checkpoint参数不一致：{checkpoint_path}")
        model.eval()
        batch_prediction_list = []
        with torch.no_grad():
            for video_features, video_masks, true_totals, participant_ids in video_test_loader:
                logits = model(video_features.to(device), video_masks.to(device))
                batch_prediction_list.append(logits.argmax(dim=1).cpu())
        question_predictions = torch.cat(batch_prediction_list)
        if len(question_predictions) != len(video_test_dataset):
            raise RuntimeError(f"Q{question_index + 1}预测数量错误")
        question_prediction_list.append(question_predictions)
        selected_checkpoint_rows.append({"Seed": seed, "Question_Index": question_index, "Question_Number": question_index + 1, "Question_Name": PHQ8_QUESTION_COLUMNS[question_index], "Checkpoint_Path": str(checkpoint_path), "Checkpoint_Epoch": checkpoint["epoch"], "Validation_CCC": checkpoint["val_metrics"]["ccc"]})
        print(f"Seed {seed} | Q{question_index + 1} | Epoch {checkpoint['epoch']:02d} | Val CCC {checkpoint['val_metrics']['ccc']:.6f}")
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    question_predictions = torch.stack(question_prediction_list, dim=1)
    predicted_totals = question_predictions.sum(dim=1)
    true_totals = video_test_dataset.true_totals.clone()
    total_ccc = calculate_ccc(predicted_totals, true_totals)
    total_rmse = float(torch.sqrt(torch.mean((predicted_totals.float() - true_totals.float()) ** 2)).item())
    total_mae = float(torch.mean(torch.abs(predicted_totals.float() - true_totals.float())).item())
    exact_accuracy = float((predicted_totals == true_totals).float().mean().item())
    prediction_df = pd.DataFrame({"Participant_ID": video_test_dataset.participant_ids})
    for question_index in range(len(PHQ8_QUESTION_COLUMNS)):
        prediction_df[f"Q{question_index + 1}_Predicted"] = question_predictions[:, question_index].numpy()
    prediction_df["Predicted_Total"] = predicted_totals.numpy()
    prediction_df["True_Total"] = true_totals.numpy()
    prediction_path = VIDEO_CHECKPOINT_ROOT / f"seed_{seed}_test_predictions_56.csv"
    selected_checkpoint_path = VIDEO_CHECKPOINT_ROOT / f"seed_{seed}_test_selected_checkpoints.csv"
    prediction_df.to_csv(prediction_path, index=False)
    pd.DataFrame(selected_checkpoint_rows).to_csv(selected_checkpoint_path, index=False)
    print("=" * 70)
    print(f"Video-only Seed {seed}独立测试结果")
    print("=" * 70)
    print("测试参与者数量：", len(prediction_df))
    print(f"CCC：{total_ccc:.6f}")
    print(f"RMSE：{total_rmse:.6f}")
    print(f"MAE：{total_mae:.6f}")
    print(f"完全相等比例：{exact_accuracy:.6f}")
    print(f"预测总分范围：{int(predicted_totals.min())}～{int(predicted_totals.max())}")
    print(f"真实总分范围：{int(true_totals.min())}～{int(true_totals.max())}")
    print("预测文件：", prediction_path)
    return {"seed": seed, "sample_count": len(prediction_df), "ccc": total_ccc, "rmse": total_rmse, "mae": total_mae, "exact_accuracy": exact_accuracy, "predicted_totals": predicted_totals, "true_totals": true_totals, "prediction_path": prediction_path, "selected_checkpoint_path": selected_checkpoint_path}

In [86]:
video_seed42_test_result = evaluate_video_seed_on_test(seed=42)
video_seed100_test_result = evaluate_video_seed_on_test(seed=100)
video_seed1234_test_result = evaluate_video_seed_on_test(seed=1234)

Seed 42 | Q1 | Epoch 02 | Val CCC 0.272326
Seed 42 | Q2 | Epoch 04 | Val CCC 0.224460
Seed 42 | Q3 | Epoch 21 | Val CCC 0.309804
Seed 42 | Q4 | Epoch 40 | Val CCC 0.340570
Seed 42 | Q5 | Epoch 27 | Val CCC 0.144876
Seed 42 | Q6 | Epoch 41 | Val CCC 0.429989
Seed 42 | Q7 | Epoch 09 | Val CCC 0.188525
Seed 42 | Q8 | Epoch 10 | Val CCC 0.226757
Video-only Seed 42独立测试结果
测试参与者数量： 56
CCC：0.159762
RMSE：7.063488
MAE：5.464286
完全相等比例：0.071429
预测总分范围：0～15
真实总分范围：0～22
预测文件： /workspace/E-DAIC/checkpoints/phq8_video_paperlike/seed_42_test_predictions_56.csv
Seed 100 | Q1 | Epoch 06 | Val CCC 0.279613
Seed 100 | Q2 | Epoch 04 | Val CCC 0.212202
Seed 100 | Q3 | Epoch 45 | Val CCC 0.319661
Seed 100 | Q4 | Epoch 21 | Val CCC 0.308466
Seed 100 | Q5 | Epoch 22 | Val CCC 0.132492
Seed 100 | Q6 | Epoch 43 | Val CCC 0.478824
Seed 100 | Q7 | Epoch 33 | Val CCC 0.220881
Seed 100 | Q8 | Epoch 03 | Val CCC 0.135560
Video-only Seed 100独立测试结果
测试参与者数量： 56
CCC：0.153693
RMSE：7.293588
MAE：5.660714
完全相等比例：0.089286
预测总分

In [87]:
all_video_test_results = [video_seed42_test_result, video_seed100_test_result, video_seed1234_test_result]
video_test_seed_rows = []
for result in all_video_test_results:
    video_test_seed_rows.append({"Seed": result["seed"], "Sample_Count": result["sample_count"], "CCC": result["ccc"], "RMSE": result["rmse"], "MAE": result["mae"], "Exact_Accuracy": result["exact_accuracy"], "Predicted_Min": int(result["predicted_totals"].min()), "Predicted_Max": int(result["predicted_totals"].max()), "True_Min": int(result["true_totals"].min()), "True_Max": int(result["true_totals"].max())})
video_test_seed_summary_df = pd.DataFrame(video_test_seed_rows)
display(video_test_seed_summary_df)

,Seed,Sample_Count,CCC,RMSE,MAE,Exact_Accuracy,Predicted_Min,Predicted_Max,True_Min,True_Max
0,42,56,0.159762,7.063488,5.464286,0.071429,0,15,0,22
1,100,56,0.153693,7.293588,5.660714,0.089286,0,18,0,22
2,1234,56,0.074478,7.388456,5.910714,0.089286,1,19,0,22


In [88]:
video_three_seed_rows = []
for metric_name in ["CCC", "RMSE", "MAE", "Exact_Accuracy"]:
    video_three_seed_rows.append({"Metric": metric_name, "Mean": video_test_seed_summary_df[metric_name].mean(), "Std": video_test_seed_summary_df[metric_name].std(ddof=1)})
video_three_seed_summary_df = pd.DataFrame(video_three_seed_rows)
display(video_three_seed_summary_df)

,Metric,Mean,Std
0,CCC,0.129311,0.047584
1,RMSE,7.248511,0.167108
2,MAE,5.678571,0.223749
3,Exact_Accuracy,0.083333,0.010310


In [89]:
video_test_seed_summary_path = VIDEO_CHECKPOINT_ROOT / "three_seed_test_results_56.csv"
video_three_seed_summary_path = VIDEO_CHECKPOINT_ROOT / "three_seed_test_summary_56.csv"
video_test_seed_summary_df.to_csv(video_test_seed_summary_path, index=False)
video_three_seed_summary_df.to_csv(video_three_seed_summary_path, index=False)
print("逐Seed测试结果：", video_test_seed_summary_path)
print("三Seed测试汇总：", video_three_seed_summary_path)
print("逐Seed文件存在：", video_test_seed_summary_path.exists())
print("汇总文件存在：", video_three_seed_summary_path.exists())

逐Seed测试结果： /workspace/E-DAIC/checkpoints/phq8_video_paperlike/three_seed_test_results_56.csv
三Seed测试汇总： /workspace/E-DAIC/checkpoints/phq8_video_paperlike/three_seed_test_summary_56.csv
逐Seed文件存在： True
汇总文件存在： True


In [90]:
candidate_excluded_test_id = 637
test_keep_mask_55 = torch.tensor([participant_id != candidate_excluded_test_id for participant_id in video_test_dataset.participant_ids], dtype=torch.bool)
print("排除参与者：", candidate_excluded_test_id)
print("排除前人数：", len(video_test_dataset))
print("排除后人数：", int(test_keep_mask_55.sum()))

排除参与者： 637
排除前人数： 56
排除后人数： 55


In [91]:
video_test_55_rows = []
for result in all_video_test_results:
    predicted_totals_55 = result["predicted_totals"][test_keep_mask_55]
    true_totals_55 = result["true_totals"][test_keep_mask_55]
    ccc_55 = calculate_ccc(predicted_totals_55, true_totals_55)
    rmse_55 = float(torch.sqrt(torch.mean((predicted_totals_55.float() - true_totals_55.float()) ** 2)).item())
    mae_55 = float(torch.mean(torch.abs(predicted_totals_55.float() - true_totals_55.float())).item())
    exact_accuracy_55 = float((predicted_totals_55 == true_totals_55).float().mean().item())
    video_test_55_rows.append({"Seed": result["seed"], "Sample_Count": len(predicted_totals_55), "CCC": ccc_55, "RMSE": rmse_55, "MAE": mae_55, "Exact_Accuracy": exact_accuracy_55})
video_test_55_summary_df = pd.DataFrame(video_test_55_rows)
display(video_test_55_summary_df)

,Seed,Sample_Count,CCC,RMSE,MAE,Exact_Accuracy
0,42,55,0.167893,6.998702,5.381818,0.072727
1,100,55,0.167858,7.208580,5.563636,0.090909
2,1234,55,0.089761,7.306286,5.818182,0.090909


In [92]:
video_three_seed_55_rows = []
for metric_name in ["CCC", "RMSE", "MAE", "Exact_Accuracy"]:
    video_three_seed_55_rows.append({"Metric": metric_name, "Mean": video_test_55_summary_df[metric_name].mean(), "Std": video_test_55_summary_df[metric_name].std(ddof=1)})
video_three_seed_55_summary_df = pd.DataFrame(video_three_seed_55_rows)
display(video_three_seed_55_summary_df)

,Metric,Mean,Std
0,CCC,0.141837,0.045099
1,RMSE,7.171189,0.157164
2,MAE,5.587879,0.219190
3,Exact_Accuracy,0.084848,0.010497


In [93]:
video_test_55_summary_df.to_csv(VIDEO_CHECKPOINT_ROOT / "three_seed_test_results_candidate55.csv", index=False)
video_three_seed_55_summary_df.to_csv(VIDEO_CHECKPOINT_ROOT / "three_seed_test_summary_candidate55.csv", index=False)

## text+vedio

In [94]:
TEXT_CACHE_ROOT = Path("/workspace/E-DAIC/cache/phq8_text_embeddings")
TEXT_CHECKPOINT_ROOT = Path("/workspace/E-DAIC/checkpoints/phq8_text_paperlike")
TEXT_VIDEO_CHECKPOINT_ROOT = Path("/workspace/E-DAIC/checkpoints/phq8_text_video_paperlike")
TEXT_FEATURE_DIM = 768
VIDEO_FEATURE_DIM = 2048
ENCODER_OUTPUT_DIM = 100
FUSION_NUM_EPOCHS = 20
FUSION_LEARNING_RATE = 5e-4
FUSION_ADAM_EPSILON = 1e-8
FUSION_WEIGHT_DECAY = 1e-3
FUSION_DROPOUT = 0.8
FUSION_MLP_FIRST_DROPOUT = 0.8
FUSION_MLP_LAST_DROPOUT = 0.5
TEXT_VIDEO_CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)

In [96]:
text_cache_candidates = sorted(TEXT_CACHE_ROOT.rglob("302.pt"))
print("找到的302文本缓存：", text_cache_candidates)
if len(text_cache_candidates) != 1:
    raise RuntimeError(f"预期找到一个302文本缓存，实际找到{len(text_cache_candidates)}个")
TEXT_CACHE_DIR = text_cache_candidates[0].parent
print("正式文本缓存目录：", TEXT_CACHE_DIR)

找到的302文本缓存： [PosixPath('/workspace/E-DAIC/cache/phq8_text_embeddings/distilroberta_842eaed4_turns120_v1/302.pt')]
正式文本缓存目录： /workspace/E-DAIC/cache/phq8_text_embeddings/distilroberta_842eaed4_turns120_v1


In [97]:
def get_text_video_best_loss_paths(question_index, seed):
    text_checkpoint_path = TEXT_CHECKPOINT_ROOT / f"q{question_index + 1}_{PHQ8_QUESTION_COLUMNS[question_index]}" / f"seed_{seed}" / "best_loss.pt"
    video_checkpoint_path = VIDEO_CHECKPOINT_ROOT / f"seed_{seed}" / f"question_{question_index + 1}" / "best_loss.pt"
    if not text_checkpoint_path.exists():
        raise FileNotFoundError(f"文本Best Loss不存在：{text_checkpoint_path}")
    if not video_checkpoint_path.exists():
        raise FileNotFoundError(f"视频Best Loss不存在：{video_checkpoint_path}")
    text_checkpoint = torch.load(text_checkpoint_path, map_location="cpu", weights_only=False)
    video_checkpoint = torch.load(video_checkpoint_path, map_location="cpu", weights_only=False)
    if text_checkpoint["seed"] != seed or text_checkpoint["question_index"] != question_index or text_checkpoint["checkpoint_type"] != "best_loss":
        raise RuntimeError(f"文本Checkpoint元数据错误：{text_checkpoint_path}")
    if video_checkpoint["seed"] != seed or video_checkpoint["question_index"] != question_index or video_checkpoint["checkpoint_type"] != "best_loss":
        raise RuntimeError(f"视频Checkpoint元数据错误：{video_checkpoint_path}")
    return text_checkpoint_path, video_checkpoint_path

In [98]:
q1_text_best_loss_path, q1_video_best_loss_path = get_text_video_best_loss_paths(question_index=0, seed=42)
print("Q1文本Best Loss：", q1_text_best_loss_path)
print("Q1视频Best Loss：", q1_video_best_loss_path)

Q1文本Best Loss： /workspace/E-DAIC/checkpoints/phq8_text_paperlike/q1_PHQ_8NoInterest/seed_42/best_loss.pt
Q1视频Best Loss： /workspace/E-DAIC/checkpoints/phq8_video_paperlike/seed_42/question_1/best_loss.pt


In [99]:
def load_text_video_cache(participant_id):
    participant_id = int(participant_id)
    text_cache_path = TEXT_CACHE_DIR / f"{participant_id}.pt"
    video_cache_path = VIDEO_CACHE_DIR / f"{participant_id}.pt"
    if not text_cache_path.exists():
        raise FileNotFoundError(f"文本缓存不存在：{text_cache_path}")
    if not video_cache_path.exists():
        raise FileNotFoundError(f"视频缓存不存在：{video_cache_path}")
    text_cache = torch.load(text_cache_path, map_location="cpu", weights_only=False)
    video_cache = torch.load(video_cache_path, map_location="cpu", weights_only=False)
    text_features = text_cache["embeddings"].float()
    text_padding_mask = text_cache["key_padding_mask"].bool()
    video_features = video_cache["video_features"].float()
    video_padding_mask = video_cache["padding_mask"].bool()
    if text_features.shape != (MAX_TURNS, TEXT_FEATURE_DIM):
        raise RuntimeError(f"参与者{participant_id}文本形状错误：{text_features.shape}")
    if text_padding_mask.shape != (MAX_TURNS,):
        raise RuntimeError(f"参与者{participant_id}文本Mask错误：{text_padding_mask.shape}")
    if video_features.shape != (MAX_TURNS, VIDEO_FEATURE_DIM):
        raise RuntimeError(f"参与者{participant_id}视频形状错误：{video_features.shape}")
    if video_padding_mask.shape != (MAX_TURNS,):
        raise RuntimeError(f"参与者{participant_id}视频Mask错误：{video_padding_mask.shape}")
    if not torch.isfinite(text_features).all():
        raise RuntimeError(f"参与者{participant_id}文本包含NaN或Inf")
    if not torch.isfinite(video_features).all():
        raise RuntimeError(f"参与者{participant_id}视频包含NaN或Inf")
    return text_features, text_padding_mask, video_features, video_padding_mask


### text+vedio dataset

In [100]:
class TextVideoPHQ8Dataset(Dataset):
    def __init__(self, metadata_df, split_name):
        super().__init__()
        self.metadata_df = metadata_df.reset_index(drop=True).copy()
        self.participant_ids = self.metadata_df["Participant_ID"].astype(int).tolist()
        self.labels = torch.tensor(self.metadata_df[PHQ8_QUESTION_COLUMNS].to_numpy(), dtype=torch.long)
        text_feature_list = []
        text_mask_list = []
        video_feature_list = []
        video_mask_list = []
        for index, participant_id in enumerate(self.participant_ids):
            text_features, text_mask, video_features, video_mask = load_text_video_cache(participant_id)
            text_feature_list.append(text_features)
            text_mask_list.append(text_mask)
            video_feature_list.append(video_features)
            video_mask_list.append(video_mask)
            if (index + 1) % 50 == 0 or index + 1 == len(self.participant_ids):
                print(f"{split_name} Text+Video加载进度：{index + 1}/{len(self.participant_ids)}")
        self.text_features = torch.stack(text_feature_list, dim=0)
        self.text_masks = torch.stack(text_mask_list, dim=0)
        self.video_features = torch.stack(video_feature_list, dim=0)
        self.video_masks = torch.stack(video_mask_list, dim=0)

    def __len__(self):
        return len(self.participant_ids)

    def __getitem__(self, index):
        return self.text_features[index], self.text_masks[index], self.video_features[index], self.video_masks[index], self.labels[index]

In [101]:
text_video_train_dataset = TextVideoPHQ8Dataset(video_train_metadata_df, split_name="train")
text_video_val_dataset = TextVideoPHQ8Dataset(video_val_metadata_df, split_name="val")

train Text+Video加载进度：50/163
train Text+Video加载进度：100/163
train Text+Video加载进度：150/163
train Text+Video加载进度：163/163
val Text+Video加载进度：50/55
val Text+Video加载进度：55/55


In [102]:
def create_text_video_dataloaders(seed):
    train_generator = torch.Generator()
    train_generator.manual_seed(seed)
    train_loader = DataLoader(text_video_train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, drop_last=False, generator=train_generator)
    val_loader = DataLoader(text_video_val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, drop_last=False)
    return train_loader, val_loader

In [103]:
text_video_train_loader, text_video_val_loader = create_text_video_dataloaders(seed=42)
batch_text_features, batch_text_masks, batch_video_features, batch_video_masks, batch_all_labels = next(iter(text_video_train_loader))
print("训练样本数：", len(text_video_train_dataset))
print("验证样本数：", len(text_video_val_dataset))
print("文本特征：", batch_text_features.shape)
print("文本Mask：", batch_text_masks.shape)
print("视频特征：", batch_video_features.shape)
print("视频Mask：", batch_video_masks.shape)
print("全部标签：", batch_all_labels.shape)
print("文本全部有限：", bool(torch.isfinite(batch_text_features).all()))
print("视频全部有限：", bool(torch.isfinite(batch_video_features).all()))

训练样本数： 163
验证样本数： 55
文本特征： torch.Size([10, 120, 768])
文本Mask： torch.Size([10, 120])
视频特征： torch.Size([10, 120, 2048])
视频Mask： torch.Size([10, 120])
全部标签： torch.Size([10, 8])
文本全部有限： True
视频全部有限： True


In [104]:
class PretrainedTextEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=TEXT_FEATURE_DIM, hidden_size=50, batch_first=True, bidirectional=True)
        self.attention = nn.MultiheadAttention(embed_dim=ENCODER_OUTPUT_DIM, num_heads=ATTENTION_HEADS, dropout=0.5, batch_first=True)

    def forward(self, text_features, text_padding_mask):
        lstm_output, _ = self.lstm(text_features)
        text_encoding, _ = self.attention(lstm_output, lstm_output, lstm_output, key_padding_mask=text_padding_mask, need_weights=False)
        text_encoding = text_encoding.masked_fill(text_padding_mask.unsqueeze(-1), 0.0)
        return text_encoding

In [105]:
class PretrainedVideoEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=VIDEO_FEATURE_DIM, hidden_size=50, batch_first=True, bidirectional=True)
        self.attention1 = nn.MultiheadAttention(embed_dim=ENCODER_OUTPUT_DIM, num_heads=ATTENTION_HEADS, dropout=0.2, batch_first=True)
        self.attention2 = nn.MultiheadAttention(embed_dim=ENCODER_OUTPUT_DIM, num_heads=ATTENTION_HEADS, dropout=0.2, batch_first=True)

    def forward(self, video_features, video_padding_mask):
        lstm_output, _ = self.lstm(video_features)
        attention1_output, _ = self.attention1(lstm_output, lstm_output, lstm_output, key_padding_mask=video_padding_mask, need_weights=False)
        attention1_output = attention1_output.masked_fill(video_padding_mask.unsqueeze(-1), 0.0)
        video_encoding, _ = self.attention2(attention1_output, attention1_output, attention1_output, key_padding_mask=video_padding_mask, need_weights=False)
        video_encoding = video_encoding.masked_fill(video_padding_mask.unsqueeze(-1), 0.0)
        return video_encoding

In [106]:
def extract_state_dict_by_prefixes(full_state_dict, prefixes):
    selected_state_dict = {}
    for parameter_name, parameter_value in full_state_dict.items():
        if any(parameter_name.startswith(prefix) for prefix in prefixes):
            selected_state_dict[parameter_name] = parameter_value
    return selected_state_dict

In [107]:
def load_pretrained_text_video_encoders(question_index, seed):
    text_checkpoint_path, video_checkpoint_path = get_text_video_best_loss_paths(question_index, seed)
    text_checkpoint = torch.load(text_checkpoint_path, map_location="cpu", weights_only=False)
    video_checkpoint = torch.load(video_checkpoint_path, map_location="cpu", weights_only=False)
    text_encoder = PretrainedTextEncoder()
    video_encoder = PretrainedVideoEncoder()
    text_encoder_state = extract_state_dict_by_prefixes(text_checkpoint["model_state_dict"], prefixes=("lstm.", "attention."))
    video_encoder_state = extract_state_dict_by_prefixes(video_checkpoint["model_state_dict"], prefixes=("lstm.", "attention1.", "attention2."))
    text_load_result = text_encoder.load_state_dict(text_encoder_state, strict=True)
    video_load_result = video_encoder.load_state_dict(video_encoder_state, strict=True)
    print("文本Best Loss：", text_checkpoint_path)
    print("视频Best Loss：", video_checkpoint_path)
    print("文本保存Epoch：", text_checkpoint["epoch"])
    print("视频保存Epoch：", video_checkpoint["epoch"])
    print("文本缺少参数：", text_load_result.missing_keys)
    print("文本多余参数：", text_load_result.unexpected_keys)
    print("视频缺少参数：", video_load_result.missing_keys)
    print("视频多余参数：", video_load_result.unexpected_keys)
    return text_encoder, video_encoder, text_checkpoint, video_checkpoint, text_checkpoint_path, video_checkpoint_path

In [108]:
class TextVideoQuestMF(nn.Module):
    def __init__(self, text_encoder, video_encoder):
        super().__init__()
        self.text_encoder = text_encoder
        self.video_encoder = video_encoder
        self.video_to_text_cross_attention = nn.MultiheadAttention(embed_dim=ENCODER_OUTPUT_DIM, num_heads=ATTENTION_HEADS, dropout=FUSION_DROPOUT, batch_first=True)
        self.text_to_video_cross_attention = nn.MultiheadAttention(embed_dim=ENCODER_OUTPUT_DIM, num_heads=ATTENTION_HEADS, dropout=FUSION_DROPOUT, batch_first=True)
        self.text_self_attention = nn.MultiheadAttention(embed_dim=ENCODER_OUTPUT_DIM, num_heads=ATTENTION_HEADS, dropout=FUSION_DROPOUT, batch_first=True)
        self.video_self_attention = nn.MultiheadAttention(embed_dim=ENCODER_OUTPUT_DIM, num_heads=ATTENTION_HEADS, dropout=FUSION_DROPOUT, batch_first=True)
        self.mlp = nn.Sequential(nn.Flatten(), nn.Dropout(FUSION_MLP_FIRST_DROPOUT), nn.Linear(MAX_TURNS * ENCODER_OUTPUT_DIM * 2, 256), nn.ReLU(), nn.Dropout(FUSION_MLP_LAST_DROPOUT), nn.Linear(256, NUM_CLASSES))
        for parameter in self.text_encoder.parameters():
            parameter.requires_grad = False
        self.text_encoder.eval()

    def train(self, mode=True):
        super().train(mode)
        self.text_encoder.eval()
        return self

    def forward(self, text_features, text_padding_mask, video_features, video_padding_mask, return_intermediates=False):
        text_encoding = self.text_encoder(text_features, text_padding_mask)
        video_encoding = self.video_encoder(video_features, video_padding_mask)
        video_to_text, _ = self.video_to_text_cross_attention(text_encoding, video_encoding, video_encoding, key_padding_mask=video_padding_mask, need_weights=False)
        video_to_text = video_to_text.masked_fill(text_padding_mask.unsqueeze(-1), 0.0)
        text_to_video, _ = self.text_to_video_cross_attention(video_encoding, text_encoding, text_encoding, key_padding_mask=text_padding_mask, need_weights=False)
        text_to_video = text_to_video.masked_fill(video_padding_mask.unsqueeze(-1), 0.0)
        text_fused_encoding, _ = self.text_self_attention(video_to_text, video_to_text, video_to_text, key_padding_mask=text_padding_mask, need_weights=False)
        text_fused_encoding = text_fused_encoding.masked_fill(text_padding_mask.unsqueeze(-1), 0.0)
        video_fused_encoding, _ = self.video_self_attention(text_to_video, text_to_video, text_to_video, key_padding_mask=video_padding_mask, need_weights=False)
        video_fused_encoding = video_fused_encoding.masked_fill(video_padding_mask.unsqueeze(-1), 0.0)
        fused_encoding = torch.cat((text_fused_encoding, video_fused_encoding), dim=2)
        logits = self.mlp(fused_encoding)
        if return_intermediates:
            return logits, {"text_encoding": text_encoding, "video_encoding": video_encoding, "video_to_text": video_to_text, "text_to_video": text_to_video, "text_fused_encoding": text_fused_encoding, "video_fused_encoding": video_fused_encoding, "fused_encoding": fused_encoding}
        return logits

In [109]:
set_seed(42)
q1_text_encoder, q1_video_encoder, q1_text_checkpoint, q1_video_checkpoint, q1_text_checkpoint_path, q1_video_checkpoint_path = load_pretrained_text_video_encoders(question_index=0, seed=42)
q1_text_video_model = TextVideoQuestMF(q1_text_encoder, q1_video_encoder).to(device)
q1_text_video_model.train()

文本Best Loss： /workspace/E-DAIC/checkpoints/phq8_text_paperlike/q1_PHQ_8NoInterest/seed_42/best_loss.pt
视频Best Loss： /workspace/E-DAIC/checkpoints/phq8_video_paperlike/seed_42/question_1/best_loss.pt
文本保存Epoch： 13
视频保存Epoch： 2
文本缺少参数： []
文本多余参数： []
视频缺少参数： []
视频多余参数： []


TextVideoQuestMF(
  (text_encoder): PretrainedTextEncoder(
    (lstm): LSTM(768, 50, batch_first=True, bidirectional=True)
    (attention): MultiheadAttention(
      (out_proj): NonDynamicallyQuantizableLinear(in_features=100, out_features=100, bias=True)
    )
  )
  (video_encoder): PretrainedVideoEncoder(
    (lstm): LSTM(2048, 50, batch_first=True, bidirectional=True)
    (attention1): MultiheadAttention(
      (out_proj): NonDynamicallyQuantizableLinear(in_features=100, out_features=100, bias=True)
    )
    (attention2): MultiheadAttention(
      (out_proj): NonDynamicallyQuantizableLinear(in_features=100, out_features=100, bias=True)
    )
  )
  (video_to_text_cross_attention): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=100, out_features=100, bias=True)
  )
  (text_to_video_cross_attention): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=100, out_features=100, bias=True)
  )
  (text_self_attention): MultiheadAt

In [110]:
text_total_parameters = sum(parameter.numel() for parameter in q1_text_video_model.text_encoder.parameters())
text_trainable_parameters = sum(parameter.numel() for parameter in q1_text_video_model.text_encoder.parameters() if parameter.requires_grad)
video_trainable_parameters = sum(parameter.numel() for parameter in q1_text_video_model.video_encoder.parameters() if parameter.requires_grad)
fusion_attention_parameters = sum(parameter.numel() for name, parameter in q1_text_video_model.named_parameters() if ("cross_attention" in name or "self_attention" in name) and parameter.requires_grad)
fusion_mlp_parameters = sum(parameter.numel() for parameter in q1_text_video_model.mlp.parameters() if parameter.requires_grad)
total_trainable_parameters = sum(parameter.numel() for parameter in q1_text_video_model.parameters() if parameter.requires_grad)
print("文本编码器总参数：", f"{text_total_parameters:,}")
print("文本可训练参数：", f"{text_trainable_parameters:,}")
print("视频可训练参数：", f"{video_trainable_parameters:,}")
print("融合Attention参数：", f"{fusion_attention_parameters:,}")
print("融合MLP参数：", f"{fusion_mlp_parameters:,}")
print("总可训练参数：", f"{total_trainable_parameters:,}")
print("文本编码器处于训练模式：", q1_text_video_model.text_encoder.training)
print("视频编码器处于训练模式：", q1_text_video_model.video_encoder.training)

文本编码器总参数： 368,400
文本可训练参数： 0
视频可训练参数： 920,800
融合Attention参数： 161,600
融合MLP参数： 6,145,284
总可训练参数： 7,227,684
文本编码器处于训练模式： False
视频编码器处于训练模式： True


In [111]:
q1_text_video_model.eval()
with torch.no_grad():
    q1_text_video_logits, q1_text_video_intermediates = q1_text_video_model(batch_text_features.to(device), batch_text_masks.to(device), batch_video_features.to(device), batch_video_masks.to(device), return_intermediates=True)
print("文本编码：", q1_text_video_intermediates["text_encoding"].shape)
print("视频编码：", q1_text_video_intermediates["video_encoding"].shape)
print("Video→Text：", q1_text_video_intermediates["video_to_text"].shape)
print("Text→Video：", q1_text_video_intermediates["text_to_video"].shape)
print("拼接后：", q1_text_video_intermediates["fused_encoding"].shape)
print("最终Logits：", q1_text_video_logits.shape)
print("Logits全部有限：", bool(torch.isfinite(q1_text_video_logits).all()))

文本编码： torch.Size([10, 120, 100])
视频编码： torch.Size([10, 120, 100])
Video→Text： torch.Size([10, 120, 100])
Text→Video： torch.Size([10, 120, 100])
拼接后： torch.Size([10, 120, 200])
最终Logits： torch.Size([10, 4])
Logits全部有限： True


In [112]:
def calculate_gradient_statistics(parameters):
    squared_gradient_sum = 0.0
    gradient_tensor_count = 0
    for parameter in parameters:
        if parameter.grad is not None:
            squared_gradient_sum += parameter.grad.detach().pow(2).sum().item()
            gradient_tensor_count += 1
    return squared_gradient_sum ** 0.5, gradient_tensor_count

In [113]:
q1_train_labels = text_video_train_dataset.labels[:, 0]
q1_class_counts = torch.bincount(q1_train_labels, minlength=NUM_CLASSES)
q1_class_weights = torch.pow(len(q1_train_labels) / q1_class_counts.float(), BETA)
q1_text_video_criterion = ImbOLLLoss(q1_class_weights, alpha=ALPHA).to(device)
print("Q1类别数量：", q1_class_counts.tolist())
print("Q1 ImbOLL权重：", q1_class_weights.tolist())

Q1类别数量： [81, 57, 20, 5]
Q1 ImbOLL权重： [1.4185717105865479, 1.6910496950149536, 2.854820728302002, 5.709641456604004]


In [114]:
q1_text_video_optimizer = torch.optim.AdamW((parameter for parameter in q1_text_video_model.parameters() if parameter.requires_grad), lr=FUSION_LEARNING_RATE, eps=FUSION_ADAM_EPSILON, weight_decay=FUSION_WEIGHT_DECAY)
q1_text_video_model.train()
diagnostic_text_features = batch_text_features.to(device)
diagnostic_text_masks = batch_text_masks.to(device)
diagnostic_video_features = batch_video_features.to(device)
diagnostic_video_masks = batch_video_masks.to(device)
diagnostic_targets = batch_all_labels[:, 0].to(device)
q1_text_video_optimizer.zero_grad(set_to_none=True)
diagnostic_logits = q1_text_video_model(diagnostic_text_features, diagnostic_text_masks, diagnostic_video_features, diagnostic_video_masks)
diagnostic_loss = q1_text_video_criterion(diagnostic_logits, diagnostic_targets)
diagnostic_loss.backward()
text_gradient_norm, text_gradient_tensor_count = calculate_gradient_statistics(q1_text_video_model.text_encoder.parameters())
video_gradient_norm, video_gradient_tensor_count = calculate_gradient_statistics(q1_text_video_model.video_encoder.parameters())
fusion_parameters = list(q1_text_video_model.video_to_text_cross_attention.parameters()) + list(q1_text_video_model.text_to_video_cross_attention.parameters()) + list(q1_text_video_model.text_self_attention.parameters()) + list(q1_text_video_model.video_self_attention.parameters()) + list(q1_text_video_model.mlp.parameters())
fusion_gradient_norm, fusion_gradient_tensor_count = calculate_gradient_statistics(fusion_parameters)
print("Logits形状：", diagnostic_logits.shape)
print("ImbOLL：", diagnostic_loss.item())
print("预测概率全部有限：", bool(torch.isfinite(torch.softmax(diagnostic_logits, dim=1)).all()))
print("预测类别：", diagnostic_logits.argmax(dim=1))
print("真实类别：", diagnostic_targets)
print("文本梯度范数：", text_gradient_norm)
print("文本梯度张量数：", text_gradient_tensor_count)
print("视频梯度范数：", video_gradient_norm)
print("视频梯度张量数：", video_gradient_tensor_count)
print("融合层梯度范数：", fusion_gradient_norm)
print("融合层梯度张量数：", fusion_gradient_tensor_count)

Logits形状： torch.Size([10, 4])
ImbOLL： 2.2568275928497314
预测概率全部有限： True
预测类别： tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1], device='cuda:0')
真实类别： tensor([1, 0, 1, 0, 0, 0, 0, 0, 0, 1], device='cuda:0')
文本梯度范数： 0.0
文本梯度张量数： 0
视频梯度范数： 0.033716787751368535
视频梯度张量数： 16
融合层梯度范数： 1.093400904704733
融合层梯度张量数： 20


In [115]:
def run_text_video_epoch(model, data_loader, question_index, criterion, optimizer=None):
    is_training = optimizer is not None
    if is_training:
        model.train()
    else:
        model.eval()
    total_loss = 0.0
    total_samples = 0
    prediction_list = []
    target_list = []
    for text_features, text_masks, video_features, video_masks, all_labels in data_loader:
        text_features = text_features.to(device)
        text_masks = text_masks.to(device)
        video_features = video_features.to(device)
        video_masks = video_masks.to(device)
        targets = all_labels[:, question_index].to(device)
        if is_training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(is_training):
            logits = model(text_features, text_masks, video_features, video_masks)
            loss = criterion(logits, targets)
            if is_training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                optimizer.step()
        batch_size = targets.shape[0]
        total_loss += loss.item() * batch_size
        total_samples += batch_size
        prediction_list.append(logits.argmax(dim=1).detach().cpu())
        target_list.append(targets.detach().cpu())
    predictions = torch.cat(prediction_list)
    targets = torch.cat(target_list)
    metrics = calculate_video_metrics(predictions, targets)
    metrics["loss"] = total_loss / total_samples
    metrics["predictions"] = predictions
    metrics["targets"] = targets
    return metrics

In [116]:
set_seed(42)
text_video_train_loader, text_video_val_loader = create_text_video_dataloaders(seed=42)
q1_text_encoder, q1_video_encoder, q1_text_checkpoint, q1_video_checkpoint, q1_text_checkpoint_path, q1_video_checkpoint_path = load_pretrained_text_video_encoders(question_index=0, seed=42)
q1_text_video_model = TextVideoQuestMF(q1_text_encoder, q1_video_encoder).to(device)
q1_train_labels = text_video_train_dataset.labels[:, 0]
q1_class_counts = torch.bincount(q1_train_labels, minlength=NUM_CLASSES)
q1_class_weights = torch.pow(len(q1_train_labels) / q1_class_counts.float(), BETA)
q1_text_video_criterion = ImbOLLLoss(q1_class_weights, alpha=ALPHA).to(device)
q1_text_video_optimizer = torch.optim.AdamW((parameter for parameter in q1_text_video_model.parameters() if parameter.requires_grad), lr=FUSION_LEARNING_RATE, eps=FUSION_ADAM_EPSILON, weight_decay=FUSION_WEIGHT_DECAY)

文本Best Loss： /workspace/E-DAIC/checkpoints/phq8_text_paperlike/q1_PHQ_8NoInterest/seed_42/best_loss.pt
视频Best Loss： /workspace/E-DAIC/checkpoints/phq8_video_paperlike/seed_42/question_1/best_loss.pt
文本保存Epoch： 13
视频保存Epoch： 2
文本缺少参数： []
文本多余参数： []
视频缺少参数： []
视频多余参数： []


In [117]:
q1_text_video_train_metrics = run_text_video_epoch(q1_text_video_model, text_video_train_loader, question_index=0, criterion=q1_text_video_criterion, optimizer=q1_text_video_optimizer)
q1_text_video_val_metrics = run_text_video_epoch(q1_text_video_model, text_video_val_loader, question_index=0, criterion=q1_text_video_criterion)

In [118]:
print(f"一轮训练 Loss：{q1_text_video_train_metrics['loss']:.6f}")
print(f"一轮训练 Accuracy：{q1_text_video_train_metrics['accuracy']:.6f}")
print(f"一轮训练 Micro F1：{q1_text_video_train_metrics['micro_f1']:.6f}")
print(f"一轮训练 Macro F1：{q1_text_video_train_metrics['macro_f1']:.6f}")
print(f"一轮训练 Weighted F1：{q1_text_video_train_metrics['weighted_f1']:.6f}")
print(f"一轮训练 CCC：{q1_text_video_train_metrics['ccc']:.6f}")
print(f"一轮训练 RMSE：{q1_text_video_train_metrics['rmse']:.6f}")
print(f"一轮训练 MAE：{q1_text_video_train_metrics['mae']:.6f}")
print("一轮训练混淆矩阵：")
print(q1_text_video_train_metrics["confusion_matrix"])
print(f"一轮验证 Loss：{q1_text_video_val_metrics['loss']:.6f}")
print(f"一轮验证 Accuracy：{q1_text_video_val_metrics['accuracy']:.6f}")
print(f"一轮验证 Micro F1：{q1_text_video_val_metrics['micro_f1']:.6f}")
print(f"一轮验证 Macro F1：{q1_text_video_val_metrics['macro_f1']:.6f}")
print(f"一轮验证 Weighted F1：{q1_text_video_val_metrics['weighted_f1']:.6f}")
print(f"一轮验证 CCC：{q1_text_video_val_metrics['ccc']:.6f}")
print(f"一轮验证 RMSE：{q1_text_video_val_metrics['rmse']:.6f}")
print(f"一轮验证 MAE：{q1_text_video_val_metrics['mae']:.6f}")
print("一轮验证混淆矩阵：")
print(q1_text_video_val_metrics["confusion_matrix"])

一轮训练 Loss：2.150871
一轮训练 Accuracy：0.429448
一轮训练 Micro F1：0.429448
一轮训练 Macro F1：0.271970
一轮训练 Weighted F1：0.404077
一轮训练 CCC：0.215361
一轮训练 RMSE：0.825216
一轮训练 MAE：0.607362
一轮训练混淆矩阵：
tensor([[28, 53,  0,  0],
        [17, 40,  0,  0],
        [ 1, 17,  2,  0],
        [ 0,  5,  0,  0]])
一轮验证 Loss：2.717357
一轮验证 Accuracy：0.527273
一轮验证 Micro F1：0.527273
一轮验证 Macro F1：0.300992
一轮验证 Weighted F1：0.447362
一轮验证 CCC：0.320330
一轮验证 RMSE：0.962950
一轮验证 MAE：0.600000
一轮验证混淆矩阵：
tensor([[24,  1,  0,  0],
        [15,  4,  3,  0],
        [ 2,  1,  1,  0],
        [ 2,  1,  1,  0]])


In [119]:
def save_text_video_checkpoint(path, checkpoint_type, epoch, question_index, seed, model, optimizer, train_metrics, val_metrics, class_counts, class_weights, best_val_loss, best_val_ccc, best_loss_epoch, best_ccc_epoch, text_checkpoint_path, video_checkpoint_path):
    checkpoint = {}
    checkpoint["checkpoint_type"] = checkpoint_type
    checkpoint["epoch"] = epoch
    checkpoint["question_index"] = question_index
    checkpoint["question_number"] = question_index + 1
    checkpoint["question"] = PHQ8_QUESTION_COLUMNS[question_index]
    checkpoint["seed"] = seed
    checkpoint["model_state_dict"] = model.state_dict()
    checkpoint["optimizer_state_dict"] = optimizer.state_dict()
    checkpoint["train_metrics"] = compact_video_metrics(train_metrics)
    checkpoint["val_metrics"] = compact_video_metrics(val_metrics)
    checkpoint["class_counts"] = class_counts.tolist()
    checkpoint["class_weights"] = class_weights.tolist()
    checkpoint["best_val_loss"] = float(best_val_loss)
    checkpoint["best_val_ccc"] = float(best_val_ccc)
    checkpoint["best_loss_epoch"] = int(best_loss_epoch)
    checkpoint["best_ccc_epoch"] = int(best_ccc_epoch)
    checkpoint["text_pretrained_checkpoint"] = str(text_checkpoint_path)
    checkpoint["video_pretrained_checkpoint"] = str(video_checkpoint_path)
    checkpoint["video_variant"] = "masked_padding_current_video"
    checkpoint["config"] = {"max_turns": MAX_TURNS, "text_feature_dim": TEXT_FEATURE_DIM, "video_feature_dim": VIDEO_FEATURE_DIM, "encoder_output_dim": ENCODER_OUTPUT_DIM, "attention_heads": ATTENTION_HEADS, "fusion_dropout": FUSION_DROPOUT, "mlp_first_dropout": FUSION_MLP_FIRST_DROPOUT, "mlp_last_dropout": FUSION_MLP_LAST_DROPOUT, "num_classes": NUM_CLASSES, "batch_size": BATCH_SIZE, "num_epochs": FUSION_NUM_EPOCHS, "learning_rate": FUSION_LEARNING_RATE, "adam_epsilon": FUSION_ADAM_EPSILON, "weight_decay": FUSION_WEIGHT_DECAY, "max_grad_norm": MAX_GRAD_NORM, "alpha": ALPHA, "beta": BETA, "text_frozen": True}
    torch.save(checkpoint, path)

In [120]:
def train_text_video_question(seed, question_index):
    set_seed(seed)
    train_loader, val_loader = create_text_video_dataloaders(seed)
    text_encoder, video_encoder, text_checkpoint, video_checkpoint, text_checkpoint_path, video_checkpoint_path = load_pretrained_text_video_encoders(question_index, seed)
    model = TextVideoQuestMF(text_encoder, video_encoder).to(device)
    train_labels = text_video_train_dataset.labels[:, question_index]
    class_counts = torch.bincount(train_labels, minlength=NUM_CLASSES)
    class_weights = torch.pow(len(train_labels) / class_counts.float(), BETA)
    criterion = ImbOLLLoss(class_weights, alpha=ALPHA).to(device)
    optimizer = torch.optim.AdamW((parameter for parameter in model.parameters() if parameter.requires_grad), lr=FUSION_LEARNING_RATE, eps=FUSION_ADAM_EPSILON, weight_decay=FUSION_WEIGHT_DECAY)
    checkpoint_dir = TEXT_VIDEO_CHECKPOINT_ROOT / f"seed_{seed}" / f"question_{question_index + 1}"
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    best_loss_path = checkpoint_dir / "best_loss.pt"
    best_ccc_path = checkpoint_dir / "best_ccc.pt"
    last_path = checkpoint_dir / "last.pt"
    history_path = checkpoint_dir / "history.csv"
    best_val_loss = float("inf")
    best_val_ccc = float("-inf")
    best_loss_epoch = -1
    best_ccc_epoch = -1
    history_rows = []
    print("=" * 100)
    print(f"开始训练Text+Video：Seed={seed}，Q{question_index + 1}，{PHQ8_QUESTION_COLUMNS[question_index]}")
    print("类别数量：", class_counts.tolist())
    print("ImbOLL权重：", class_weights.tolist())
    print("文本预训练模型：", text_checkpoint_path)
    print("视频预训练模型：", video_checkpoint_path)
    print("=" * 100)
    for epoch in range(1, FUSION_NUM_EPOCHS + 1):
        epoch_start_time = time.perf_counter()
        train_metrics = run_text_video_epoch(model, train_loader, question_index, criterion, optimizer)
        val_metrics = run_text_video_epoch(model, val_loader, question_index, criterion)
        elapsed_seconds = time.perf_counter() - epoch_start_time
        improved_loss = val_metrics["loss"] < best_val_loss
        improved_ccc = val_metrics["ccc"] > best_val_ccc
        if improved_loss:
            best_val_loss = val_metrics["loss"]
            best_loss_epoch = epoch
        if improved_ccc:
            best_val_ccc = val_metrics["ccc"]
            best_ccc_epoch = epoch
        if improved_loss:
            save_text_video_checkpoint(best_loss_path, "best_loss", epoch, question_index, seed, model, optimizer, train_metrics, val_metrics, class_counts, class_weights, best_val_loss, best_val_ccc, best_loss_epoch, best_ccc_epoch, text_checkpoint_path, video_checkpoint_path)
        if improved_ccc:
            save_text_video_checkpoint(best_ccc_path, "best_ccc", epoch, question_index, seed, model, optimizer, train_metrics, val_metrics, class_counts, class_weights, best_val_loss, best_val_ccc, best_loss_epoch, best_ccc_epoch, text_checkpoint_path, video_checkpoint_path)
        history_rows.append({"Epoch": epoch, "Train_Loss": train_metrics["loss"], "Train_Accuracy": train_metrics["accuracy"], "Train_CCC": train_metrics["ccc"], "Train_RMSE": train_metrics["rmse"], "Train_MAE": train_metrics["mae"], "Val_Loss": val_metrics["loss"], "Val_Accuracy": val_metrics["accuracy"], "Val_CCC": val_metrics["ccc"], "Val_RMSE": val_metrics["rmse"], "Val_MAE": val_metrics["mae"], "Elapsed_Seconds": elapsed_seconds})
        print(f"Epoch {epoch:02d}/{FUSION_NUM_EPOCHS} | Train Loss {train_metrics['loss']:.6f} | Val Loss {val_metrics['loss']:.6f} | Val CCC {val_metrics['ccc']:.6f} | Val Acc {val_metrics['accuracy']:.6f} | Val RMSE {val_metrics['rmse']:.6f} | {elapsed_seconds:.2f}s")
    save_text_video_checkpoint(last_path, "last", FUSION_NUM_EPOCHS, question_index, seed, model, optimizer, train_metrics, val_metrics, class_counts, class_weights, best_val_loss, best_val_ccc, best_loss_epoch, best_ccc_epoch, text_checkpoint_path, video_checkpoint_path)
    history_df = pd.DataFrame(history_rows)
    history_df.to_csv(history_path, index=False)
    print("=" * 100)
    print(f"训练完成：Seed={seed}，Q{question_index + 1}")
    print(f"最佳Val Loss：{best_val_loss:.6f}，Epoch={best_loss_epoch}")
    print(f"最佳Val CCC：{best_val_ccc:.6f}，Epoch={best_ccc_epoch}")
    print("=" * 100)
    return {"seed": seed, "question_index": question_index, "question_number": question_index + 1, "question": PHQ8_QUESTION_COLUMNS[question_index], "best_val_loss": best_val_loss, "best_val_ccc": best_val_ccc, "best_loss_epoch": best_loss_epoch, "best_ccc_epoch": best_ccc_epoch, "best_loss_path": best_loss_path, "best_ccc_path": best_ccc_path, "last_path": last_path, "history_path": history_path}

In [121]:
text_video_q1_seed42_result = train_text_video_question(seed=42, question_index=0)

文本Best Loss： /workspace/E-DAIC/checkpoints/phq8_text_paperlike/q1_PHQ_8NoInterest/seed_42/best_loss.pt
视频Best Loss： /workspace/E-DAIC/checkpoints/phq8_video_paperlike/seed_42/question_1/best_loss.pt
文本保存Epoch： 13
视频保存Epoch： 2
文本缺少参数： []
文本多余参数： []
视频缺少参数： []
视频多余参数： []
开始训练Text+Video：Seed=42，Q1，PHQ_8NoInterest
类别数量： [81, 57, 20, 5]
ImbOLL权重： [1.4185717105865479, 1.6910496950149536, 2.854820728302002, 5.709641456604004]
文本预训练模型： /workspace/E-DAIC/checkpoints/phq8_text_paperlike/q1_PHQ_8NoInterest/seed_42/best_loss.pt
视频预训练模型： /workspace/E-DAIC/checkpoints/phq8_video_paperlike/seed_42/question_1/best_loss.pt
Epoch 01/20 | Train Loss 2.150871 | Val Loss 2.717357 | Val CCC 0.320330 | Val Acc 0.527273 | Val RMSE 0.962950 | 1.37s
Epoch 02/20 | Train Loss 1.317898 | Val Loss 3.563488 | Val CCC 0.374030 | Val Acc 0.600000 | Val RMSE 0.894427 | 1.21s
Epoch 03/20 | Train Loss 1.384124 | Val Loss 2.487444 | Val CCC 0.312109 | Val Acc 0.545455 | Val RMSE 0.894427 | 1.27s
Epoch 04/20 | Train Loss 1

In [122]:
print("Best Loss存在：", text_video_q1_seed42_result["best_loss_path"].exists())
print("Best CCC存在：", text_video_q1_seed42_result["best_ccc_path"].exists())
print("Last存在：", text_video_q1_seed42_result["last_path"].exists())
print("History存在：", text_video_q1_seed42_result["history_path"].exists())
print("Best Loss路径：", text_video_q1_seed42_result["best_loss_path"])
print("Best CCC路径：", text_video_q1_seed42_result["best_ccc_path"])

Best Loss存在： True
Best CCC存在： True
Last存在： True
History存在： True
Best Loss路径： /workspace/E-DAIC/checkpoints/phq8_text_video_paperlike/seed_42/question_1/best_loss.pt
Best CCC路径： /workspace/E-DAIC/checkpoints/phq8_text_video_paperlike/seed_42/question_1/best_ccc.pt


In [123]:
text_video_q1_best_ccc_path = text_video_q1_seed42_result["best_ccc_path"]
text_video_q1_checkpoint = torch.load(text_video_q1_best_ccc_path, map_location="cpu", weights_only=False)
restored_text_video_q1_model = TextVideoQuestMF(PretrainedTextEncoder(), PretrainedVideoEncoder()).to(device)
text_video_load_result = restored_text_video_q1_model.load_state_dict(text_video_q1_checkpoint["model_state_dict"], strict=True)
restored_q1_class_weights = torch.tensor(text_video_q1_checkpoint["class_weights"], dtype=torch.float32)
restored_q1_criterion = ImbOLLLoss(restored_q1_class_weights, alpha=ALPHA).to(device)
_, restored_text_video_val_loader = create_text_video_dataloaders(seed=42)
restored_q1_val_metrics = run_text_video_epoch(restored_text_video_q1_model, restored_text_video_val_loader, question_index=0, criterion=restored_q1_criterion)


In [124]:
print("Checkpoint：", text_video_q1_best_ccc_path)
print("类型：", text_video_q1_checkpoint["checkpoint_type"])
print("Epoch：", text_video_q1_checkpoint["epoch"])
print("Video变体：", text_video_q1_checkpoint["video_variant"])
print("文本预训练Checkpoint：", text_video_q1_checkpoint["text_pretrained_checkpoint"])
print("视频预训练Checkpoint：", text_video_q1_checkpoint["video_pretrained_checkpoint"])
print("缺少参数：", text_video_load_result.missing_keys)
print("多余参数：", text_video_load_result.unexpected_keys)
print("文本处于训练模式：", restored_text_video_q1_model.text_encoder.training)
print("文本可训练参数：", sum(parameter.numel() for parameter in restored_text_video_q1_model.text_encoder.parameters() if parameter.requires_grad))

Checkpoint： /workspace/E-DAIC/checkpoints/phq8_text_video_paperlike/seed_42/question_1/best_ccc.pt
类型： best_ccc
Epoch： 6
Video变体： masked_padding_current_video
文本预训练Checkpoint： /workspace/E-DAIC/checkpoints/phq8_text_paperlike/q1_PHQ_8NoInterest/seed_42/best_loss.pt
视频预训练Checkpoint： /workspace/E-DAIC/checkpoints/phq8_video_paperlike/seed_42/question_1/best_loss.pt
缺少参数： []
多余参数： []
文本处于训练模式： False
文本可训练参数： 0


In [125]:
text_video_checkpoint_metric_names = ["loss", "accuracy", "micro_f1", "macro_f1", "weighted_f1", "ccc", "rmse", "mae"]
print("保存指标与恢复指标对比：")
for metric_name in text_video_checkpoint_metric_names:
    saved_value = float(text_video_q1_checkpoint["val_metrics"][metric_name])
    restored_value = float(restored_q1_val_metrics[metric_name])
    difference = abs(saved_value - restored_value)
    print(f"{metric_name:>12} | 保存 {saved_value:.9f} | 恢复 {restored_value:.9f} | 差值 {difference:.3e}")

保存指标与恢复指标对比：
        loss | 保存 2.729871696 | 恢复 2.729871696 | 差值 0.000e+00
    accuracy | 保存 0.618181825 | 恢复 0.618181825 | 差值 0.000e+00
    micro_f1 | 保存 0.618181825 | 恢复 0.618181825 | 差值 0.000e+00
    macro_f1 | 保存 0.400655136 | 恢复 0.400655136 | 差值 0.000e+00
 weighted_f1 | 保存 0.598017915 | 恢复 0.598017915 | 差值 0.000e+00
         ccc | 保存 0.430304824 | 恢复 0.430304824 | 差值 0.000e+00
        rmse | 保存 0.797724068 | 恢复 0.797724068 | 差值 0.000e+00
         mae | 保存 0.454545468 | 恢复 0.454545468 | 差值 0.000e+00


In [126]:
maximum_metric_difference = max(abs(float(text_video_q1_checkpoint["val_metrics"][metric_name]) - float(restored_q1_val_metrics[metric_name])) for metric_name in text_video_checkpoint_metric_names)
saved_confusion_matrix = torch.tensor(text_video_q1_checkpoint["val_metrics"]["confusion_matrix"])
restored_confusion_matrix = restored_q1_val_metrics["confusion_matrix"]
assert text_video_load_result.missing_keys == []
assert text_video_load_result.unexpected_keys == []
assert maximum_metric_difference < 1e-6
assert torch.equal(saved_confusion_matrix, restored_confusion_matrix)
assert sum(parameter.numel() for parameter in restored_text_video_q1_model.text_encoder.parameters() if parameter.requires_grad) == 0
print("最大指标差值：", maximum_metric_difference)
print("Checkpoint恢复验证通过")

最大指标差值： 0.0
Checkpoint恢复验证通过


In [128]:
del restored_text_video_q1_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [129]:
text_video_seed42_results = [text_video_q1_seed42_result]
for question_index in range(1, len(PHQ8_QUESTION_COLUMNS)):
    question_result = train_text_video_question(seed=42, question_index=question_index)
    text_video_seed42_results.append(question_result)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

文本Best Loss： /workspace/E-DAIC/checkpoints/phq8_text_paperlike/q2_PHQ_8Depressed/seed_42/best_loss.pt
视频Best Loss： /workspace/E-DAIC/checkpoints/phq8_video_paperlike/seed_42/question_2/best_loss.pt
文本保存Epoch： 9
视频保存Epoch： 8
文本缺少参数： []
文本多余参数： []
视频缺少参数： []
视频多余参数： []
开始训练Text+Video：Seed=42，Q2，PHQ_8Depressed
类别数量： [68, 63, 18, 14]
ImbOLL权重： [1.548243761062622, 1.6085091829299927, 3.009244918823242, 3.412163257598877]
文本预训练模型： /workspace/E-DAIC/checkpoints/phq8_text_paperlike/q2_PHQ_8Depressed/seed_42/best_loss.pt
视频预训练模型： /workspace/E-DAIC/checkpoints/phq8_video_paperlike/seed_42/question_2/best_loss.pt
Epoch 01/20 | Train Loss 2.314834 | Val Loss 2.288081 | Val CCC 0.375887 | Val Acc 0.545455 | Val RMSE 0.934199 | 1.21s
Epoch 02/20 | Train Loss 1.596882 | Val Loss 2.111849 | Val CCC 0.471816 | Val Acc 0.472727 | Val RMSE 0.914529 | 1.41s
Epoch 03/20 | Train Loss 1.601615 | Val Loss 2.061613 | Val CCC 0.531440 | Val Acc 0.545455 | Val RMSE 0.873863 | 0.89s
Epoch 04/20 | Train Loss 1.712

In [130]:
print("Seed 42结果数量：", len(text_video_seed42_results))
print("Best Loss全部存在：", all(result["best_loss_path"].exists() for result in text_video_seed42_results))
print("Best CCC全部存在：", all(result["best_ccc_path"].exists() for result in text_video_seed42_results))
print("Last全部存在：", all(result["last_path"].exists() for result in text_video_seed42_results))
print("History全部存在：", all(result["history_path"].exists() for result in text_video_seed42_results))

Seed 42结果数量： 8
Best Loss全部存在： True
Best CCC全部存在： True
Last全部存在： True
History全部存在： True


In [131]:
text_video_seed42_summary_rows = []
for result in text_video_seed42_results:
    text_video_seed42_summary_rows.append({"Seed": result["seed"], "Question_Index": result["question_index"], "Question_Number": result["question_number"], "Question_Name": result["question"], "Best_Val_Loss": result["best_val_loss"], "Best_Loss_Epoch": result["best_loss_epoch"], "Best_Val_CCC": result["best_val_ccc"], "Best_CCC_Epoch": result["best_ccc_epoch"]})
text_video_seed42_summary_df = pd.DataFrame(text_video_seed42_summary_rows)
text_video_seed42_summary_path = TEXT_VIDEO_CHECKPOINT_ROOT / "seed_42_question_summary.csv"
text_video_seed42_summary_df.to_csv(text_video_seed42_summary_path, index=False)
display(text_video_seed42_summary_df)
print("汇总文件：", text_video_seed42_summary_path)

,Seed,Question_Index,Question_Number,Question_Name,Best_Val_Loss,Best_Loss_Epoch,Best_Val_CCC,Best_CCC_Epoch
0,42,0,1,PHQ_8NoInterest,2.487444,3,0.430305,6
1,42,1,2,PHQ_8Depressed,1.972233,5,0.560924,9
2,42,2,3,PHQ_8Sleep,2.296455,7,0.495027,13
3,42,3,4,PHQ_8Tired,2.202472,13,0.393745,9
4,42,4,5,PHQ_8Appetite,2.113170,4,0.521569,7
5,42,5,6,PHQ_8Failure,2.009271,9,0.505618,15
6,42,6,7,PHQ_8Concentrating,1.949242,4,0.561753,10
7,42,7,8,PHQ_8Moving,1.833960,1,0.345725,19


汇总文件： /workspace/E-DAIC/checkpoints/phq8_text_video_paperlike/seed_42_question_summary.csv


In [132]:
text_video_seed42_best_ccc_rows = []
for result in text_video_seed42_results:
    checkpoint = torch.load(result["best_ccc_path"], map_location="cpu", weights_only=False)
    text_video_seed42_best_ccc_rows.append({"Question_Number": checkpoint["question_number"], "Question_Name": checkpoint["question"], "Checkpoint_Epoch": checkpoint["epoch"], "CCC": checkpoint["val_metrics"]["ccc"], "RMSE": checkpoint["val_metrics"]["rmse"], "MAE": checkpoint["val_metrics"]["mae"], "Accuracy": checkpoint["val_metrics"]["accuracy"]})
text_video_seed42_best_ccc_df = pd.DataFrame(text_video_seed42_best_ccc_rows)
display(text_video_seed42_best_ccc_df)

,Question_Number,Question_Name,Checkpoint_Epoch,CCC,RMSE,MAE,Accuracy
0,1,PHQ_8NoInterest,6,0.430305,0.797724,0.454545,0.618182
1,2,PHQ_8Depressed,9,0.560924,0.831209,0.509091,0.581818
2,3,PHQ_8Sleep,13,0.495027,1.144155,0.763636,0.454545
3,4,PHQ_8Tired,9,0.393745,0.884205,0.672727,0.381818
4,5,PHQ_8Appetite,7,0.521569,0.943880,0.600000,0.527273
5,6,PHQ_8Failure,15,0.505618,1.009050,0.690909,0.472727
6,7,PHQ_8Concentrating,10,0.561753,0.809040,0.581818,0.454545
7,8,PHQ_8Moving,19,0.345725,0.762770,0.472727,0.581818


In [133]:
def evaluate_text_video_seed_on_validation(seed):
    val_loader = DataLoader(text_video_val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, drop_last=False)
    question_prediction_list = []
    selected_checkpoint_rows = []
    for question_index in range(len(PHQ8_QUESTION_COLUMNS)):
        checkpoint_path = TEXT_VIDEO_CHECKPOINT_ROOT / f"seed_{seed}" / f"question_{question_index + 1}" / "best_ccc.pt"
        if not checkpoint_path.exists():
            raise FileNotFoundError(f"Checkpoint不存在：{checkpoint_path}")
        checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
        if checkpoint["seed"] != seed or checkpoint["question_index"] != question_index or checkpoint["checkpoint_type"] != "best_ccc":
            raise RuntimeError(f"Checkpoint元数据错误：{checkpoint_path}")
        model = TextVideoQuestMF(PretrainedTextEncoder(), PretrainedVideoEncoder()).to(device)
        load_result = model.load_state_dict(checkpoint["model_state_dict"], strict=True)
        if load_result.missing_keys or load_result.unexpected_keys:
            raise RuntimeError(f"Checkpoint参数不一致：{checkpoint_path}")
        model.eval()
        batch_prediction_list = []
        with torch.no_grad():
            for text_features, text_masks, video_features, video_masks, all_labels in val_loader:
                logits = model(text_features.to(device), text_masks.to(device), video_features.to(device), video_masks.to(device))
                batch_prediction_list.append(logits.argmax(dim=1).cpu())
        question_predictions = torch.cat(batch_prediction_list)
        if len(question_predictions) != len(text_video_val_dataset):
            raise RuntimeError(f"Q{question_index + 1}预测数量错误")
        question_prediction_list.append(question_predictions)
        selected_checkpoint_rows.append({"Seed": seed, "Question_Index": question_index, "Question_Number": question_index + 1, "Question_Name": PHQ8_QUESTION_COLUMNS[question_index], "Checkpoint_Path": str(checkpoint_path), "Checkpoint_Epoch": checkpoint["epoch"], "Validation_CCC": checkpoint["val_metrics"]["ccc"]})
        print(f"Q{question_index + 1} | Epoch {checkpoint['epoch']:02d} | Val CCC {checkpoint['val_metrics']['ccc']:.6f}")
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    question_predictions = torch.stack(question_prediction_list, dim=1)
    question_targets = text_video_val_dataset.labels.clone()
    predicted_totals = question_predictions.sum(dim=1)
    true_totals = question_targets.sum(dim=1)
    total_ccc = calculate_ccc(predicted_totals, true_totals)
    total_rmse = float(torch.sqrt(torch.mean((predicted_totals.float() - true_totals.float()) ** 2)).item())
    total_mae = float(torch.mean(torch.abs(predicted_totals.float() - true_totals.float())).item())
    exact_accuracy = float((predicted_totals == true_totals).float().mean().item())
    prediction_df = pd.DataFrame({"Participant_ID": text_video_val_dataset.participant_ids})
    for question_index in range(len(PHQ8_QUESTION_COLUMNS)):
        prediction_df[f"Q{question_index + 1}_Predicted"] = question_predictions[:, question_index].numpy()
        prediction_df[f"Q{question_index + 1}_True"] = question_targets[:, question_index].numpy()
    prediction_df["Predicted_Total"] = predicted_totals.numpy()
    prediction_df["True_Total"] = true_totals.numpy()
    prediction_path = TEXT_VIDEO_CHECKPOINT_ROOT / f"seed_{seed}_validation_predictions.csv"
    selected_checkpoint_path = TEXT_VIDEO_CHECKPOINT_ROOT / f"seed_{seed}_validation_selected_checkpoints.csv"
    prediction_df.to_csv(prediction_path, index=False)
    pd.DataFrame(selected_checkpoint_rows).to_csv(selected_checkpoint_path, index=False)
    print("=" * 70)
    print(f"Text+Video Seed {seed}验证结果")
    print("=" * 70)
    print("验证参与者数量：", len(prediction_df))
    print(f"总分CCC：{total_ccc:.6f}")
    print(f"总分RMSE：{total_rmse:.6f}")
    print(f"总分MAE：{total_mae:.6f}")
    print(f"完全相等比例：{exact_accuracy:.6f}")
    print(f"预测总分范围：{int(predicted_totals.min())}～{int(predicted_totals.max())}")
    print(f"真实总分范围：{int(true_totals.min())}～{int(true_totals.max())}")
    print("预测文件：", prediction_path)
    return {"seed": seed, "sample_count": len(prediction_df), "ccc": total_ccc, "rmse": total_rmse, "mae": total_mae, "exact_accuracy": exact_accuracy, "question_predictions": question_predictions, "question_targets": question_targets, "predicted_totals": predicted_totals, "true_totals": true_totals, "prediction_path": prediction_path, "selected_checkpoint_path": selected_checkpoint_path}

In [134]:
text_video_seed42_validation_result = evaluate_text_video_seed_on_validation(seed=42)

Q1 | Epoch 06 | Val CCC 0.430305
Q2 | Epoch 09 | Val CCC 0.560924
Q3 | Epoch 13 | Val CCC 0.495027
Q4 | Epoch 09 | Val CCC 0.393745
Q5 | Epoch 07 | Val CCC 0.521569
Q6 | Epoch 15 | Val CCC 0.505618
Q7 | Epoch 10 | Val CCC 0.561753
Q8 | Epoch 19 | Val CCC 0.345725
Text+Video Seed 42验证结果
验证参与者数量： 55
总分CCC：0.680283
总分RMSE：4.421024
总分MAE：3.218182
完全相等比例：0.072727
预测总分范围：1～22
真实总分范围：0～20
预测文件： /workspace/E-DAIC/checkpoints/phq8_text_video_paperlike/seed_42_validation_predictions.csv


In [135]:
def train_all_text_video_questions_for_seed(seed):
    seed_results = []
    print("#" * 100)
    print(f"开始训练Text+Video全部问题：Seed={seed}")
    print("#" * 100)
    for question_index in range(len(PHQ8_QUESTION_COLUMNS)):
        result = train_text_video_question(seed=seed, question_index=question_index)
        seed_results.append(result)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    summary_rows = []
    for result in seed_results:
        summary_rows.append({"Seed": result["seed"], "Question_Index": result["question_index"], "Question_Number": result["question_number"], "Question_Name": result["question"], "Best_Val_Loss": result["best_val_loss"], "Best_Loss_Epoch": result["best_loss_epoch"], "Best_Val_CCC": result["best_val_ccc"], "Best_CCC_Epoch": result["best_ccc_epoch"]})
    summary_df = pd.DataFrame(summary_rows)
    summary_path = TEXT_VIDEO_CHECKPOINT_ROOT / f"seed_{seed}_question_summary.csv"
    summary_df.to_csv(summary_path, index=False)
    print(f"Seed {seed}全部问题训练完成")
    print("汇总文件：", summary_path)
    return seed_results, summary_df

In [136]:
text_video_seed100_results, text_video_seed100_summary_df = train_all_text_video_questions_for_seed(seed=100)
display(text_video_seed100_summary_df)

####################################################################################################
开始训练Text+Video全部问题：Seed=100
####################################################################################################
文本Best Loss： /workspace/E-DAIC/checkpoints/phq8_text_paperlike/q1_PHQ_8NoInterest/seed_100/best_loss.pt
视频Best Loss： /workspace/E-DAIC/checkpoints/phq8_video_paperlike/seed_100/question_1/best_loss.pt
文本保存Epoch： 10
视频保存Epoch： 14
文本缺少参数： []
文本多余参数： []
视频缺少参数： []
视频多余参数： []
开始训练Text+Video：Seed=100，Q1，PHQ_8NoInterest
类别数量： [81, 57, 20, 5]
ImbOLL权重： [1.4185717105865479, 1.6910496950149536, 2.854820728302002, 5.709641456604004]
文本预训练模型： /workspace/E-DAIC/checkpoints/phq8_text_paperlike/q1_PHQ_8NoInterest/seed_100/best_loss.pt
视频预训练模型： /workspace/E-DAIC/checkpoints/phq8_video_paperlike/seed_100/question_1/best_loss.pt
Epoch 01/20 | Train Loss 2.345769 | Val Loss 2.816593 | Val CCC 0.258903 | Val Acc 0.490909 | Val RMSE 1.009050 | 1.72s
Epoch 02/20 | Train Loss 1.495

,Seed,Question_Index,Question_Number,Question_Name,Best_Val_Loss,Best_Loss_Epoch,Best_Val_CCC,Best_CCC_Epoch
0,100,0,1,PHQ_8NoInterest,2.533762,4,0.455722,20
1,100,1,2,PHQ_8Depressed,1.931162,17,0.514706,4
2,100,2,3,PHQ_8Sleep,2.276807,3,0.396825,14
3,100,3,4,PHQ_8Tired,2.132430,8,0.478673,13
4,100,4,5,PHQ_8Appetite,2.150371,11,0.497122,17
5,100,5,6,PHQ_8Failure,2.063133,4,0.447357,5
6,100,6,7,PHQ_8Concentrating,2.094383,17,0.490192,7
7,100,7,8,PHQ_8Moving,2.357591,10,0.306064,5


In [137]:
text_video_seed1234_results, text_video_seed1234_summary_df = train_all_text_video_questions_for_seed(seed=1234)
display(text_video_seed1234_summary_df)

####################################################################################################
开始训练Text+Video全部问题：Seed=1234
####################################################################################################
文本Best Loss： /workspace/E-DAIC/checkpoints/phq8_text_paperlike/q1_PHQ_8NoInterest/seed_1234/best_loss.pt
视频Best Loss： /workspace/E-DAIC/checkpoints/phq8_video_paperlike/seed_1234/question_1/best_loss.pt
文本保存Epoch： 7
视频保存Epoch： 15
文本缺少参数： []
文本多余参数： []
视频缺少参数： []
视频多余参数： []
开始训练Text+Video：Seed=1234，Q1，PHQ_8NoInterest
类别数量： [81, 57, 20, 5]
ImbOLL权重： [1.4185717105865479, 1.6910496950149536, 2.854820728302002, 5.709641456604004]
文本预训练模型： /workspace/E-DAIC/checkpoints/phq8_text_paperlike/q1_PHQ_8NoInterest/seed_1234/best_loss.pt
视频预训练模型： /workspace/E-DAIC/checkpoints/phq8_video_paperlike/seed_1234/question_1/best_loss.pt
Epoch 01/20 | Train Loss 2.209556 | Val Loss 2.475116 | Val CCC 0.208096 | Val Acc 0.545455 | Val RMSE 0.981650 | 1.11s
Epoch 02/20 | Train Loss 

,Seed,Question_Index,Question_Number,Question_Name,Best_Val_Loss,Best_Loss_Epoch,Best_Val_CCC,Best_CCC_Epoch
0,1234,0,1,PHQ_8NoInterest,2.439806,16,0.488372,8
1,1234,1,2,PHQ_8Depressed,1.978245,4,0.538213,12
2,1234,2,3,PHQ_8Sleep,2.226618,4,0.477732,12
3,1234,3,4,PHQ_8Tired,2.109944,5,0.550302,2
4,1234,4,5,PHQ_8Appetite,2.033050,19,0.514473,19
5,1234,5,6,PHQ_8Failure,1.993349,1,0.506329,2
6,1234,6,7,PHQ_8Concentrating,2.043192,13,0.514056,11
7,1234,7,8,PHQ_8Moving,1.886370,3,0.157549,3


In [138]:
all_text_video_training_results = {}
all_text_video_training_results[42] = text_video_seed42_results
all_text_video_training_results[100] = text_video_seed100_results
all_text_video_training_results[1234] = text_video_seed1234_results

In [139]:
for seed, seed_results in all_text_video_training_results.items():
    print("=" * 70)
    print("Seed：", seed)
    print("问题数量：", len(seed_results))
    print("Best Loss全部存在：", all(result["best_loss_path"].exists() for result in seed_results))
    print("Best CCC全部存在：", all(result["best_ccc_path"].exists() for result in seed_results))
    print("Last全部存在：", all(result["last_path"].exists() for result in seed_results))
    print("History全部存在：", all(result["history_path"].exists() for result in seed_results))

Seed： 42
问题数量： 8
Best Loss全部存在： True
Best CCC全部存在： True
Last全部存在： True
History全部存在： True
Seed： 100
问题数量： 8
Best Loss全部存在： True
Best CCC全部存在： True
Last全部存在： True
History全部存在： True
Seed： 1234
问题数量： 8
Best Loss全部存在： True
Best CCC全部存在： True
Last全部存在： True
History全部存在： True


In [140]:
text_video_seed100_validation_result = evaluate_text_video_seed_on_validation(seed=100)
text_video_seed1234_validation_result = evaluate_text_video_seed_on_validation(seed=1234)

Q1 | Epoch 20 | Val CCC 0.455722
Q2 | Epoch 04 | Val CCC 0.514706
Q3 | Epoch 14 | Val CCC 0.396825
Q4 | Epoch 13 | Val CCC 0.478673
Q5 | Epoch 17 | Val CCC 0.497122
Q6 | Epoch 05 | Val CCC 0.447357
Q7 | Epoch 07 | Val CCC 0.490192
Q8 | Epoch 05 | Val CCC 0.306064
Text+Video Seed 100验证结果
验证参与者数量： 55
总分CCC：0.625042
总分RMSE：4.837355
总分MAE：3.654546
完全相等比例：0.054545
预测总分范围：0～17
真实总分范围：0～20
预测文件： /workspace/E-DAIC/checkpoints/phq8_text_video_paperlike/seed_100_validation_predictions.csv
Q1 | Epoch 08 | Val CCC 0.488372
Q2 | Epoch 12 | Val CCC 0.538213
Q3 | Epoch 12 | Val CCC 0.477732
Q4 | Epoch 02 | Val CCC 0.550302
Q5 | Epoch 19 | Val CCC 0.514473
Q6 | Epoch 02 | Val CCC 0.506329
Q7 | Epoch 11 | Val CCC 0.514056
Q8 | Epoch 03 | Val CCC 0.157549
Text+Video Seed 1234验证结果
验证参与者数量： 55
总分CCC：0.661026
总分RMSE：4.568668
总分MAE：3.236364
完全相等比例：0.181818
预测总分范围：0～20
真实总分范围：0～20
预测文件： /workspace/E-DAIC/checkpoints/phq8_text_video_paperlike/seed_1234_validation_predictions.csv


In [141]:
all_text_video_validation_results = [text_video_seed42_validation_result, text_video_seed100_validation_result, text_video_seed1234_validation_result]
text_video_validation_seed_rows = []
for result in all_text_video_validation_results:
    text_video_validation_seed_rows.append({"Seed": result["seed"], "Sample_Count": result["sample_count"], "CCC": result["ccc"], "RMSE": result["rmse"], "MAE": result["mae"], "Exact_Accuracy": result["exact_accuracy"], "Predicted_Min": int(result["predicted_totals"].min()), "Predicted_Max": int(result["predicted_totals"].max()), "True_Min": int(result["true_totals"].min()), "True_Max": int(result["true_totals"].max())})
text_video_validation_seed_summary_df = pd.DataFrame(text_video_validation_seed_rows)
display(text_video_validation_seed_summary_df)

,Seed,Sample_Count,CCC,RMSE,MAE,Exact_Accuracy,Predicted_Min,Predicted_Max,True_Min,True_Max
0,42,55,0.680283,4.421024,3.218182,0.072727,1,22,0,20
1,100,55,0.625042,4.837355,3.654546,0.054545,0,17,0,20
2,1234,55,0.661026,4.568668,3.236364,0.181818,0,20,0,20


In [142]:
text_video_validation_statistics_rows = []
for metric_name in ["CCC", "RMSE", "MAE", "Exact_Accuracy"]:
    text_video_validation_statistics_rows.append({"Metric": metric_name, "Mean": text_video_validation_seed_summary_df[metric_name].mean(), "Std": text_video_validation_seed_summary_df[metric_name].std(ddof=1)})
text_video_validation_statistics_df = pd.DataFrame(text_video_validation_statistics_rows)
display(text_video_validation_statistics_df)

,Metric,Mean,Std
0,CCC,0.655450,0.028039
1,RMSE,4.609015,0.211078
2,MAE,3.369697,0.246854
3,Exact_Accuracy,0.103030,0.068835


In [143]:
text_video_validation_statistics_rows = []
for metric_name in ["CCC", "RMSE", "MAE", "Exact_Accuracy"]:
    text_video_validation_statistics_rows.append({"Metric": metric_name, "Mean": text_video_validation_seed_summary_df[metric_name].mean(), "Std": text_video_validation_seed_summary_df[metric_name].std(ddof=1)})
text_video_validation_statistics_df = pd.DataFrame(text_video_validation_statistics_rows)
display(text_video_validation_statistics_df)

,Metric,Mean,Std
0,CCC,0.655450,0.028039
1,RMSE,4.609015,0.211078
2,MAE,3.369697,0.246854
3,Exact_Accuracy,0.103030,0.068835


In [144]:
text_video_validation_results_path = TEXT_VIDEO_CHECKPOINT_ROOT / "three_seed_validation_results.csv"
text_video_validation_summary_path = TEXT_VIDEO_CHECKPOINT_ROOT / "three_seed_validation_summary.csv"
text_video_validation_seed_summary_df.to_csv(text_video_validation_results_path, index=False)
text_video_validation_statistics_df.to_csv(text_video_validation_summary_path, index=False)
print("逐Seed验证结果：", text_video_validation_results_path)
print("三Seed验证汇总：", text_video_validation_summary_path)
print("逐Seed文件存在：", text_video_validation_results_path.exists())
print("汇总文件存在：", text_video_validation_summary_path.exists())

逐Seed验证结果： /workspace/E-DAIC/checkpoints/phq8_text_video_paperlike/three_seed_validation_results.csv
三Seed验证汇总： /workspace/E-DAIC/checkpoints/phq8_text_video_paperlike/three_seed_validation_summary.csv
逐Seed文件存在： True
汇总文件存在： True


In [145]:
class TextVideoPHQ8TestDataset(Dataset):
    def __init__(self, metadata_df):
        super().__init__()
        self.metadata_df = metadata_df.reset_index(drop=True).copy()
        if "PHQ_Score" not in self.metadata_df.columns:
            raise KeyError(f"测试集缺少PHQ_Score字段：{self.metadata_df.columns.tolist()}")
        self.participant_ids = self.metadata_df["Participant_ID"].astype(int).tolist()
        self.true_totals = torch.tensor(self.metadata_df["PHQ_Score"].to_numpy(), dtype=torch.long)
        text_feature_list = []
        text_mask_list = []
        video_feature_list = []
        video_mask_list = []
        for index, participant_id in enumerate(self.participant_ids):
            text_features, text_mask, video_features, video_mask = load_text_video_cache(participant_id)
            text_feature_list.append(text_features)
            text_mask_list.append(text_mask)
            video_feature_list.append(video_features)
            video_mask_list.append(video_mask)
            if (index + 1) % 20 == 0 or index + 1 == len(self.participant_ids):
                print(f"test Text+Video加载进度：{index + 1}/{len(self.participant_ids)}")
        self.text_features = torch.stack(text_feature_list, dim=0)
        self.text_masks = torch.stack(text_mask_list, dim=0)
        self.video_features = torch.stack(video_feature_list, dim=0)
        self.video_masks = torch.stack(video_mask_list, dim=0)
        if self.text_features.shape != (len(self.participant_ids), MAX_TURNS, TEXT_FEATURE_DIM):
            raise RuntimeError(f"测试文本形状错误：{self.text_features.shape}")
        if self.video_features.shape != (len(self.participant_ids), MAX_TURNS, VIDEO_FEATURE_DIM):
            raise RuntimeError(f"测试视频形状错误：{self.video_features.shape}")
        if not torch.isfinite(self.text_features).all():
            raise RuntimeError("测试文本包含NaN或Inf")
        if not torch.isfinite(self.video_features).all():
            raise RuntimeError("测试视频包含NaN或Inf")

    def __len__(self):
        return len(self.participant_ids)

    def __getitem__(self, index):
        return self.text_features[index], self.text_masks[index], self.video_features[index], self.video_masks[index], self.true_totals[index], self.participant_ids[index]

In [146]:
text_video_test_dataset = TextVideoPHQ8TestDataset(video_test_metadata_df)
text_video_test_loader = DataLoader(text_video_test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, drop_last=False)

test Text+Video加载进度：20/56
test Text+Video加载进度：40/56
test Text+Video加载进度：56/56


In [147]:
test_text_features, test_text_masks, test_video_features, test_video_masks, test_true_totals, test_participant_ids = next(iter(text_video_test_loader))
print("测试参与者数量：", len(text_video_test_dataset))
print("测试批次数：", len(text_video_test_loader))
print("测试文本特征：", test_text_features.shape)
print("测试文本Mask：", test_text_masks.shape)
print("测试视频特征：", test_video_features.shape)
print("测试视频Mask：", test_video_masks.shape)
print("测试真实总分：", test_true_totals.shape)
print("测试参与者：", test_participant_ids)
print("文本全部有限：", bool(torch.isfinite(test_text_features).all()))
print("视频全部有限：", bool(torch.isfinite(test_video_features).all()))
print("真实总分范围：", int(text_video_test_dataset.true_totals.min()), "～", int(text_video_test_dataset.true_totals.max()))

测试参与者数量： 56
测试批次数： 6
测试文本特征： torch.Size([10, 120, 768])
测试文本Mask： torch.Size([10, 120])
测试视频特征： torch.Size([10, 120, 2048])
测试视频Mask： torch.Size([10, 120])
测试真实总分： torch.Size([10])
测试参与者： tensor([600, 602, 604, 605, 606, 607, 609, 615, 618, 619])
文本全部有限： True
视频全部有限： True
真实总分范围： 0 ～ 22


In [148]:
def evaluate_text_video_seed_on_test(seed):
    question_prediction_list = []
    selected_checkpoint_rows = []
    for question_index in range(len(PHQ8_QUESTION_COLUMNS)):
        checkpoint_path = TEXT_VIDEO_CHECKPOINT_ROOT / f"seed_{seed}" / f"question_{question_index + 1}" / "best_ccc.pt"
        if not checkpoint_path.exists():
            raise FileNotFoundError(f"Checkpoint不存在：{checkpoint_path}")
        checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
        if checkpoint["seed"] != seed or checkpoint["question_index"] != question_index or checkpoint["checkpoint_type"] != "best_ccc":
            raise RuntimeError(f"Checkpoint元数据错误：{checkpoint_path}")
        model = TextVideoQuestMF(PretrainedTextEncoder(), PretrainedVideoEncoder()).to(device)
        load_result = model.load_state_dict(checkpoint["model_state_dict"], strict=True)
        if load_result.missing_keys or load_result.unexpected_keys:
            raise RuntimeError(f"Checkpoint参数不一致：{checkpoint_path}")
        model.eval()
        batch_prediction_list = []
        with torch.no_grad():
            for text_features, text_masks, video_features, video_masks, true_totals, participant_ids in text_video_test_loader:
                logits = model(text_features.to(device), text_masks.to(device), video_features.to(device), video_masks.to(device))
                batch_prediction_list.append(logits.argmax(dim=1).cpu())
        question_predictions = torch.cat(batch_prediction_list)
        if len(question_predictions) != len(text_video_test_dataset):
            raise RuntimeError(f"Q{question_index + 1}预测数量错误")
        question_prediction_list.append(question_predictions)
        selected_checkpoint_rows.append({"Seed": seed, "Question_Index": question_index, "Question_Number": question_index + 1, "Question_Name": PHQ8_QUESTION_COLUMNS[question_index], "Checkpoint_Path": str(checkpoint_path), "Checkpoint_Epoch": checkpoint["epoch"], "Validation_CCC": checkpoint["val_metrics"]["ccc"], "Video_Variant": checkpoint["video_variant"]})
        print(f"Seed {seed} | Q{question_index + 1} | Epoch {checkpoint['epoch']:02d} | Val CCC {checkpoint['val_metrics']['ccc']:.6f}")
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    question_predictions = torch.stack(question_prediction_list, dim=1)
    predicted_totals = question_predictions.sum(dim=1)
    true_totals = text_video_test_dataset.true_totals.clone()
    total_ccc = calculate_ccc(predicted_totals, true_totals)
    total_rmse = float(torch.sqrt(torch.mean((predicted_totals.float() - true_totals.float()) ** 2)).item())
    total_mae = float(torch.mean(torch.abs(predicted_totals.float() - true_totals.float())).item())
    exact_accuracy = float((predicted_totals == true_totals).float().mean().item())
    prediction_df = pd.DataFrame({"Participant_ID": text_video_test_dataset.participant_ids})
    for question_index in range(len(PHQ8_QUESTION_COLUMNS)):
        prediction_df[f"Q{question_index + 1}_Predicted"] = question_predictions[:, question_index].numpy()
    prediction_df["Predicted_Total"] = predicted_totals.numpy()
    prediction_df["True_Total"] = true_totals.numpy()
    prediction_path = TEXT_VIDEO_CHECKPOINT_ROOT / f"seed_{seed}_test_predictions_56.csv"
    selected_checkpoint_path = TEXT_VIDEO_CHECKPOINT_ROOT / f"seed_{seed}_test_selected_checkpoints.csv"
    prediction_df.to_csv(prediction_path, index=False)
    pd.DataFrame(selected_checkpoint_rows).to_csv(selected_checkpoint_path, index=False)
    print("=" * 70)
    print(f"Text+Video Seed {seed}独立测试结果")
    print("=" * 70)
    print("测试参与者数量：", len(prediction_df))
    print(f"CCC：{total_ccc:.6f}")
    print(f"RMSE：{total_rmse:.6f}")
    print(f"MAE：{total_mae:.6f}")
    print(f"完全相等比例：{exact_accuracy:.6f}")
    print(f"预测总分范围：{int(predicted_totals.min())}～{int(predicted_totals.max())}")
    print(f"真实总分范围：{int(true_totals.min())}～{int(true_totals.max())}")
    print("预测文件：", prediction_path)
    return {"seed": seed, "sample_count": len(prediction_df), "ccc": total_ccc, "rmse": total_rmse, "mae": total_mae, "exact_accuracy": exact_accuracy, "predicted_totals": predicted_totals, "true_totals": true_totals, "prediction_path": prediction_path, "selected_checkpoint_path": selected_checkpoint_path}

In [149]:
text_video_seed42_test_result = evaluate_text_video_seed_on_test(seed=42)
text_video_seed100_test_result = evaluate_text_video_seed_on_test(seed=100)
text_video_seed1234_test_result = evaluate_text_video_seed_on_test(seed=1234)

Seed 42 | Q1 | Epoch 06 | Val CCC 0.430305
Seed 42 | Q2 | Epoch 09 | Val CCC 0.560924
Seed 42 | Q3 | Epoch 13 | Val CCC 0.495027
Seed 42 | Q4 | Epoch 09 | Val CCC 0.393745
Seed 42 | Q5 | Epoch 07 | Val CCC 0.521569
Seed 42 | Q6 | Epoch 15 | Val CCC 0.505618
Seed 42 | Q7 | Epoch 10 | Val CCC 0.561753
Seed 42 | Q8 | Epoch 19 | Val CCC 0.345725
Text+Video Seed 42独立测试结果
测试参与者数量： 56
CCC：0.586931
RMSE：5.896427
MAE：4.446429
完全相等比例：0.089286
预测总分范围：1～22
真实总分范围：0～22
预测文件： /workspace/E-DAIC/checkpoints/phq8_text_video_paperlike/seed_42_test_predictions_56.csv
Seed 100 | Q1 | Epoch 20 | Val CCC 0.455722
Seed 100 | Q2 | Epoch 04 | Val CCC 0.514706
Seed 100 | Q3 | Epoch 14 | Val CCC 0.396825
Seed 100 | Q4 | Epoch 13 | Val CCC 0.478673
Seed 100 | Q5 | Epoch 17 | Val CCC 0.497122
Seed 100 | Q6 | Epoch 05 | Val CCC 0.447357
Seed 100 | Q7 | Epoch 07 | Val CCC 0.490192
Seed 100 | Q8 | Epoch 05 | Val CCC 0.306064
Text+Video Seed 100独立测试结果
测试参与者数量： 56
CCC：0.606763
RMSE：5.791003
MAE：4.464286
完全相等比例：0.125000

In [150]:
all_text_video_test_results = [text_video_seed42_test_result, text_video_seed100_test_result, text_video_seed1234_test_result]
text_video_test_seed_rows = []
for result in all_text_video_test_results:
    text_video_test_seed_rows.append({"Seed": result["seed"], "Sample_Count": result["sample_count"], "CCC": result["ccc"], "RMSE": result["rmse"], "MAE": result["mae"], "Exact_Accuracy": result["exact_accuracy"], "Predicted_Min": int(result["predicted_totals"].min()), "Predicted_Max": int(result["predicted_totals"].max()), "True_Min": int(result["true_totals"].min()), "True_Max": int(result["true_totals"].max())})
text_video_test_seed_summary_df = pd.DataFrame(text_video_test_seed_rows)
display(text_video_test_seed_summary_df)

,Seed,Sample_Count,CCC,RMSE,MAE,Exact_Accuracy,Predicted_Min,Predicted_Max,True_Min,True_Max
0,42,56,0.586931,5.896427,4.446429,0.089286,1,22,0,22
1,100,56,0.606763,5.791003,4.464286,0.125000,0,20,0,22
2,1234,56,0.633930,5.390137,4.017857,0.107143,0,20,0,22


In [151]:
text_video_three_seed_rows = []
for metric_name in ["CCC", "RMSE", "MAE", "Exact_Accuracy"]:
    text_video_three_seed_rows.append({"Metric": metric_name, "Mean": text_video_test_seed_summary_df[metric_name].mean(), "Std": text_video_test_seed_summary_df[metric_name].std(ddof=1)})
text_video_three_seed_summary_df = pd.DataFrame(text_video_three_seed_rows)
display(text_video_three_seed_summary_df)

,Metric,Mean,Std
0,CCC,0.609208,0.023594
1,RMSE,5.692522,0.267126
2,MAE,4.309524,0.252749
3,Exact_Accuracy,0.107143,0.017857


In [152]:
text_video_test_results_56_path = TEXT_VIDEO_CHECKPOINT_ROOT / "three_seed_test_results_56.csv"
text_video_test_summary_56_path = TEXT_VIDEO_CHECKPOINT_ROOT / "three_seed_test_summary_56.csv"
text_video_test_seed_summary_df.to_csv(text_video_test_results_56_path, index=False)
text_video_three_seed_summary_df.to_csv(text_video_test_summary_56_path, index=False)
print("56人逐Seed结果：", text_video_test_results_56_path)
print("56人汇总结果：", text_video_test_summary_56_path)

56人逐Seed结果： /workspace/E-DAIC/checkpoints/phq8_text_video_paperlike/three_seed_test_results_56.csv
56人汇总结果： /workspace/E-DAIC/checkpoints/phq8_text_video_paperlike/three_seed_test_summary_56.csv


In [154]:
text_video_test_keep_mask_55 = torch.tensor([participant_id != 637 for participant_id in text_video_test_dataset.participant_ids], dtype=torch.bool)
print("55人协议实际人数：", int(text_video_test_keep_mask_55.sum()))

55人协议实际人数： 55


In [155]:
text_video_test_55_rows = []
for result in all_text_video_test_results:
    predicted_totals_55 = result["predicted_totals"][text_video_test_keep_mask_55]
    true_totals_55 = result["true_totals"][text_video_test_keep_mask_55]
    ccc_55 = calculate_ccc(predicted_totals_55, true_totals_55)
    rmse_55 = float(torch.sqrt(torch.mean((predicted_totals_55.float() - true_totals_55.float()) ** 2)).item())
    mae_55 = float(torch.mean(torch.abs(predicted_totals_55.float() - true_totals_55.float())).item())
    exact_accuracy_55 = float((predicted_totals_55 == true_totals_55).float().mean().item())
    text_video_test_55_rows.append({"Seed": result["seed"], "Sample_Count": len(predicted_totals_55), "CCC": ccc_55, "RMSE": rmse_55, "MAE": mae_55, "Exact_Accuracy": exact_accuracy_55})
text_video_test_55_summary_df = pd.DataFrame(text_video_test_55_rows)
display(text_video_test_55_summary_df)

,Seed,Sample_Count,CCC,RMSE,MAE,Exact_Accuracy
0,42,55,0.574119,5.943675,4.490909,0.090909
1,100,55,0.594663,5.841855,4.527273,0.127273
2,1234,55,0.618996,5.437245,4.072727,0.109091


In [156]:
text_video_three_seed_55_rows = []
for metric_name in ["CCC", "RMSE", "MAE", "Exact_Accuracy"]:
    text_video_three_seed_55_rows.append({"Metric": metric_name, "Mean": text_video_test_55_summary_df[metric_name].mean(), "Std": text_video_test_55_summary_df[metric_name].std(ddof=1)})
text_video_three_seed_55_summary_df = pd.DataFrame(text_video_three_seed_55_rows)
display(text_video_three_seed_55_summary_df)

,Metric,Mean,Std
0,CCC,0.595926,0.022465
1,RMSE,5.740925,0.267877
2,MAE,4.363636,0.252590
3,Exact_Accuracy,0.109091,0.018182


In [157]:
text_video_test_results_55_path = TEXT_VIDEO_CHECKPOINT_ROOT / "three_seed_test_results_candidate55.csv"
text_video_test_summary_55_path = TEXT_VIDEO_CHECKPOINT_ROOT / "three_seed_test_summary_candidate55.csv"
text_video_test_55_summary_df.to_csv(text_video_test_results_55_path, index=False)
text_video_three_seed_55_summary_df.to_csv(text_video_test_summary_55_path, index=False)
print("55人逐Seed结果：", text_video_test_results_55_path)
print("55人汇总结果：", text_video_test_summary_55_path)

55人逐Seed结果： /workspace/E-DAIC/checkpoints/phq8_text_video_paperlike/three_seed_test_results_candidate55.csv
55人汇总结果： /workspace/E-DAIC/checkpoints/phq8_text_video_paperlike/three_seed_test_summary_candidate55.csv


### no-mask版

In [158]:
TEXT_VIDEO_NOMASK_ROOT = Path("/workspace/E-DAIC/checkpoints/phq8_text_video_nomask_ablation")
TEXT_VIDEO_NOMASK_ROOT.mkdir(parents=True, exist_ok=True)
print("No-Mask目录：", TEXT_VIDEO_NOMASK_ROOT)

No-Mask目录： /workspace/E-DAIC/checkpoints/phq8_text_video_nomask_ablation


In [159]:
class PretrainedTextEncoderNoMask(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=TEXT_FEATURE_DIM, hidden_size=50, batch_first=True, bidirectional=True)
        self.attention = nn.MultiheadAttention(embed_dim=ENCODER_OUTPUT_DIM, num_heads=ATTENTION_HEADS, dropout=0.5, batch_first=True)

    def forward(self, text_features, text_padding_mask):
        lstm_output, _ = self.lstm(text_features)
        text_encoding, _ = self.attention(lstm_output, lstm_output, lstm_output, key_padding_mask=text_padding_mask, need_weights=False)
        return text_encoding

In [165]:
class PretrainedVideoEncoderNoMask(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=VIDEO_FEATURE_DIM, hidden_size=50, batch_first=True, bidirectional=True)
        self.attention1 = nn.MultiheadAttention(embed_dim=ENCODER_OUTPUT_DIM, num_heads=ATTENTION_HEADS, dropout=0.2, batch_first=True)
        self.attention2 = nn.MultiheadAttention(embed_dim=ENCODER_OUTPUT_DIM, num_heads=ATTENTION_HEADS, dropout=0.2, batch_first=True)

    def forward(self, video_features, video_padding_mask):
        lstm_output, _ = self.lstm(video_features)
        attention1_output, _ = self.attention1(lstm_output, lstm_output, lstm_output, key_padding_mask=video_padding_mask, need_weights=False)
        video_encoding, _ = self.attention2(attention1_output, attention1_output, attention1_output, key_padding_mask=video_padding_mask, need_weights=False)
        return video_encoding

In [166]:
def load_pretrained_text_video_encoders_nomask(question_index, seed):
    text_checkpoint_path, video_checkpoint_path = get_text_video_best_loss_paths(question_index, seed)
    text_checkpoint = torch.load(text_checkpoint_path, map_location="cpu", weights_only=False)
    video_checkpoint = torch.load(video_checkpoint_path, map_location="cpu", weights_only=False)
    text_encoder = PretrainedTextEncoderNoMask()
    video_encoder = PretrainedVideoEncoderNoMask()
    text_encoder_state = extract_state_dict_by_prefixes(text_checkpoint["model_state_dict"], prefixes=("lstm.", "attention."))
    video_encoder_state = extract_state_dict_by_prefixes(video_checkpoint["model_state_dict"], prefixes=("lstm.", "attention1.", "attention2."))
    text_encoder.load_state_dict(text_encoder_state, strict=True)
    video_encoder.load_state_dict(video_encoder_state, strict=True)
    return text_encoder, video_encoder, text_checkpoint_path, video_checkpoint_path

In [167]:
class TextVideoQuestMFNoMask(nn.Module):
    def __init__(self, text_encoder, video_encoder):
        super().__init__()
        self.text_encoder = text_encoder
        self.video_encoder = video_encoder
        self.video_to_text_cross_attention = nn.MultiheadAttention(embed_dim=ENCODER_OUTPUT_DIM, num_heads=ATTENTION_HEADS, dropout=FUSION_DROPOUT, batch_first=True)
        self.text_to_video_cross_attention = nn.MultiheadAttention(embed_dim=ENCODER_OUTPUT_DIM, num_heads=ATTENTION_HEADS, dropout=FUSION_DROPOUT, batch_first=True)
        self.text_self_attention = nn.MultiheadAttention(embed_dim=ENCODER_OUTPUT_DIM, num_heads=ATTENTION_HEADS, dropout=FUSION_DROPOUT, batch_first=True)
        self.video_self_attention = nn.MultiheadAttention(embed_dim=ENCODER_OUTPUT_DIM, num_heads=ATTENTION_HEADS, dropout=FUSION_DROPOUT, batch_first=True)
        self.mlp = nn.Sequential(nn.Flatten(), nn.Dropout(FUSION_MLP_FIRST_DROPOUT), nn.Linear(MAX_TURNS * ENCODER_OUTPUT_DIM * 2, 256), nn.ReLU(), nn.Dropout(FUSION_MLP_LAST_DROPOUT), nn.Linear(256, NUM_CLASSES))
        for parameter in self.text_encoder.parameters():
            parameter.requires_grad = False
        self.text_encoder.eval()

    def train(self, mode=True):
        super().train(mode)
        self.text_encoder.eval()
        return self

    def forward(self, text_features, text_padding_mask, video_features, video_padding_mask, return_intermediates=False):
        text_encoding = self.text_encoder(text_features, text_padding_mask)
        video_encoding = self.video_encoder(video_features, video_padding_mask)
        video_to_text, _ = self.video_to_text_cross_attention(text_encoding, video_encoding, video_encoding, key_padding_mask=video_padding_mask, need_weights=False)
        text_to_video, _ = self.text_to_video_cross_attention(video_encoding, text_encoding, text_encoding, key_padding_mask=text_padding_mask, need_weights=False)
        text_fused_encoding, _ = self.text_self_attention(video_to_text, video_to_text, video_to_text, key_padding_mask=text_padding_mask, need_weights=False)
        video_fused_encoding, _ = self.video_self_attention(text_to_video, text_to_video, text_to_video, key_padding_mask=video_padding_mask, need_weights=False)
        fused_encoding = torch.cat((text_fused_encoding, video_fused_encoding), dim=2)
        logits = self.mlp(fused_encoding)
        if return_intermediates:
            return logits, {"text_encoding": text_encoding, "video_encoding": video_encoding, "video_to_text": video_to_text, "text_to_video": text_to_video, "text_fused_encoding": text_fused_encoding, "video_fused_encoding": video_fused_encoding, "fused_encoding": fused_encoding}
        return logits

In [168]:
set_seed(42)
nomask_text_encoder, nomask_video_encoder, nomask_text_checkpoint_path, nomask_video_checkpoint_path = load_pretrained_text_video_encoders_nomask(question_index=0, seed=42)
nomask_diagnostic_model = TextVideoQuestMFNoMask(nomask_text_encoder, nomask_video_encoder).to(device)
nomask_diagnostic_model.eval()
with torch.no_grad():
    nomask_logits, nomask_intermediates = nomask_diagnostic_model(batch_text_features.to(device), batch_text_masks.to(device), batch_video_features.to(device), batch_video_masks.to(device), return_intermediates=True)
print("No-Mask拼接形状：", nomask_intermediates["fused_encoding"].shape)
print("No-Mask Logits形状：", nomask_logits.shape)
print("No-Mask Logits全部有限：", bool(torch.isfinite(nomask_logits).all()))
print("No-Mask可训练参数：", f"{sum(parameter.numel() for parameter in nomask_diagnostic_model.parameters() if parameter.requires_grad):,}")

No-Mask拼接形状： torch.Size([10, 120, 200])
No-Mask Logits形状： torch.Size([10, 4])
No-Mask Logits全部有限： True
No-Mask可训练参数： 7,227,684


In [169]:
def save_text_video_nomask_checkpoint(path, checkpoint_type, epoch, question_index, seed, model, optimizer, train_metrics, val_metrics, class_counts, class_weights, best_val_loss, best_val_ccc, best_loss_epoch, best_ccc_epoch, text_checkpoint_path, video_checkpoint_path):
    save_text_video_checkpoint(path, checkpoint_type, epoch, question_index, seed, model, optimizer, train_metrics, val_metrics, class_counts, class_weights, best_val_loss, best_val_ccc, best_loss_epoch, best_ccc_epoch, text_checkpoint_path, video_checkpoint_path)
    checkpoint = torch.load(path, map_location="cpu", weights_only=False)
    checkpoint["video_variant"] = "no_padding_output_zero_ablation"
    checkpoint["ablation_variable"] = "masked_fill_removed"
    torch.save(checkpoint, path)

In [170]:
def train_text_video_nomask_question(seed, question_index):
    set_seed(seed)
    train_loader, val_loader = create_text_video_dataloaders(seed)
    text_encoder, video_encoder, text_checkpoint_path, video_checkpoint_path = load_pretrained_text_video_encoders_nomask(question_index, seed)
    model = TextVideoQuestMFNoMask(text_encoder, video_encoder).to(device)
    train_labels = text_video_train_dataset.labels[:, question_index]
    class_counts = torch.bincount(train_labels, minlength=NUM_CLASSES)
    class_weights = torch.pow(len(train_labels) / class_counts.float(), BETA)
    criterion = ImbOLLLoss(class_weights, alpha=ALPHA).to(device)
    optimizer = torch.optim.AdamW((parameter for parameter in model.parameters() if parameter.requires_grad), lr=FUSION_LEARNING_RATE, eps=FUSION_ADAM_EPSILON, weight_decay=FUSION_WEIGHT_DECAY)
    checkpoint_dir = TEXT_VIDEO_NOMASK_ROOT / f"seed_{seed}" / f"question_{question_index + 1}"
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    best_loss_path = checkpoint_dir / "best_loss.pt"
    best_ccc_path = checkpoint_dir / "best_ccc.pt"
    last_path = checkpoint_dir / "last.pt"
    history_path = checkpoint_dir / "history.csv"
    best_val_loss = float("inf")
    best_val_ccc = float("-inf")
    best_loss_epoch = -1
    best_ccc_epoch = -1
    history_rows = []
    print("=" * 100)
    print(f"开始No-Mask消融：Seed={seed}，Q{question_index + 1}")
    print("=" * 100)
    for epoch in range(1, FUSION_NUM_EPOCHS + 1):
        epoch_start_time = time.perf_counter()
        train_metrics = run_text_video_epoch(model, train_loader, question_index, criterion, optimizer)
        val_metrics = run_text_video_epoch(model, val_loader, question_index, criterion)
        elapsed_seconds = time.perf_counter() - epoch_start_time
        improved_loss = val_metrics["loss"] < best_val_loss
        improved_ccc = val_metrics["ccc"] > best_val_ccc
        if improved_loss:
            best_val_loss = val_metrics["loss"]
            best_loss_epoch = epoch
        if improved_ccc:
            best_val_ccc = val_metrics["ccc"]
            best_ccc_epoch = epoch
        if improved_loss:
            save_text_video_nomask_checkpoint(best_loss_path, "best_loss", epoch, question_index, seed, model, optimizer, train_metrics, val_metrics, class_counts, class_weights, best_val_loss, best_val_ccc, best_loss_epoch, best_ccc_epoch, text_checkpoint_path, video_checkpoint_path)
        if improved_ccc:
            save_text_video_nomask_checkpoint(best_ccc_path, "best_ccc", epoch, question_index, seed, model, optimizer, train_metrics, val_metrics, class_counts, class_weights, best_val_loss, best_val_ccc, best_loss_epoch, best_ccc_epoch, text_checkpoint_path, video_checkpoint_path)
        history_rows.append({"Epoch": epoch, "Train_Loss": train_metrics["loss"], "Train_CCC": train_metrics["ccc"], "Val_Loss": val_metrics["loss"], "Val_CCC": val_metrics["ccc"], "Val_RMSE": val_metrics["rmse"], "Val_MAE": val_metrics["mae"], "Elapsed_Seconds": elapsed_seconds})
        print(f"Epoch {epoch:02d}/{FUSION_NUM_EPOCHS} | Train Loss {train_metrics['loss']:.6f} | Val Loss {val_metrics['loss']:.6f} | Val CCC {val_metrics['ccc']:.6f} | {elapsed_seconds:.2f}s")
    save_text_video_nomask_checkpoint(last_path, "last", FUSION_NUM_EPOCHS, question_index, seed, model, optimizer, train_metrics, val_metrics, class_counts, class_weights, best_val_loss, best_val_ccc, best_loss_epoch, best_ccc_epoch, text_checkpoint_path, video_checkpoint_path)
    pd.DataFrame(history_rows).to_csv(history_path, index=False)
    print(f"完成：Seed={seed}，Q{question_index + 1}，Best CCC={best_val_ccc:.6f}，Epoch={best_ccc_epoch}")
    return {"seed": seed, "question_index": question_index, "best_val_loss": best_val_loss, "best_val_ccc": best_val_ccc, "best_loss_epoch": best_loss_epoch, "best_ccc_epoch": best_ccc_epoch, "best_loss_path": best_loss_path, "best_ccc_path": best_ccc_path, "last_path": last_path, "history_path": history_path}

In [171]:
all_text_video_nomask_results = {}
for seed in SEEDS:
    seed_results = []
    for question_index in range(len(PHQ8_QUESTION_COLUMNS)):
        result = train_text_video_nomask_question(seed, question_index)
        seed_results.append(result)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    all_text_video_nomask_results[seed] = seed_results

开始No-Mask消融：Seed=42，Q1
Epoch 01/20 | Train Loss 2.113703 | Val Loss 2.694212 | Val CCC 0.310641 | 4.01s
Epoch 02/20 | Train Loss 1.281714 | Val Loss 3.753850 | Val CCC 0.337017 | 0.91s
Epoch 03/20 | Train Loss 1.396936 | Val Loss 2.496702 | Val CCC 0.353632 | 1.18s
Epoch 04/20 | Train Loss 1.226295 | Val Loss 2.841679 | Val CCC 0.251309 | 0.88s
Epoch 05/20 | Train Loss 1.133127 | Val Loss 3.314361 | Val CCC 0.309392 | 1.03s
Epoch 06/20 | Train Loss 1.194140 | Val Loss 3.038854 | Val CCC 0.446889 | 1.03s
Epoch 07/20 | Train Loss 1.253988 | Val Loss 2.539497 | Val CCC 0.491329 | 0.93s
Epoch 08/20 | Train Loss 1.150188 | Val Loss 3.028973 | Val CCC 0.270101 | 0.98s
Epoch 09/20 | Train Loss 1.063609 | Val Loss 2.562483 | Val CCC 0.488300 | 1.07s
Epoch 10/20 | Train Loss 1.122283 | Val Loss 2.941964 | Val CCC 0.363753 | 1.04s
Epoch 11/20 | Train Loss 1.124791 | Val Loss 3.899661 | Val CCC 0.263703 | 0.88s
Epoch 12/20 | Train Loss 0.993394 | Val Loss 2.710430 | Val CCC 0.488300 | 1.23s
Epoch

In [172]:
for seed, seed_results in all_text_video_nomask_results.items():
    print("=" * 70)
    print("Seed：", seed)
    print("问题数量：", len(seed_results))
    print("Best Loss全部存在：", all(result["best_loss_path"].exists() for result in seed_results))
    print("Best CCC全部存在：", all(result["best_ccc_path"].exists() for result in seed_results))
    print("Last全部存在：", all(result["last_path"].exists() for result in seed_results))

Seed： 42
问题数量： 8
Best Loss全部存在： True
Best CCC全部存在： True
Last全部存在： True
Seed： 100
问题数量： 8
Best Loss全部存在： True
Best CCC全部存在： True
Last全部存在： True
Seed： 1234
问题数量： 8
Best Loss全部存在： True
Best CCC全部存在： True
Last全部存在： True


In [173]:
def evaluate_text_video_nomask_seed_on_validation(seed):
    val_loader = DataLoader(text_video_val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, drop_last=False)
    question_prediction_list = []
    for question_index in range(len(PHQ8_QUESTION_COLUMNS)):
        checkpoint_path = TEXT_VIDEO_NOMASK_ROOT / f"seed_{seed}" / f"question_{question_index + 1}" / "best_ccc.pt"
        if not checkpoint_path.exists():
            raise FileNotFoundError(f"No-Mask Checkpoint不存在：{checkpoint_path}")
        checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
        if checkpoint["seed"] != seed or checkpoint["question_index"] != question_index or checkpoint["checkpoint_type"] != "best_ccc":
            raise RuntimeError(f"Checkpoint元数据错误：{checkpoint_path}")
        model = TextVideoQuestMFNoMask(PretrainedTextEncoderNoMask(), PretrainedVideoEncoderNoMask()).to(device)
        load_result = model.load_state_dict(checkpoint["model_state_dict"], strict=True)
        if load_result.missing_keys or load_result.unexpected_keys:
            raise RuntimeError(f"Checkpoint参数不一致：{checkpoint_path}")
        model.eval()
        batch_prediction_list = []
        with torch.no_grad():
            for text_features, text_masks, video_features, video_masks, all_labels in val_loader:
                logits = model(text_features.to(device), text_masks.to(device), video_features.to(device), video_masks.to(device))
                batch_prediction_list.append(logits.argmax(dim=1).cpu())
        question_predictions = torch.cat(batch_prediction_list)
        question_prediction_list.append(question_predictions)
        print(f"Seed {seed} | Q{question_index + 1} | Epoch {checkpoint['epoch']:02d} | Val CCC {checkpoint['val_metrics']['ccc']:.6f}")
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    question_predictions = torch.stack(question_prediction_list, dim=1)
    question_targets = text_video_val_dataset.labels.clone()
    predicted_totals = question_predictions.sum(dim=1)
    true_totals = question_targets.sum(dim=1)
    total_ccc = calculate_ccc(predicted_totals, true_totals)
    total_rmse = float(torch.sqrt(torch.mean((predicted_totals.float() - true_totals.float()) ** 2)).item())
    total_mae = float(torch.mean(torch.abs(predicted_totals.float() - true_totals.float())).item())
    exact_accuracy = float((predicted_totals == true_totals).float().mean().item())
    print("=" * 70)
    print(f"No-Mask Seed {seed}验证结果")
    print(f"CCC：{total_ccc:.6f}")
    print(f"RMSE：{total_rmse:.6f}")
    print(f"MAE：{total_mae:.6f}")
    print(f"完全相等比例：{exact_accuracy:.6f}")
    return {"seed": seed, "sample_count": len(true_totals), "ccc": total_ccc, "rmse": total_rmse, "mae": total_mae, "exact_accuracy": exact_accuracy, "predicted_totals": predicted_totals, "true_totals": true_totals}

In [174]:
text_video_nomask_seed42_validation_result = evaluate_text_video_nomask_seed_on_validation(seed=42)
text_video_nomask_seed100_validation_result = evaluate_text_video_nomask_seed_on_validation(seed=100)
text_video_nomask_seed1234_validation_result = evaluate_text_video_nomask_seed_on_validation(seed=1234)

Seed 42 | Q1 | Epoch 17 | Val CCC 0.507003
Seed 42 | Q2 | Epoch 09 | Val CCC 0.560000
Seed 42 | Q3 | Epoch 08 | Val CCC 0.486281
Seed 42 | Q4 | Epoch 17 | Val CCC 0.414504
Seed 42 | Q5 | Epoch 06 | Val CCC 0.517218
Seed 42 | Q6 | Epoch 06 | Val CCC 0.513761
Seed 42 | Q7 | Epoch 10 | Val CCC 0.558587
Seed 42 | Q8 | Epoch 02 | Val CCC 0.306794
No-Mask Seed 42验证结果
CCC：0.685568
RMSE：4.538723
MAE：3.400000
完全相等比例：0.109091
Seed 100 | Q1 | Epoch 09 | Val CCC 0.430052
Seed 100 | Q2 | Epoch 16 | Val CCC 0.547945
Seed 100 | Q3 | Epoch 05 | Val CCC 0.417760
Seed 100 | Q4 | Epoch 13 | Val CCC 0.520323
Seed 100 | Q5 | Epoch 19 | Val CCC 0.472589
Seed 100 | Q6 | Epoch 04 | Val CCC 0.446271
Seed 100 | Q7 | Epoch 20 | Val CCC 0.488423
Seed 100 | Q8 | Epoch 13 | Val CCC 0.298643
No-Mask Seed 100验证结果
CCC：0.639383
RMSE：4.750120
MAE：3.509091
完全相等比例：0.072727
Seed 1234 | Q1 | Epoch 03 | Val CCC 0.511910
Seed 1234 | Q2 | Epoch 06 | Val CCC 0.521739
Seed 1234 | Q3 | Epoch 18 | Val CCC 0.433305
Seed 1234 | Q4 |

In [175]:
all_text_video_nomask_validation_results = [text_video_nomask_seed42_validation_result, text_video_nomask_seed100_validation_result, text_video_nomask_seed1234_validation_result]
text_video_nomask_validation_rows = []
for result in all_text_video_nomask_validation_results:
    text_video_nomask_validation_rows.append({"Seed": result["seed"], "Sample_Count": result["sample_count"], "CCC": result["ccc"], "RMSE": result["rmse"], "MAE": result["mae"], "Exact_Accuracy": result["exact_accuracy"]})
text_video_nomask_validation_df = pd.DataFrame(text_video_nomask_validation_rows)
display(text_video_nomask_validation_df)

,Seed,Sample_Count,CCC,RMSE,MAE,Exact_Accuracy
0,42,55,0.685568,4.538723,3.400000,0.109091
1,100,55,0.639383,4.750120,3.509091,0.072727
2,1234,55,0.675825,4.612236,3.345454,0.145455


In [176]:
text_video_nomask_statistics_rows = []
for metric_name in ["CCC", "RMSE", "MAE", "Exact_Accuracy"]:
    text_video_nomask_statistics_rows.append({"Metric": metric_name, "Mean": text_video_nomask_validation_df[metric_name].mean(), "Std": text_video_nomask_validation_df[metric_name].std(ddof=1)})
text_video_nomask_statistics_df = pd.DataFrame(text_video_nomask_statistics_rows)
display(text_video_nomask_statistics_df)

,Metric,Mean,Std
0,CCC,0.666926,0.024345
1,RMSE,4.633693,0.107319
2,MAE,3.418182,0.083320
3,Exact_Accuracy,0.109091,0.036364


In [177]:
mask_nomask_paired_df = text_video_validation_seed_summary_df[["Seed", "CCC", "RMSE", "MAE"]].merge(text_video_nomask_validation_df[["Seed", "CCC", "RMSE", "MAE"]], on="Seed", suffixes=("_Mask", "_NoMask"), validate="one_to_one")
mask_nomask_paired_df["CCC_Delta_NoMask_Minus_Mask"] = mask_nomask_paired_df["CCC_NoMask"] - mask_nomask_paired_df["CCC_Mask"]
mask_nomask_paired_df["RMSE_Delta_NoMask_Minus_Mask"] = mask_nomask_paired_df["RMSE_NoMask"] - mask_nomask_paired_df["RMSE_Mask"]
mask_nomask_paired_df["MAE_Delta_NoMask_Minus_Mask"] = mask_nomask_paired_df["MAE_NoMask"] - mask_nomask_paired_df["MAE_Mask"]
display(mask_nomask_paired_df)

,Seed,CCC_Mask,RMSE_Mask,MAE_Mask,CCC_NoMask,RMSE_NoMask,MAE_NoMask,CCC_Delta_NoMask_Minus_Mask,RMSE_Delta_NoMask_Minus_Mask,MAE_Delta_NoMask_Minus_Mask
0,42,0.680283,4.421024,3.218182,0.685568,4.538723,3.400000,0.005286,0.117699,0.181818
1,100,0.625042,4.837355,3.654546,0.639383,4.750120,3.509091,0.014342,-0.087235,-0.145455
2,1234,0.661026,4.568668,3.236364,0.675825,4.612236,3.345454,0.014799,0.043569,0.109091


In [178]:
mask_nomask_summary_rows = []
mask_nomask_summary_rows.append({"Variant": "Mask", "CCC_Mean": text_video_validation_seed_summary_df["CCC"].mean(), "CCC_Std": text_video_validation_seed_summary_df["CCC"].std(ddof=1), "RMSE_Mean": text_video_validation_seed_summary_df["RMSE"].mean(), "RMSE_Std": text_video_validation_seed_summary_df["RMSE"].std(ddof=1), "MAE_Mean": text_video_validation_seed_summary_df["MAE"].mean(), "MAE_Std": text_video_validation_seed_summary_df["MAE"].std(ddof=1)})
mask_nomask_summary_rows.append({"Variant": "No-Mask", "CCC_Mean": text_video_nomask_validation_df["CCC"].mean(), "CCC_Std": text_video_nomask_validation_df["CCC"].std(ddof=1), "RMSE_Mean": text_video_nomask_validation_df["RMSE"].mean(), "RMSE_Std": text_video_nomask_validation_df["RMSE"].std(ddof=1), "MAE_Mean": text_video_nomask_validation_df["MAE"].mean(), "MAE_Std": text_video_nomask_validation_df["MAE"].std(ddof=1)})
mask_nomask_summary_df = pd.DataFrame(mask_nomask_summary_rows)
display(mask_nomask_summary_df)

,Variant,CCC_Mean,CCC_Std,RMSE_Mean,RMSE_Std,MAE_Mean,MAE_Std
0,Mask,0.655450,0.028039,4.609015,0.211078,3.369697,0.246854
1,No-Mask,0.666926,0.024345,4.633693,0.107319,3.418182,0.083320


In [179]:
mask_nomask_paired_df = text_video_validation_seed_summary_df[["Seed", "CCC", "RMSE", "MAE"]].merge(text_video_nomask_validation_df[["Seed", "CCC", "RMSE", "MAE"]], on="Seed", suffixes=("_Mask", "_NoMask"), validate="one_to_one")
mask_nomask_paired_df["CCC_Delta_NoMask_Minus_Mask"] = mask_nomask_paired_df["CCC_NoMask"] - mask_nomask_paired_df["CCC_Mask"]
mask_nomask_paired_df["RMSE_Delta_NoMask_Minus_Mask"] = mask_nomask_paired_df["RMSE_NoMask"] - mask_nomask_paired_df["RMSE_Mask"]
mask_nomask_paired_df["MAE_Delta_NoMask_Minus_Mask"] = mask_nomask_paired_df["MAE_NoMask"] - mask_nomask_paired_df["MAE_Mask"]
display(mask_nomask_paired_df)

,Seed,CCC_Mask,RMSE_Mask,MAE_Mask,CCC_NoMask,RMSE_NoMask,MAE_NoMask,CCC_Delta_NoMask_Minus_Mask,RMSE_Delta_NoMask_Minus_Mask,MAE_Delta_NoMask_Minus_Mask
0,42,0.680283,4.421024,3.218182,0.685568,4.538723,3.400000,0.005286,0.117699,0.181818
1,100,0.625042,4.837355,3.654546,0.639383,4.750120,3.509091,0.014342,-0.087235,-0.145455
2,1234,0.661026,4.568668,3.236364,0.675825,4.612236,3.345454,0.014799,0.043569,0.109091


In [180]:
mask_nomask_summary_rows = []
mask_nomask_summary_rows.append({"Variant": "Mask", "CCC_Mean": text_video_validation_seed_summary_df["CCC"].mean(), "CCC_Std": text_video_validation_seed_summary_df["CCC"].std(ddof=1), "RMSE_Mean": text_video_validation_seed_summary_df["RMSE"].mean(), "RMSE_Std": text_video_validation_seed_summary_df["RMSE"].std(ddof=1), "MAE_Mean": text_video_validation_seed_summary_df["MAE"].mean(), "MAE_Std": text_video_validation_seed_summary_df["MAE"].std(ddof=1)})
mask_nomask_summary_rows.append({"Variant": "No-Mask", "CCC_Mean": text_video_nomask_validation_df["CCC"].mean(), "CCC_Std": text_video_nomask_validation_df["CCC"].std(ddof=1), "RMSE_Mean": text_video_nomask_validation_df["RMSE"].mean(), "RMSE_Std": text_video_nomask_validation_df["RMSE"].std(ddof=1), "MAE_Mean": text_video_nomask_validation_df["MAE"].mean(), "MAE_Std": text_video_nomask_validation_df["MAE"].std(ddof=1)})
mask_nomask_summary_df = pd.DataFrame(mask_nomask_summary_rows)
display(mask_nomask_summary_df)

,Variant,CCC_Mean,CCC_Std,RMSE_Mean,RMSE_Std,MAE_Mean,MAE_Std
0,Mask,0.655450,0.028039,4.609015,0.211078,3.369697,0.246854
1,No-Mask,0.666926,0.024345,4.633693,0.107319,3.418182,0.083320


In [181]:
text_video_nomask_validation_df.to_csv(TEXT_VIDEO_NOMASK_ROOT / "three_seed_validation_results.csv", index=False)
text_video_nomask_statistics_df.to_csv(TEXT_VIDEO_NOMASK_ROOT / "three_seed_validation_summary.csv", index=False)
mask_nomask_paired_df.to_csv(TEXT_VIDEO_NOMASK_ROOT / "mask_nomask_paired_validation_comparison.csv", index=False)
mask_nomask_summary_df.to_csv(TEXT_VIDEO_NOMASK_ROOT / "mask_nomask_validation_summary.csv", index=False)

In [182]:
def evaluate_text_video_nopostzero_seed_on_test(seed):
    question_prediction_list = []
    selected_checkpoint_rows = []
    for question_index in range(len(PHQ8_QUESTION_COLUMNS)):
        checkpoint_path = TEXT_VIDEO_NOMASK_ROOT / f"seed_{seed}" / f"question_{question_index + 1}" / "best_ccc.pt"
        if not checkpoint_path.exists():
            raise FileNotFoundError(f"NoPostZero Checkpoint不存在：{checkpoint_path}")
        checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
        if checkpoint["seed"] != seed or checkpoint["question_index"] != question_index or checkpoint["checkpoint_type"] != "best_ccc":
            raise RuntimeError(f"Checkpoint元数据错误：{checkpoint_path}")
        model = TextVideoQuestMFNoMask(PretrainedTextEncoderNoMask(), PretrainedVideoEncoderNoMask()).to(device)
        load_result = model.load_state_dict(checkpoint["model_state_dict"], strict=True)
        if load_result.missing_keys or load_result.unexpected_keys:
            raise RuntimeError(f"Checkpoint参数不一致：{checkpoint_path}")
        model.eval()
        batch_prediction_list = []
        with torch.no_grad():
            for text_features, text_masks, video_features, video_masks, true_totals, participant_ids in text_video_test_loader:
                logits = model(text_features.to(device), text_masks.to(device), video_features.to(device), video_masks.to(device))
                batch_prediction_list.append(logits.argmax(dim=1).cpu())
        question_predictions = torch.cat(batch_prediction_list)
        if len(question_predictions) != len(text_video_test_dataset):
            raise RuntimeError(f"Q{question_index + 1}预测数量错误")
        question_prediction_list.append(question_predictions)
        selected_checkpoint_rows.append({"Seed": seed, "Question_Index": question_index, "Question_Number": question_index + 1, "Question_Name": PHQ8_QUESTION_COLUMNS[question_index], "Checkpoint_Path": str(checkpoint_path), "Checkpoint_Epoch": checkpoint["epoch"], "Validation_CCC": checkpoint["val_metrics"]["ccc"], "Variant": "NoPostZero"})
        print(f"Seed {seed} | Q{question_index + 1} | Epoch {checkpoint['epoch']:02d} | Val CCC {checkpoint['val_metrics']['ccc']:.6f}")
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    question_predictions = torch.stack(question_prediction_list, dim=1)
    predicted_totals = question_predictions.sum(dim=1)
    true_totals = text_video_test_dataset.true_totals.clone()
    total_ccc = calculate_ccc(predicted_totals, true_totals)
    total_rmse = float(torch.sqrt(torch.mean((predicted_totals.float() - true_totals.float()) ** 2)).item())
    total_mae = float(torch.mean(torch.abs(predicted_totals.float() - true_totals.float())).item())
    exact_accuracy = float((predicted_totals == true_totals).float().mean().item())
    prediction_df = pd.DataFrame({"Participant_ID": text_video_test_dataset.participant_ids})
    for question_index in range(len(PHQ8_QUESTION_COLUMNS)):
        prediction_df[f"Q{question_index + 1}_Predicted"] = question_predictions[:, question_index].numpy()
    prediction_df["Predicted_Total"] = predicted_totals.numpy()
    prediction_df["True_Total"] = true_totals.numpy()
    prediction_path = TEXT_VIDEO_NOMASK_ROOT / f"seed_{seed}_test_predictions_56.csv"
    selected_checkpoint_path = TEXT_VIDEO_NOMASK_ROOT / f"seed_{seed}_test_selected_checkpoints.csv"
    prediction_df.to_csv(prediction_path, index=False)
    pd.DataFrame(selected_checkpoint_rows).to_csv(selected_checkpoint_path, index=False)
    print("=" * 70)
    print(f"Text+Video NoPostZero Seed {seed}测试结果")
    print(f"测试人数：{len(prediction_df)}")
    print(f"CCC：{total_ccc:.6f}")
    print(f"RMSE：{total_rmse:.6f}")
    print(f"MAE：{total_mae:.6f}")
    print(f"完全相等比例：{exact_accuracy:.6f}")
    print(f"预测范围：{int(predicted_totals.min())}～{int(predicted_totals.max())}")
    print(f"真实范围：{int(true_totals.min())}～{int(true_totals.max())}")
    return {"seed": seed, "sample_count": len(prediction_df), "ccc": total_ccc, "rmse": total_rmse, "mae": total_mae, "exact_accuracy": exact_accuracy, "predicted_totals": predicted_totals, "true_totals": true_totals, "prediction_path": prediction_path}

In [183]:
text_video_nopostzero_seed42_test_result = evaluate_text_video_nopostzero_seed_on_test(seed=42)
text_video_nopostzero_seed100_test_result = evaluate_text_video_nopostzero_seed_on_test(seed=100)
text_video_nopostzero_seed1234_test_result = evaluate_text_video_nopostzero_seed_on_test(seed=1234)

Seed 42 | Q1 | Epoch 17 | Val CCC 0.507003
Seed 42 | Q2 | Epoch 09 | Val CCC 0.560000
Seed 42 | Q3 | Epoch 08 | Val CCC 0.486281
Seed 42 | Q4 | Epoch 17 | Val CCC 0.414504
Seed 42 | Q5 | Epoch 06 | Val CCC 0.517218
Seed 42 | Q6 | Epoch 06 | Val CCC 0.513761
Seed 42 | Q7 | Epoch 10 | Val CCC 0.558587
Seed 42 | Q8 | Epoch 02 | Val CCC 0.306794
Text+Video NoPostZero Seed 42测试结果
测试人数：56
CCC：0.609228
RMSE：6.005949
MAE：4.500000
完全相等比例：0.142857
预测范围：0～22
真实范围：0～22
Seed 100 | Q1 | Epoch 09 | Val CCC 0.430052
Seed 100 | Q2 | Epoch 16 | Val CCC 0.547945
Seed 100 | Q3 | Epoch 05 | Val CCC 0.417760
Seed 100 | Q4 | Epoch 13 | Val CCC 0.520323
Seed 100 | Q5 | Epoch 19 | Val CCC 0.472589
Seed 100 | Q6 | Epoch 04 | Val CCC 0.446271
Seed 100 | Q7 | Epoch 20 | Val CCC 0.488423
Seed 100 | Q8 | Epoch 13 | Val CCC 0.298643
Text+Video NoPostZero Seed 100测试结果
测试人数：56
CCC：0.656879
RMSE：5.467436
MAE：4.250000
完全相等比例：0.089286
预测范围：0～22
真实范围：0～22
Seed 1234 | Q1 | Epoch 03 | Val CCC 0.511910
Seed 1234 | Q2 | Epoch

In [184]:
all_text_video_nopostzero_test_results = [text_video_nopostzero_seed42_test_result, text_video_nopostzero_seed100_test_result, text_video_nopostzero_seed1234_test_result]
text_video_nopostzero_test_rows = []
for result in all_text_video_nopostzero_test_results:
    text_video_nopostzero_test_rows.append({"Seed": result["seed"], "Sample_Count": result["sample_count"], "CCC": result["ccc"], "RMSE": result["rmse"], "MAE": result["mae"], "Exact_Accuracy": result["exact_accuracy"]})
text_video_nopostzero_test_df = pd.DataFrame(text_video_nopostzero_test_rows)
display(text_video_nopostzero_test_df)

,Seed,Sample_Count,CCC,RMSE,MAE,Exact_Accuracy
0,42,56,0.609228,6.005949,4.500000,0.142857
1,100,56,0.656879,5.467436,4.250000,0.089286
2,1234,56,0.637801,5.538824,4.071429,0.125000


In [185]:
all_text_video_nopostzero_test_results = [text_video_nopostzero_seed42_test_result, text_video_nopostzero_seed100_test_result, text_video_nopostzero_seed1234_test_result]
text_video_nopostzero_test_rows = []
for result in all_text_video_nopostzero_test_results:
    text_video_nopostzero_test_rows.append({"Seed": result["seed"], "Sample_Count": result["sample_count"], "CCC": result["ccc"], "RMSE": result["rmse"], "MAE": result["mae"], "Exact_Accuracy": result["exact_accuracy"]})
text_video_nopostzero_test_df = pd.DataFrame(text_video_nopostzero_test_rows)
display(text_video_nopostzero_test_df)

,Seed,Sample_Count,CCC,RMSE,MAE,Exact_Accuracy
0,42,56,0.609228,6.005949,4.500000,0.142857
1,100,56,0.656879,5.467436,4.250000,0.089286
2,1234,56,0.637801,5.538824,4.071429,0.125000


In [186]:
text_video_nopostzero_test_statistics_rows = []
for metric_name in ["CCC", "RMSE", "MAE", "Exact_Accuracy"]:
    text_video_nopostzero_test_statistics_rows.append({"Metric": metric_name, "Mean": text_video_nopostzero_test_df[metric_name].mean(), "Std": text_video_nopostzero_test_df[metric_name].std(ddof=1)})
text_video_nopostzero_test_statistics_df = pd.DataFrame(text_video_nopostzero_test_statistics_rows)
display(text_video_nopostzero_test_statistics_df)

,Metric,Mean,Std
0,CCC,0.634636,0.023983
1,RMSE,5.670736,0.292489
2,MAE,4.273810,0.215275
3,Exact_Accuracy,0.119048,0.027277


In [187]:
text_video_nopostzero_keep_mask_55 = torch.tensor([participant_id != 637 for participant_id in text_video_test_dataset.participant_ids], dtype=torch.bool)
text_video_nopostzero_test_55_rows = []
for result in all_text_video_nopostzero_test_results:
    predicted_totals_55 = result["predicted_totals"][text_video_nopostzero_keep_mask_55]
    true_totals_55 = result["true_totals"][text_video_nopostzero_keep_mask_55]
    ccc_55 = calculate_ccc(predicted_totals_55, true_totals_55)
    rmse_55 = float(torch.sqrt(torch.mean((predicted_totals_55.float() - true_totals_55.float()) ** 2)).item())
    mae_55 = float(torch.mean(torch.abs(predicted_totals_55.float() - true_totals_55.float())).item())
    exact_accuracy_55 = float((predicted_totals_55 == true_totals_55).float().mean().item())
    text_video_nopostzero_test_55_rows.append({"Seed": result["seed"], "Sample_Count": len(predicted_totals_55), "CCC": ccc_55, "RMSE": rmse_55, "MAE": mae_55, "Exact_Accuracy": exact_accuracy_55})
text_video_nopostzero_test_55_df = pd.DataFrame(text_video_nopostzero_test_55_rows)
display(text_video_nopostzero_test_55_df)

,Seed,Sample_Count,CCC,RMSE,MAE,Exact_Accuracy
0,42,55,0.596663,6.036254,4.509091,0.145455
1,100,55,0.646045,5.510321,4.290909,0.090909
2,1234,55,0.622839,5.574292,4.090909,0.127273


In [188]:
text_video_nopostzero_test_55_statistics_rows = []
for metric_name in ["CCC", "RMSE", "MAE", "Exact_Accuracy"]:
    text_video_nopostzero_test_55_statistics_rows.append({"Metric": metric_name, "Mean": text_video_nopostzero_test_55_df[metric_name].mean(), "Std": text_video_nopostzero_test_55_df[metric_name].std(ddof=1)})
text_video_nopostzero_test_55_statistics_df = pd.DataFrame(text_video_nopostzero_test_55_statistics_rows)
display(text_video_nopostzero_test_55_statistics_df)

,Metric,Mean,Std
0,CCC,0.621849,0.024706
1,RMSE,5.706955,0.286969
2,MAE,4.296970,0.209157
3,Exact_Accuracy,0.121212,0.027773


In [189]:
text_video_nopostzero_test_df.to_csv(TEXT_VIDEO_NOMASK_ROOT / "three_seed_test_results_56.csv", index=False)
text_video_nopostzero_test_statistics_df.to_csv(TEXT_VIDEO_NOMASK_ROOT / "three_seed_test_summary_56.csv", index=False)
text_video_nopostzero_test_55_df.to_csv(TEXT_VIDEO_NOMASK_ROOT / "three_seed_test_results_candidate55.csv", index=False)
text_video_nopostzero_test_55_statistics_df.to_csv(TEXT_VIDEO_NOMASK_ROOT / "three_seed_test_summary_candidate55.csv", index=False)

In [190]:
final_ablation_rows = []
final_ablation_rows.append({"Variant": "PostZero", "Protocol": "56", "CCC": text_video_test_seed_summary_df["CCC"].mean(), "CCC_Std": text_video_test_seed_summary_df["CCC"].std(ddof=1), "RMSE": text_video_test_seed_summary_df["RMSE"].mean(), "MAE": text_video_test_seed_summary_df["MAE"].mean()})
final_ablation_rows.append({"Variant": "NoPostZero", "Protocol": "56", "CCC": text_video_nopostzero_test_df["CCC"].mean(), "CCC_Std": text_video_nopostzero_test_df["CCC"].std(ddof=1), "RMSE": text_video_nopostzero_test_df["RMSE"].mean(), "MAE": text_video_nopostzero_test_df["MAE"].mean()})
final_ablation_rows.append({"Variant": "PostZero", "Protocol": "Candidate55", "CCC": text_video_test_55_summary_df["CCC"].mean(), "CCC_Std": text_video_test_55_summary_df["CCC"].std(ddof=1), "RMSE": text_video_test_55_summary_df["RMSE"].mean(), "MAE": text_video_test_55_summary_df["MAE"].mean()})
final_ablation_rows.append({"Variant": "NoPostZero", "Protocol": "Candidate55", "CCC": text_video_nopostzero_test_55_df["CCC"].mean(), "CCC_Std": text_video_nopostzero_test_55_df["CCC"].std(ddof=1), "RMSE": text_video_nopostzero_test_55_df["RMSE"].mean(), "MAE": text_video_nopostzero_test_55_df["MAE"].mean()})
final_ablation_df = pd.DataFrame(final_ablation_rows)
display(final_ablation_df)
final_ablation_df.to_csv(TEXT_VIDEO_NOMASK_ROOT / "postzero_vs_nopostzero_test_comparison.csv", index=False)

,Variant,Protocol,CCC,CCC_Std,RMSE,MAE
0,PostZero,56,0.609208,0.023594,5.692522,4.309524
1,NoPostZero,56,0.634636,0.023983,5.670736,4.273810
2,PostZero,Candidate55,0.595926,0.022465,5.740925,4.363636
3,NoPostZero,Candidate55,0.621849,0.024706,5.706955,4.296970
